# Notebook 08 — ES Daily Data & Events
## ES Daily Forward Experiment v0.1 — Databento / Headline Arena data engineering

**Role of this notebook:** validate the historical ES data contract first, then build the leakage-safe daily modeling table used later by Notebook 09.

This notebook is intentionally **data engineering only**.

It does **not**:

- train Logistic Regression,
- tune hyperparameters,
- calibrate probabilities,
- optimize Brier score,
- submit forecasts to Headline Arena,
- automate live forecasting,
- merge the experimental PR into `main`.

---

## Current project state

- Stable weekly model: **v1.2.0 — frozen**
- Experimental branch: `experiment/headline-arena-es-daily`
- Experimental PR: **open / unmerged**
- Candidate historical provider: **Databento**
- Dataset: `GLBX.MDP3`
- Product: CME E-mini S&P 500 futures (`ES`)
- Continuous selector: `ES.v.0`
- Input symbology: `continuous`
- Main historical schema: `ohlcv-1m`
- Precision diagnostic schema: `trades`
- Raw market-data timestamps: UTC
- Analysis timezone: `America/New_York`

### Previously verified Databento checkpoint

A successful metadata-only run on 2026-10-01 returned approximately:

- `ohlcv-1m` availability beginning **2010-06-06**
- full available `ES.v.0` minute quote: **~$20.83**
- approximately **5.705 million** minute records
- approximately **0.3195 GB** billable size

These values are **not hardcoded as authoritative**. The metadata cells below re-query Databento so the notebook records the values that apply when it is run.

---

## Scientific dependency

```text
provider price viable
        ↓
Arena ↔ Databento alignment
        ↓
provider accepted
        ↓
full historical acquisition
        ↓
daily session table
        ↓
leakage-safe feature table
        ↓
Notebook 09 modeling
```

The most important rule is:

> **Do not build the model until we have demonstrated that our historical target reconstruction corresponds to Headline Arena's externally settled ES contract.**

## 0. Safety controls

All paid-download switches start disabled.

To reduce the chance of an accidental full-history request, the full batch download also requires a literal confirmation phrase later in the notebook.

In [1]:
ENABLE_VALIDATION_DOWNLOAD = False
ENABLE_FULL_HISTORY_DOWNLOAD = False

# This remains False until the Arena ↔ Databento review is completed manually.
DATABENTO_ACCEPTED = False

# Additional guard for the full-history batch request.
FULL_DOWNLOAD_CONFIRMATION = ""

print("ENABLE_VALIDATION_DOWNLOAD =", ENABLE_VALIDATION_DOWNLOAD)
print("ENABLE_FULL_HISTORY_DOWNLOAD =", ENABLE_FULL_HISTORY_DOWNLOAD)
print("DATABENTO_ACCEPTED =", DATABENTO_ACCEPTED)
print("FULL_DOWNLOAD_CONFIRMATION set =", bool(FULL_DOWNLOAD_CONFIRMATION))

ENABLE_VALIDATION_DOWNLOAD = False
ENABLE_FULL_HISTORY_DOWNLOAD = False
DATABENTO_ACCEPTED = False
FULL_DOWNLOAD_CONFIRMATION set = False


## 1. Imports, `.env`, and project paths

The Databento key must stay in the local project-root `.env`:

```text
DATABENTO_API_KEY=db-...
```

The notebook checks that the key exists but never prints it.

In [2]:
from pathlib import Path
from zoneinfo import ZoneInfo
from datetime import datetime
import hashlib
import json
import os
import re
import sys
import urllib.request
import urllib.parse
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import databento as db


# ------------------------------------------------------------
# Project-root detection
# ------------------------------------------------------------

HERE = Path.cwd()

PROJECT_ROOT = (
    HERE.parent
    if HERE.name.lower() == "notebooks"
    else HERE
)


# ------------------------------------------------------------
# Verify that Jupyter is using THIS project's .venv
# ------------------------------------------------------------

EXPECTED_PYTHON = (
    PROJECT_ROOT
    / ".venv"
    / "Scripts"
    / "python.exe"
)

actual_python = Path(sys.executable).resolve()
expected_python = EXPECTED_PYTHON.resolve()

print("Python executable:", actual_python)

if (
    os.path.normcase(str(actual_python))
    != os.path.normcase(str(expected_python))
):
    raise RuntimeError(
        "WRONG JUPYTER KERNEL.\n\n"
        f"Expected:\n{expected_python}\n\n"
        f"Actual:\n{actual_python}\n\n"
        "Select 'Python (.venv Market Mood)' "
        "and restart the notebook."
    )

print("Project .venv kernel check: PASS")


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "databento"
PROCESSED_DIR = DATA_DIR / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Load project .env
# ------------------------------------------------------------

ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

assert ENV_PATH.exists(), (
    f"Expected local .env at: {ENV_PATH}"
)

assert os.getenv("DATABENTO_API_KEY"), (
    "DATABENTO_API_KEY was not found "
    "in the project-root .env file."
)


# ------------------------------------------------------------
# Databento client
# ------------------------------------------------------------

client = db.Historical()


# ------------------------------------------------------------
# Time zones
# ------------------------------------------------------------

UTC = ZoneInfo("UTC")
NY = ZoneInfo("America/New_York")


print("Project root:", PROJECT_ROOT)
print(".env found:", ENV_PATH.exists())
print("Databento key loaded:", True)
print("Databento package:", db.__file__)

Python executable: C:\Users\melny\Market-Mood-Forecasting\.venv\Scripts\python.exe
Project .venv kernel check: PASS
Project root: C:\Users\melny\Market-Mood-Forecasting
.env found: True
Databento key loaded: True
Databento package: C:\Users\melny\Market-Mood-Forecasting\.venv\Lib\site-packages\databento\__init__.py


In [3]:
# ------------------------------------------------------------
# Databento metadata retry helper
# ------------------------------------------------------------
#
# Databento metadata endpoints can occasionally return
# temporary gateway/server errors such as 502, 503, or 504.
#
# We retry only transient failures.
# Authentication, permission, malformed-request, and other
# non-transient errors still fail immediately.

TRANSIENT_DATABENTO_STATUS_CODES = {
    429,
    502,
    503,
    504,
}


def metadata_call_with_retry(
    func,
    *,
    max_attempts=5,
    base_wait_seconds=2,
    **kwargs,
):
    last_error = None

    for attempt in range(
        1,
        max_attempts + 1,
    ):
        try:
            return func(**kwargs)

        except Exception as exc:
            last_error = exc

            status = getattr(
                exc,
                "http_status",
                None,
            )

            if status is None:
                status = getattr(
                    exc,
                    "status_code",
                    None,
                )

            message = str(exc)

            transient = (
                status
                in TRANSIENT_DATABENTO_STATUS_CODES
                or any(
                    str(code) in message
                    for code
                    in TRANSIENT_DATABENTO_STATUS_CODES
                )
            )

            if not transient:
                raise

            if attempt == max_attempts:
                break

            wait_seconds = (
                base_wait_seconds
                * (2 ** (attempt - 1))
            )

            print(
                "Temporary Databento metadata error "
                f"during {func.__name__}."
            )

            print(
                f"Attempt {attempt}/{max_attempts} failed. "
                f"Retrying in {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "Databento metadata request failed after "
        f"{max_attempts} attempts."
    ) from last_error


print("Databento metadata retry helper ready.")

Databento metadata retry helper ready.


### 1.1 Git safety check

The notebook does not modify `.gitignore`; it verifies that the key local paths are protected.

At minimum, `.env`, `.venv/`, and the raw Databento directory must be excluded from Git.

This check now **fails closed**: if any required protection is missing, the notebook stops before continuing.


In [4]:
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

gitignore_text = ""
if GITIGNORE_PATH.exists():
    gitignore_text = GITIGNORE_PATH.read_text(
        encoding="utf-8",
        errors="ignore",
    )

normalized_gitignore = gitignore_text.replace("\\", "/")

checks = {
    ".env ignored": bool(
        re.search(r"(?m)^\s*\.env\s*$", gitignore_text)
    ),
    ".venv ignored": bool(
        re.search(r"(?m)^\s*\.venv/?\s*$", gitignore_text)
    ),
    "raw Databento explicitly ignored": (
        "data/raw/databento" in normalized_gitignore
        or "data/raw/" in normalized_gitignore
        or bool(
            re.search(
                r"(?m)^\s*data/raw/?\s*$",
                normalized_gitignore,
            )
        )
    ),
}

git_safety = pd.Series(
    checks,
    name="protected",
)

display(git_safety)

failed_git_checks = [
    name
    for name, passed in checks.items()
    if not passed
]

if failed_git_checks:
    raise RuntimeError(
        "Git safety check failed:\n- "
        + "\n- ".join(failed_git_checks)
        + "\n\nFix .gitignore before continuing."
    )

print("Git safety checks passed.")


.env ignored                        True
.venv ignored                       True
raw Databento explicitly ignored    True
Name: protected, dtype: bool

Git safety checks passed.


If this cell fails, fix `.gitignore` before continuing.

A suitable raw-data rule is:

```text
data/raw/databento/
```

Do not commit licensed raw CME / Databento files.


## 2. Freeze the candidate Databento contract

These values are the **candidate provider contract** for v0.1.

Changing them later should be documented rather than silently altering the experiment.

In [5]:
DATASET = "GLBX.MDP3"
SYMBOL = "ES.v.0"
STYPE_IN = "continuous"

PRIMARY_SCHEMA = "ohlcv-1m"
DIAGNOSTIC_SCHEMA = "trades"

ANALYSIS_TIMEZONE = "America/New_York"

# ES minimum price increment, used only to make price differences easier to read.
# It does NOT define Arena settlement.
ES_TICK_SIZE = 0.25

CONFIG = {
    "dataset": DATASET,
    "symbol": SYMBOL,
    "stype_in": STYPE_IN,
    "primary_schema": PRIMARY_SCHEMA,
    "diagnostic_schema": DIAGNOSTIC_SCHEMA,
    "analysis_timezone": ANALYSIS_TIMEZONE,
    "es_tick_size": ES_TICK_SIZE,
}

CONFIG

{'dataset': 'GLBX.MDP3',
 'symbol': 'ES.v.0',
 'stype_in': 'continuous',
 'primary_schema': 'ohlcv-1m',
 'diagnostic_schema': 'trades',
 'analysis_timezone': 'America/New_York',
 'es_tick_size': 0.25}

## 3. Databento metadata-only validation

The following calls inspect schemas, available history, record count, size, and cost.

They do **not** request the historical ES time series.

In [6]:
schemas = metadata_call_with_retry(
    client.metadata.list_schemas,
    dataset=DATASET,
)

assert PRIMARY_SCHEMA in schemas, (
    f"{PRIMARY_SCHEMA} unavailable for {DATASET}"
)

assert DIAGNOSTIC_SCHEMA in schemas, (
    f"{DIAGNOSTIC_SCHEMA} unavailable for {DATASET}"
)


dataset_range = metadata_call_with_retry(
    client.metadata.get_dataset_range,
    dataset=DATASET,
)

primary_range = (
    dataset_range["schema"][PRIMARY_SCHEMA]
)

trades_range = (
    dataset_range["schema"][DIAGNOSTIC_SCHEMA]
)

print(
    "Primary schema range:",
    primary_range,
)

print(
    "Trades range:",
    trades_range,
)

Primary schema range: {'start': '2010-06-06T00:00:00.000000000Z', 'end': '2026-10-17T06:41:46.197093000Z'}
Trades range: {'start': '2010-06-06T00:00:00.000000000Z', 'end': '2026-10-17T06:41:46.197093000Z'}


In [7]:
full_quote_start = primary_range["start"]
full_quote_end = primary_range["end"]


full_cost = metadata_call_with_retry(
    client.metadata.get_cost,
    dataset=DATASET,
    symbols=SYMBOL,
    stype_in=STYPE_IN,
    schema=PRIMARY_SCHEMA,
    start=full_quote_start,
    end=full_quote_end,
)


full_records = metadata_call_with_retry(
    client.metadata.get_record_count,
    dataset=DATASET,
    symbols=SYMBOL,
    stype_in=STYPE_IN,
    schema=PRIMARY_SCHEMA,
    start=full_quote_start,
    end=full_quote_end,
)


full_bytes = metadata_call_with_retry(
    client.metadata.get_billable_size,
    dataset=DATASET,
    symbols=SYMBOL,
    stype_in=STYPE_IN,
    schema=PRIMARY_SCHEMA,
    start=full_quote_start,
    end=full_quote_end,
)


metadata_checkpoint = pd.Series({
    "available_start": full_quote_start,
    "available_end": full_quote_end,
    "quoted_cost_usd": full_cost,
    "record_count": full_records,
    "billable_bytes": full_bytes,
    "billable_gb": (
        full_bytes / 1_000_000_000
    ),
})

metadata_checkpoint

available_start    2010-06-06T00:00:00.000000000Z
available_end      2026-10-17T06:41:46.197093000Z
quoted_cost_usd                         20.887675
record_count                              5721421
billable_bytes                          320399576
billable_gb                                0.3204
dtype: object

# Phase A — Arena ↔ Databento provider validation

## 4. Frozen eight-challenge validation design

The earlier discovery workflow used six fixed anchors and dynamically selected two bearish rows. That was useful while the validation suite was being discovered, but it is no longer appropriate after the provider-validation experiment has been completed.

This version freezes **all eight concrete Headline Arena challenge IDs** before any rerun:

1. keep the eight validation roles fixed,
2. keep the eight challenge IDs fixed,
3. refresh **free public resolved metadata from Headline Arena**,
4. hydrate those same eight challenges from the current public record,
5. recompute every class from open / settlement / dead zone,
6. run a strict integrity/coverage gate,
7. only then use the already-acquired local Databento validation files.

This prevents the provider-validation sample from drifting over time.

### Frozen validation roles

| Role | Purpose |
|---|---|
| `clear_bullish` | clear positive ordinary-session reconstruction |
| `bullish_boundary_stress` | bullish result relatively close to +dead-zone boundary |
| `clear_bearish` | clear negative ordinary-session reconstruction |
| `clear_bearish_replication` | independent negative reconstruction |
| `clear_neutral` | interior neutral case |
| `clear_neutral_replication` | independent neutral case |
| `roll_transition` | quarterly-roll / contract-transition behavior |
| `post_roll_contract_timing` | post-roll contract mapping + timing edge |

This preserves the intended **2 bullish + 2 bearish + 2 neutral + 1 roll + 1 special/timing-edge** design.

The eighth structural role is a real post-roll contract/timing check rather than a fabricated holiday example. Holiday/shortened sessions remain a required full-history quality audit later in Notebook 08.

### Source rules

- Headline Arena individual challenge rules are authoritative.
- Recent selected ES challenges use the ±0.30% dead zone; exact ±boundary is neutral.
- The notebook inspects a `dead_zone_pct` field if the public API provides one and fails closed if it conflicts with the frozen v0.1 contract.
- The public API must still prove that the frozen bearish IDs are actually bearish when the notebook reruns.
- Public Arena metadata refresh is **not a Databento market-data request** and creates no Databento data charge.


In [8]:
ARENA_RESOLVED_API = "https://headlinearena.com/api/v1/public/challenges"
ARENA_METHODOLOGY_URL = "https://headlinearena.com/methodology"

EXPECTED_VALIDATION_COUNT = 8
ES_DEAD_ZONE_PCT_V0_1 = 0.30

# All eight challenge IDs are now frozen.
# Public Arena metadata is refreshed on rerun, but the validation sample
# itself must not drift after provider validation has been completed.
FIXED_ARENA_ANCHORS = [
    {
        "validation_role": "clear_bullish",
        "challenge_id": "06bdca67-a2d1-4356-96f9-172aa26ca66c",
        "selection_reason": (
            "Clear bullish resolved ES session; independent positive plumbing check."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "bullish_boundary_stress",
        "challenge_id": "46200d1b-2d69-40a8-b226-e8daff79b307",
        "selection_reason": (
            "Resolved bullish case near the +dead-zone boundary; "
            "stresses settlement precision."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "clear_bearish",
        "challenge_id": "097db04f-3dc6-49f6-90c5-8634ac139a25",
        "selection_reason": (
            "Frozen resolved bearish ES challenge used in the completed "
            "Databento provider-validation suite."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "clear_bearish_replication",
        "challenge_id": "2f41bc55-2448-4955-9f5b-7aa86c053b35",
        "selection_reason": (
            "Frozen independent bearish ES challenge used in the completed "
            "Databento provider-validation suite."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "clear_neutral",
        "challenge_id": "e4643e45-13a6-43d1-aa5f-1c3df2541f71",
        "selection_reason": (
            "Interior neutral case; tests the center of the dead zone."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "clear_neutral_replication",
        "challenge_id": "167c568c-3bbc-43e4-91e7-bf0bf1896487",
        "selection_reason": (
            "Independent neutral case from a different month."
        ),
        "session_type": "normal",
        "expected_contract": None,
    },
    {
        "validation_role": "roll_transition",
        "challenge_id": "b4c1d697-57ea-44a4-b7b8-fb9e75ee864c",
        "selection_reason": (
            "September quarterly-roll vicinity; tests ES.v.0 contract transition."
        ),
        "session_type": "roll_adjacent",
        "expected_contract": None,
    },
    {
        "validation_role": "post_roll_contract_timing",
        "challenge_id": "82629b32-f54a-460f-ae9d-b8386c9608be",
        "selection_reason": (
            "Post-roll session publicly identified as ESZ6; tests contract mapping "
            "and normal 18:00→17:00 timing after the roll."
        ),
        "session_type": "post_roll",
        "expected_contract": "ESZ6",
    },
]

VALIDATION_ROLE_ORDER = [
    "clear_bullish",
    "bullish_boundary_stress",
    "clear_bearish",
    "clear_bearish_replication",
    "clear_neutral",
    "clear_neutral_replication",
    "roll_transition",
    "post_roll_contract_timing",
]

pd.DataFrame(FIXED_ARENA_ANCHORS)[
    ["validation_role", "challenge_id", "selection_reason"]
]


,validation_role,challenge_id,selection_reason
0,clear_bullish,06bdca67-a2d1-4356-96f9-172aa26ca66c,Clear bullish resolved ES session; independent...
1,bullish_boundary_stress,46200d1b-2d69-40a8-b226-e8daff79b307,Resolved bullish case near the +dead-zone boun...
2,clear_bearish,097db04f-3dc6-49f6-90c5-8634ac139a25,Frozen resolved bearish ES challenge used in t...
3,clear_bearish_replication,2f41bc55-2448-4955-9f5b-7aa86c053b35,Frozen independent bearish ES challenge used i...
4,clear_neutral,e4643e45-13a6-43d1-aa5f-1c3df2541f71,Interior neutral case; tests the center of the...
5,clear_neutral_replication,167c568c-3bbc-43e4-91e7-bf0bf1896487,Independent neutral case from a different month.
6,roll_transition,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,September quarterly-roll vicinity; tests ES.v....
7,post_roll_contract_timing,82629b32-f54a-460f-ae9d-b8386c9608be,Post-roll session publicly identified as ESZ6;...


### 4.1 Refresh the public Headline Arena resolved record

This step uses **free public Arena metadata only**.

The function pages through resolved challenges until all **eight frozen challenge IDs** have been found.

The sample itself is no longer selected dynamically. The API is used only to refresh and verify the authoritative resolved fields for those same eight challenges.

If Arena is temporarily unavailable, or if any frozen challenge cannot be found, stop and retry later. Do not replace missing values with guesses.


In [9]:
def _arena_get_json(params, timeout=30):
    query = urllib.parse.urlencode(params)
    url = f"{ARENA_RESOLVED_API}?{query}"

    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Market-Mood-Forecasting/0.1 provider-validation"
        },
    )

    with urllib.request.urlopen(request, timeout=timeout) as response:
        payload = json.loads(response.read().decode("utf-8"))

    if not isinstance(payload, dict) or "items" not in payload:
        raise RuntimeError(
            "Unexpected Headline Arena public API response shape."
        )

    return payload


def fetch_resolved_arena_es(
    page_size=100,
    max_requests=80,
    sleep_seconds=0.05,
):
    anchor_ids = {
        row["challenge_id"]
        for row in FIXED_ARENA_ANCHORS
    }

    offset = 0
    all_es = []
    seen_ids = set()

    for request_number in range(1, max_requests + 1):
        payload = _arena_get_json({
            "status": "resolved",
            "limit": page_size,
            "offset": offset,
        })

        items = payload.get("items", [])
        total = payload.get("total")

        if not items:
            break

        for item in items:
            if item.get("asset") != "ES":
                continue

            cid = item.get("id")
            if not cid or cid in seen_ids:
                continue

            seen_ids.add(cid)
            all_es.append(item)

        found_anchors = anchor_ids.intersection(seen_ids)

        if len(found_anchors) == len(anchor_ids):
            break

        # Some APIs cap the requested page size. Advance by actual rows returned.
        offset += len(items)

        if total is not None and offset >= int(total):
            break

        if sleep_seconds:
            time.sleep(sleep_seconds)

    es = pd.DataFrame(all_es)

    if es.empty:
        raise RuntimeError(
            "No resolved ES records were returned by Headline Arena."
        )

    if "id" not in es.columns:
        raise RuntimeError(
            "Resolved Arena response did not include challenge IDs."
        )

    es = es.drop_duplicates("id").reset_index(drop=True)

    missing_anchors = anchor_ids - set(es["id"])
    if missing_anchors:
        raise RuntimeError(
            "Could not find all eight frozen Arena anchors before the "
            f"pagination limit. Missing: {sorted(missing_anchors)}"
        )

    return es


# FREE PUBLIC-METADATA STEP — no Databento market data downloaded:
arena_es_resolved = fetch_resolved_arena_es()

print(
    f"Resolved ES records fetched from Arena: {len(arena_es_resolved)}"
)
print(
    "Outcome counts:",
    arena_es_resolved["result"]
    .astype(str)
    .str.lower()
    .value_counts()
    .to_dict(),
)


Resolved ES records fetched from Arena: 75
Outcome counts: {'neutral': 27, 'bearish': 26, 'bullish': 22}


## 5. Build the frozen eight-challenge Arena validation suite

The final table is composed of the **same eight challenge IDs that were used in the completed provider-validation experiment**.

There is no dynamic bearish selection on rerun.

For every frozen challenge, the notebook:

1. retrieves the current public resolved Arena record,
2. requires non-null open and settlement prices,
3. normalizes the challenge dead zone,
4. reconstructs the intended 18:00 ET → 17:00 ET session bounds,
5. recomputes the class from open / settlement / dead zone,
6. later verifies that the resolved outcome still matches the frozen role.

This means a fresh rerun cannot silently substitute newer bearish challenges for the already-validated ones.


In [10]:
VALID_OUTCOMES = {"bearish", "neutral", "bullish"}


def pct_points_to_decimal(dead_zone_pct):
    return float(dead_zone_pct) / 100.0


def classify_return(session_return, dead_zone_pct):
    band = pct_points_to_decimal(dead_zone_pct)

    if session_return > band:
        return "bullish"
    if session_return < -band:
        return "bearish"
    return "neutral"


# Exact +/- boundary belongs to Neutral.
assert classify_return(0.0040, 0.30) == "bullish"
assert classify_return(-0.0040, 0.30) == "bearish"
assert classify_return(0.0030, 0.30) == "neutral"
assert classify_return(-0.0030, 0.30) == "neutral"

print("Three-class target unit checks passed.")

Three-class target unit checks passed.


In [11]:
def _normal_es_session_bounds(resolve_at):
    """
    Build explicit New York session boundaries from Arena's resolution date.

    Normal US futures product day:
      previous calendar day 18:00 ET -> resolution day 17:00 ET.

    Holiday-shortened historical sessions are audited separately later.
    """
    resolve_utc = pd.Timestamp(resolve_at)

    if resolve_utc.tzinfo is None:
        resolve_utc = resolve_utc.tz_localize("UTC")
    else:
        resolve_utc = resolve_utc.tz_convert("UTC")

    resolve_et = resolve_utc.tz_convert(ANALYSIS_TIMEZONE)
    session_date = resolve_et.date()

    session_end_et = pd.Timestamp(
        f"{session_date} 17:00:00",
        tz=ANALYSIS_TIMEZONE,
    )

    previous_date = (
        pd.Timestamp(session_date) - pd.Timedelta(days=1)
    ).date()

    session_start_et = pd.Timestamp(
        f"{previous_date} 18:00:00",
        tz=ANALYSIS_TIMEZONE,
    )

    return session_start_et, session_end_et


def _normalize_arena_dead_zone_pct(record):
    """
    Use challenge-specific API value if exposed; otherwise use the frozen
    0.30% contract for this selected recent ES suite.

    Supports either 0.30 (percentage points) or 0.003 (decimal return)
    representation if the API changes representation.
    """
    raw = record.get("dead_zone_pct")

    if raw is None or pd.isna(raw):
        return ES_DEAD_ZONE_PCT_V0_1, None

    value = float(raw)

    # If represented as decimal return, convert 0.003 -> 0.30 percentage points.
    normalized = value * 100.0 if abs(value) < 0.01 else value

    if not np.isclose(
        normalized,
        ES_DEAD_ZONE_PCT_V0_1,
        atol=1e-9,
        rtol=0,
    ):
        raise RuntimeError(
            "Selected Arena challenge uses a different dead zone: "
            f"raw={raw}, normalized={normalized}%. "
            "Do not silently mix target contracts."
        )

    return normalized, raw


def _hydrate_anchor(anchor, arena_index):
    cid = anchor["challenge_id"]

    if cid not in arena_index.index:
        raise RuntimeError(f"Arena anchor not found: {cid}")

    record = arena_index.loc[cid]

    open_price = record.get("open_price")
    close_price = record.get("close_price")
    outcome = str(record.get("result", "")).lower()

    if (
        pd.isna(open_price)
        or pd.isna(close_price)
        or outcome not in VALID_OUTCOMES
    ):
        raise RuntimeError(
            f"Arena anchor {cid} does not have complete resolved fields."
        )

    dead_zone_pct, dead_zone_raw = _normalize_arena_dead_zone_pct(
        record
    )

    start_et, end_et = _normal_es_session_bounds(
        record["resolve_at"]
    )

    return {
        **anchor,
        "challenge_date": str(end_et.date()),
        "session_start_et": start_et,
        "session_end_et": end_et,
        "dead_zone_pct": dead_zone_pct,
        "arena_dead_zone_raw": dead_zone_raw,
        "arena_open": float(open_price),
        "arena_settlement": float(close_price),
        "arena_outcome": outcome,
        "arena_created_at": record.get("created_at"),
        "arena_deadline_utc": record.get("deadline"),
        "arena_resolve_at_utc": record.get("resolve_at"),
        "arena_resolved_at_utc": record.get("resolved_at"),
        "source_status": "hydrated_from_public_resolved_api",
        "ready_for_databento": True,
        "source_url": f"https://headlinearena.com/prediction/{cid}",
    }


def build_final_arena_validation_suite(arena_es):
    arena_index = (
        arena_es
        .drop_duplicates("id")
        .set_index("id", drop=False)
    )

    rows = [
        _hydrate_anchor(
            anchor,
            arena_index,
        )
        for anchor in FIXED_ARENA_ANCHORS
    ]

    final = pd.DataFrame(rows)

    final["role_order"] = pd.Categorical(
        final["validation_role"],
        categories=VALIDATION_ROLE_ORDER,
        ordered=True,
    )

    final = (
        final
        .sort_values("role_order")
        .drop(columns="role_order")
        .reset_index(drop=True)
    )

    return final


arena_validation = build_final_arena_validation_suite(
    arena_es_resolved
)

arena_validation[
    [
        "validation_role",
        "challenge_id",
        "challenge_date",
        "arena_open",
        "arena_settlement",
        "arena_outcome",
        "session_type",
        "expected_contract",
        "source_status",
    ]
]


,validation_role,challenge_id,challenge_date,arena_open,arena_settlement,arena_outcome,session_type,expected_contract,source_status
0,clear_bullish,06bdca67-a2d1-4356-96f9-172aa26ca66c,2026-09-25,7759.00,7805.75,bullish,normal,NaN,hydrated_from_public_resolved_api
1,bullish_boundary_stress,46200d1b-2d69-40a8-b226-e8daff79b307,2026-07-15,7589.75,7616.00,bullish,normal,NaN,hydrated_from_public_resolved_api
2,clear_bearish,097db04f-3dc6-49f6-90c5-8634ac139a25,2026-09-28,7796.00,7746.25,bearish,normal,NaN,hydrated_from_public_resolved_api
3,clear_bearish_replication,2f41bc55-2448-4955-9f5b-7aa86c053b35,2026-09-23,7830.00,7772.75,bearish,normal,NaN,hydrated_from_public_resolved_api
4,clear_neutral,e4643e45-13a6-43d1-aa5f-1c3df2541f71,2026-07-01,7543.25,7532.75,neutral,normal,NaN,hydrated_from_public_resolved_api
5,clear_neutral_replication,167c568c-3bbc-43e4-91e7-bf0bf1896487,2026-08-28,7733.75,7724.75,neutral,normal,NaN,hydrated_from_public_resolved_api
6,roll_transition,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,2026-09-18,7703.50,7725.00,neutral,roll_adjacent,NaN,hydrated_from_public_resolved_api
7,post_roll_contract_timing,82629b32-f54a-460f-ae9d-b8386c9608be,2026-09-29,7746.50,7738.25,neutral,post_roll,ESZ6,hydrated_from_public_resolved_api


### 5.1 Strict eight-row integrity and coverage gate

The gate requires:

- exactly 8 rows,
- exactly the eight frozen validation roles,
- exactly the eight frozen challenge IDs,
- unique challenge IDs,
- all rows ready,
- complete open / settlement / outcome fields,
- both bullish-role rows actually bullish,
- both bearish-role rows actually bearish,
- both neutral-role rows actually neutral,
- one roll-transition row,
- one post-roll timing/contract row,
- every Arena outcome reproduced from open / settlement / dead zone,
- timezone-aware New York session boundaries.

If any requirement fails, `arena_validation_clean` is not created and the Databento validation workflow cannot proceed.


In [12]:
EXPECTED_ROLES = set(VALIDATION_ROLE_ORDER)

ROLE_OUTCOME_REQUIREMENTS = {
    "clear_bullish": "bullish",
    "bullish_boundary_stress": "bullish",
    "clear_bearish": "bearish",
    "clear_bearish_replication": "bearish",
    "clear_neutral": "neutral",
    "clear_neutral_replication": "neutral",
}


def validate_arena_table(df):
    required = [
        "challenge_id",
        "challenge_date",
        "session_start_et",
        "session_end_et",
        "dead_zone_pct",
        "arena_open",
        "arena_settlement",
        "arena_outcome",
        "validation_role",
        "session_type",
        "expected_contract",
        "selection_reason",
        "source_status",
        "ready_for_databento",
        "source_url",
    ]

    missing_columns = [c for c in required if c not in df.columns]
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    if len(df) != EXPECTED_VALIDATION_COUNT:
        raise ValueError(
            f"Expected exactly {EXPECTED_VALIDATION_COUNT} challenges; "
            f"found {len(df)}."
        )

    if df["challenge_id"].duplicated().any():
        raise ValueError("Duplicate challenge IDs detected.")

    actual_roles = set(df["validation_role"])
    if actual_roles != EXPECTED_ROLES:
        raise ValueError(
            "Validation-role coverage is wrong.\n"
            f"Expected: {sorted(EXPECTED_ROLES)}\n"
            f"Actual:   {sorted(actual_roles)}"
        )

    expected_ids = {
        row["challenge_id"]
        for row in FIXED_ARENA_ANCHORS
    }
    actual_ids = set(df["challenge_id"])

    if actual_ids != expected_ids:
        raise ValueError(
            "Frozen challenge-ID coverage is wrong.\n"
            f"Missing: {sorted(expected_ids - actual_ids)}\n"
            f"Unexpected: {sorted(actual_ids - expected_ids)}"
        )

    if not df["ready_for_databento"].all():
        raise RuntimeError(
            "At least one Arena row is not ready for Databento."
        )

    if (
        df["arena_open"].isna().any()
        or df["arena_settlement"].isna().any()
        or df["arena_outcome"].isna().any()
    ):
        raise ValueError(
            "Missing authoritative Arena open/settlement/outcome."
        )

    if (
        (df["arena_open"] <= 0).any()
        or (df["arena_settlement"] <= 0).any()
    ):
        raise ValueError("Arena prices must be positive.")

    out = df.copy()

    # Already timezone-aware from the suite builder; normalize robustly.
    out["session_start_et"] = pd.to_datetime(
        out["session_start_et"],
        utc=True,
    ).dt.tz_convert(ANALYSIS_TIMEZONE)

    out["session_end_et"] = pd.to_datetime(
        out["session_end_et"],
        utc=True,
    ).dt.tz_convert(ANALYSIS_TIMEZONE)

    if (out["session_end_et"] <= out["session_start_et"]).any():
        raise ValueError("Invalid session boundary ordering.")

    out["session_start_utc"] = (
        out["session_start_et"].dt.tz_convert("UTC")
    )
    out["session_end_utc"] = (
        out["session_end_et"].dt.tz_convert("UTC")
    )

    out["arena_outcome"] = (
        out["arena_outcome"].astype(str).str.lower()
    )

    invalid_outcomes = set(out["arena_outcome"]) - VALID_OUTCOMES
    if invalid_outcomes:
        raise ValueError(
            f"Invalid Arena outcomes: {invalid_outcomes}"
        )

    out["arena_return"] = (
        out["arena_settlement"] / out["arena_open"] - 1.0
    )
    out["arena_return_pct_recomputed"] = (
        out["arena_return"] * 100.0
    )

    out["arena_outcome_recomputed"] = [
        classify_return(r, b)
        for r, b in zip(
            out["arena_return"],
            out["dead_zone_pct"],
        )
    ]

    out["arena_internal_match"] = (
        out["arena_outcome"] == out["arena_outcome_recomputed"]
    )

    if not out["arena_internal_match"].all():
        bad = out.loc[
            ~out["arena_internal_match"],
            [
                "challenge_id",
                "validation_role",
                "arena_open",
                "arena_settlement",
                "arena_return_pct_recomputed",
                "dead_zone_pct",
                "arena_outcome",
                "arena_outcome_recomputed",
            ],
        ]
        raise ValueError(
            "Arena outcome/open/settlement inconsistency:\n"
            + bad.to_string(index=False)
        )

    for role, expected_outcome in ROLE_OUTCOME_REQUIREMENTS.items():
        row = out.loc[out["validation_role"] == role]

        if len(row) != 1:
            raise ValueError(
                f"Expected exactly one row for role {role}."
            )

        actual = row.iloc[0]["arena_outcome"]
        if actual != expected_outcome:
            raise ValueError(
                f"{role} must be {expected_outcome}, got {actual}."
            )

    for structural_role in [
        "roll_transition",
        "post_roll_contract_timing",
    ]:
        if (out["validation_role"] == structural_role).sum() != 1:
            raise ValueError(
                f"Expected exactly one {structural_role} row."
            )

    print("STRICT ARENA VALIDATION GATE: PASS")
    print("Rows:", len(out))
    print(
        "Outcome mix:",
        out["arena_outcome"].value_counts().to_dict(),
    )
    print(
        "Roles:",
        out["validation_role"].tolist(),
    )

    return out.reset_index(drop=True)


# This is the ONLY dataframe allowed into Databento quote/download cells.
arena_validation_clean = validate_arena_table(
    arena_validation
)

arena_validation_clean[[
    "validation_role",
    "challenge_id",
    "challenge_date",
    "arena_open",
    "arena_settlement",
    "arena_return_pct_recomputed",
    "dead_zone_pct",
    "arena_outcome",
    "session_type",
    "expected_contract",
]]

STRICT ARENA VALIDATION GATE: PASS
Rows: 8
Outcome mix: {'neutral': 4, 'bullish': 2, 'bearish': 2}
Roles: ['clear_bullish', 'bullish_boundary_stress', 'clear_bearish', 'clear_bearish_replication', 'clear_neutral', 'clear_neutral_replication', 'roll_transition', 'post_roll_contract_timing']


,validation_role,challenge_id,challenge_date,arena_open,arena_settlement,arena_return_pct_recomputed,dead_zone_pct,arena_outcome,session_type,expected_contract
0,clear_bullish,06bdca67-a2d1-4356-96f9-172aa26ca66c,2026-09-25,7759.00,7805.75,0.602526,0.3,bullish,normal,NaN
1,bullish_boundary_stress,46200d1b-2d69-40a8-b226-e8daff79b307,2026-07-15,7589.75,7616.00,0.345861,0.3,bullish,normal,NaN
2,clear_bearish,097db04f-3dc6-49f6-90c5-8634ac139a25,2026-09-28,7796.00,7746.25,-0.638148,0.3,bearish,normal,NaN
3,clear_bearish_replication,2f41bc55-2448-4955-9f5b-7aa86c053b35,2026-09-23,7830.00,7772.75,-0.731162,0.3,bearish,normal,NaN
4,clear_neutral,e4643e45-13a6-43d1-aa5f-1c3df2541f71,2026-07-01,7543.25,7532.75,-0.139197,0.3,neutral,normal,NaN
5,clear_neutral_replication,167c568c-3bbc-43e4-91e7-bf0bf1896487,2026-08-28,7733.75,7724.75,-0.116373,0.3,neutral,normal,NaN
6,roll_transition,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,2026-09-18,7703.50,7725.00,0.279094,0.3,neutral,roll_adjacent,NaN
7,post_roll_contract_timing,82629b32-f54a-460f-ae9d-b8386c9608be,2026-09-29,7746.50,7738.25,-0.106500,0.3,neutral,post_roll,ESZ6


## 6. Optional validation-quote refresh

The eight validation challenge files have already been acquired,
validated, and stored locally.

A normal reproducibility rerun should **not** repeatedly request
eight separate Databento quotes.

Therefore:

- the quote helper remains available,
- quote refresh is disabled by default,
- local validation files are reused,
- no validation data is purchased again.

Set `REFRESH_VALIDATION_QUOTE = True` only when intentionally
refreshing these metadata quotes.

In [13]:
def quote_validation_challenges(
    arena_df,
    buffer_minutes=5,
):
    if (
        len(arena_df)
        != EXPECTED_VALIDATION_COUNT
    ):
        raise RuntimeError(
            f"Refusing to quote: expected "
            f"{EXPECTED_VALIDATION_COUNT} "
            f"validated challenges, "
            f"got {len(arena_df)}."
        )

    rows = []

    for _, row in arena_df.iterrows():
        start = (
            row["session_start_utc"]
            - pd.Timedelta(
                minutes=buffer_minutes
            )
        )

        end = (
            row["session_end_utc"]
            + pd.Timedelta(
                minutes=buffer_minutes
            )
        )

        request_kwargs = {
            "dataset": DATASET,
            "symbols": SYMBOL,
            "stype_in": STYPE_IN,
            "schema": PRIMARY_SCHEMA,
            "start": start,
            "end": end,
        }

        cost = metadata_call_with_retry(
            client.metadata.get_cost,
            **request_kwargs,
        )

        record_count = (
            metadata_call_with_retry(
                client.metadata.get_record_count,
                **request_kwargs,
            )
        )

        rows.append({
            "validation_role":
                row["validation_role"],

            "challenge_id":
                row["challenge_id"],

            "challenge_date":
                row["challenge_date"],

            "request_start_utc":
                start,

            "request_end_utc":
                end,

            "quoted_cost_usd":
                cost,

            "record_count":
                record_count,
        })

    quotes = pd.DataFrame(rows)

    total = (
        quotes["quoted_cost_usd"]
        .sum()
    )

    print(
        "Total quoted validation cost: "
        f"${total:.6f}"
    )

    return quotes


# ------------------------------------------------------------
# Normal reruns do NOT need to re-quote the eight already
# acquired validation files.
# ------------------------------------------------------------

REFRESH_VALIDATION_QUOTE = False


if REFRESH_VALIDATION_QUOTE:
    validation_quotes = (
        quote_validation_challenges(
            arena_validation_clean
        )
    )

    display(validation_quotes)

else:
    validation_quotes = None

    print(
        "Validation re-quote skipped. "
        "The eight validation files "
        "already exist locally."
    )

Validation re-quote skipped. The eight validation files already exist locally.


## 7. One-time Databento validation pull — already completed

The eight small validation files were downloaded once and are now
stored locally.

The paid-download function is retained only for reproducibility and
documentation.

For normal notebook reruns:

- keep `ENABLE_VALIDATION_DOWNLOAD = False`,
- do not call the paid download function,
- reuse the existing local files in `data/raw/databento/`.

The next subsection verifies that all eight expected local files exist.

In [14]:
def safe_challenge_filename(challenge_id):
    safe = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(challenge_id),
    )
    return safe[:120]


def download_validation_challenges(
    arena_df,
    buffer_minutes=5,
):
    if len(arena_df) != EXPECTED_VALIDATION_COUNT:
        raise RuntimeError(
            f"Refusing paid validation pull: expected "
            f"{EXPECTED_VALIDATION_COUNT} validated challenges, "
            f"got {len(arena_df)}."
        )

    if not ENABLE_VALIDATION_DOWNLOAD:
        raise RuntimeError(
            "Validation download is disabled. "
            "Review validation_quotes, then explicitly enable it."
        )

    downloaded = []

    for _, row in arena_df.iterrows():
        start = row["session_start_utc"] - pd.Timedelta(
            minutes=buffer_minutes
        )
        end = row["session_end_utc"] + pd.Timedelta(
            minutes=buffer_minutes
        )

        cost = client.metadata.get_cost(
            dataset=DATASET,
            symbols=SYMBOL,
            stype_in=STYPE_IN,
            schema=PRIMARY_SCHEMA,
            start=start,
            end=end,
        )

        filename = (
            f"arena_{safe_challenge_filename(row['challenge_id'])}"
            f".{PRIMARY_SCHEMA}.dbn.zst"
        )
        path = RAW_DIR / filename

        print(
            f"{row['validation_role']} | "
            f"{row['challenge_id']}: ${cost:.6f} "
            f"| {start} -> {end} | {path.name}"
        )

        if path.exists():
            print(
                "  Local file already exists — "
                "skipping paid re-download."
            )
            downloaded.append(path)
            continue

        client.timeseries.get_range(
            dataset=DATASET,
            symbols=SYMBOL,
            stype_in=STYPE_IN,
            schema=PRIMARY_SCHEMA,
            start=start,
            end=end,
            path=str(path),
        )

        downloaded.append(path)

    return downloaded


# One-time acquisition only:
#
# ENABLE_VALIDATION_DOWNLOAD = True
# validation_paths = download_validation_challenges(
#     arena_validation_clean
# )
# ENABLE_VALIDATION_DOWNLOAD = False

### 7.1 Reuse the already-downloaded validation files

The eight paid validation pulls have already been acquired once.

Normal notebook reruns must **reuse the local DBN/Zstd files** and must not require
`ENABLE_VALIDATION_DOWNLOAD=True`. If a file is missing, fail closed and decide
deliberately whether it should be re-acquired.

In [15]:
validation_paths = [
    RAW_DIR
    / (
        f"arena_{safe_challenge_filename(row['challenge_id'])}"
        f".{PRIMARY_SCHEMA}.dbn.zst"
    )
    for _, row in arena_validation_clean.iterrows()
]

missing_validation_files = [
    path
    for path in validation_paths
    if not path.exists()
]

if missing_validation_files:
    raise FileNotFoundError(
        "Expected local validation files are missing:\n"
        + "\n".join(str(path) for path in missing_validation_files)
        + "\n\nRe-enable the paid validation pull only after reviewing "
          "the quote and confirming that re-acquisition is intentional."
    )

assert len(validation_paths) == EXPECTED_VALIDATION_COUNT

print(
    f"Using {len(validation_paths)} existing local "
    "Databento validation files."
)


Using 8 existing local Databento validation files.


## 8. Load the saved validation files locally

After the small paid pull, all subsequent validation work should read the local DBN files instead of requesting the same data again.

In [16]:
def load_local_dbn(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    store = db.DBNStore.from_file(path)

    return store.to_df(
        pretty_ts=True,
        map_symbols=True,
        tz="UTC",
    )


def load_validation_bundle(arena_df):
    frames = []

    for _, row in arena_df.iterrows():
        filename = (
            f"arena_{safe_challenge_filename(row['challenge_id'])}"
            f".{PRIMARY_SCHEMA}.dbn.zst"
        )
        path = RAW_DIR / filename
        df = load_local_dbn(path)
        df = df.copy()
        df["source_challenge_id"] = row["challenge_id"]
        frames.append(df)

    combined = pd.concat(frames).sort_index()
    return combined


es_validation_utc = load_validation_bundle(arena_validation_clean)
display(es_validation_utc.head())
print(es_validation_utc.shape)

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,source_challenge_id
ts_event,,,,,,,,,,
2026-06-30 22:00:00+00:00,33,1,42140870,7543.25,7546.75,7540.50,7546.25,1030,ES.v.0,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:01:00+00:00,33,1,42140870,7546.50,7546.50,7543.00,7543.50,576,ES.v.0,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:02:00+00:00,33,1,42140870,7543.50,7545.75,7543.25,7545.50,257,ES.v.0,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:03:00+00:00,33,1,42140870,7545.25,7546.25,7544.25,7544.50,383,ES.v.0,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:04:00+00:00,33,1,42140870,7544.00,7545.50,7544.00,7545.00,135,ES.v.0,e4643e45-13a6-43d1-aa5f-1c3df2541f71


(11040, 10)


In [17]:
instrument_mapping = (
    es_validation_utc
    .groupby(
        ["source_challenge_id", "instrument_id"]
    )
    .size()
    .rename("rows")
    .reset_index()
)

display(instrument_mapping)

,source_challenge_id,instrument_id,rows
0,06bdca67-a2d1-4356-96f9-172aa26ca66c,10252,1380
1,097db04f-3dc6-49f6-90c5-8634ac139a25,10252,1380
2,167c568c-3bbc-43e4-91e7-bf0bf1896487,42140870,1380
3,2f41bc55-2448-4955-9f5b-7aa86c053b35,10252,1380
4,46200d1b-2d69-40a8-b226-e8daff79b307,42140870,1380
5,82629b32-f54a-460f-ae9d-b8386c9608be,10252,1380
6,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,10252,1380
7,e4643e45-13a6-43d1-aa5f-1c3df2541f71,42140870,1380


In [18]:
unique_instrument_ids = (
    es_validation_utc["instrument_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("Instrument IDs:", unique_instrument_ids)

symbology_result = client.symbology.resolve(
    dataset=DATASET,
    symbols=unique_instrument_ids,
    stype_in="instrument_id",
    stype_out="raw_symbol",
    start_date="2026-06-30",
    end_date="2026-10-01",
)

symbology_result

Instrument IDs: [42140870, 10252]


{'result': {'42140870': [{'d0': '2026-06-30',
    'd1': '2026-10-01',
    's': 'ESU6'}],
  '10252': [{'d0': '2026-06-30', 'd1': '2026-10-01', 's': 'ESZ6'}]},
 'symbols': ['42140870', '10252'],
 'stype_in': 'instrument_id',
 'stype_out': 'raw_symbol',
 'start_date': '2026-06-30',
 'end_date': '2026-10-01',
 'partial': [],
 'not_found': [],
 'message': 'OK',
 'status': 0}

In [19]:
instrument_to_raw_symbol = {}

for instrument_id, mappings in symbology_result["result"].items():
    if len(mappings) != 1:
        raise ValueError(
            f"Expected one raw-symbol mapping for {instrument_id}, "
            f"got {len(mappings)}"
        )

    instrument_to_raw_symbol[int(instrument_id)] = mappings[0]["s"]

instrument_to_raw_symbol

{42140870: 'ESU6', 10252: 'ESZ6'}

In [20]:
es_validation_utc = es_validation_utc.copy()

es_validation_utc["raw_symbol"] = (
    es_validation_utc["instrument_id"]
    .astype(int)
    .map(instrument_to_raw_symbol)
)

assert es_validation_utc["raw_symbol"].notna().all()

display(
    es_validation_utc[
        [
            "instrument_id",
            "symbol",
            "raw_symbol",
            "source_challenge_id",
        ]
    ].head()
)

,instrument_id,symbol,raw_symbol,source_challenge_id
ts_event,,,,
2026-06-30 22:00:00+00:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:01:00+00:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:02:00+00:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:03:00+00:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 22:04:00+00:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71


In [21]:
contract_mapping = (
    es_validation_utc
    .groupby(
        [
            "source_challenge_id",
            "instrument_id",
            "raw_symbol",
        ]
    )
    .size()
    .rename("rows")
    .reset_index()
)

contract_mapping = contract_mapping.merge(
    arena_validation_clean[
        [
            "challenge_id",
            "challenge_date",
            "validation_role",
            "expected_contract",
        ]
    ],
    left_on="source_challenge_id",
    right_on="challenge_id",
    how="left",
)

contract_mapping = contract_mapping[
    [
        "validation_role",
        "challenge_date",
        "source_challenge_id",
        "instrument_id",
        "raw_symbol",
        "expected_contract",
        "rows",
    ]
]

display(contract_mapping)

,validation_role,challenge_date,source_challenge_id,instrument_id,raw_symbol,expected_contract,rows
0,clear_bullish,2026-09-25,06bdca67-a2d1-4356-96f9-172aa26ca66c,10252,ESZ6,NaN,1380
1,clear_bearish,2026-09-28,097db04f-3dc6-49f6-90c5-8634ac139a25,10252,ESZ6,NaN,1380
2,clear_neutral_replication,2026-08-28,167c568c-3bbc-43e4-91e7-bf0bf1896487,42140870,ESU6,NaN,1380
3,clear_bearish_replication,2026-09-23,2f41bc55-2448-4955-9f5b-7aa86c053b35,10252,ESZ6,NaN,1380
4,bullish_boundary_stress,2026-07-15,46200d1b-2d69-40a8-b226-e8daff79b307,42140870,ESU6,NaN,1380
5,post_roll_contract_timing,2026-09-29,82629b32-f54a-460f-ae9d-b8386c9608be,10252,ESZ6,ESZ6,1380
6,roll_transition,2026-09-18,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,10252,ESZ6,NaN,1380
7,clear_neutral,2026-07-01,e4643e45-13a6-43d1-aa5f-1c3df2541f71,42140870,ESU6,NaN,1380


In [22]:
assert (
    contract_mapping.groupby("source_challenge_id").size() == 1
).all(), "A challenge unexpectedly contains multiple underlying contracts."

expected_rows = contract_mapping[
    contract_mapping["expected_contract"].notna()
]

assert (
    expected_rows["raw_symbol"]
    == expected_rows["expected_contract"]
).all(), "Expected Arena contract does not match Databento raw contract."

print("Contract mapping checks passed.")

Contract mapping checks passed.


## 9. Inspect the underlying contract mapping

`ES.v.0` is a continuous selector. We still inspect which actual ES contract supplied each challenge's observations.

This is particularly important near quarterly rollover.

In [23]:
def inspect_contracts_by_challenge(df):
    group_cols = ["source_challenge_id"]

    if "raw_symbol" in df.columns:
        contract_col = "raw_symbol"
    elif "symbol" in df.columns:
        contract_col = "symbol"
    elif "instrument_id" in df.columns:
        contract_col = "instrument_id"
    else:
        raise ValueError(
            "No raw_symbol, symbol, or instrument_id column is available."
        )

    out = (
        df.groupby(group_cols + [contract_col], dropna=False)
          .size()
          .rename("rows")
          .reset_index()
          .sort_values(
              group_cols + ["rows"],
              ascending=[True, False],
          )
    )

    return out


# Optional audit view:
# display(inspect_contracts_by_challenge(es_validation_utc))


## 10. Convert Databento timestamps to New York time

Always use timezone-aware conversion; never manually subtract four or five hours.

In [24]:
def to_new_york(df_utc):
    out = df_utc.copy()

    if out.index.tz is None:
        raise ValueError("Expected a timezone-aware Databento index.")

    out.index = out.index.tz_convert(ANALYSIS_TIMEZONE)
    return out

In [25]:
# ------------------------------------------------------------
# Convert validation data from UTC to America/New_York
# and verify that contract information is preserved
# ------------------------------------------------------------

assert "raw_symbol" in es_validation_utc.columns, (
    "raw_symbol is missing. Resolve instrument_id -> raw_symbol first."
)

assert es_validation_utc["raw_symbol"].notna().all(), (
    "Some instrument IDs were not resolved to raw CME symbols."
)

es_validation_ny = to_new_york(
    es_validation_utc
)

# Row count must not change
assert len(es_validation_ny) == len(es_validation_utc), (
    "Row count changed during timezone conversion."
)

# Index must remain timezone-aware
assert es_validation_ny.index.tz is not None, (
    "New York index is not timezone-aware."
)

# raw_symbol must survive the copy/conversion
assert "raw_symbol" in es_validation_ny.columns

assert es_validation_ny["raw_symbol"].notna().all()

print("UTC start:", es_validation_utc.index.min())
print("UTC end:  ", es_validation_utc.index.max())

print()

print("NY start: ", es_validation_ny.index.min())
print("NY end:   ", es_validation_ny.index.max())

print()

print("Rows UTC:", len(es_validation_utc))
print("Rows NY: ", len(es_validation_ny))

display(
    es_validation_ny[
        [
            "instrument_id",
            "symbol",
            "raw_symbol",
            "source_challenge_id",
        ]
    ].head()
)

print("Timezone conversion checks passed.")

UTC start: 2026-06-30 22:00:00+00:00
UTC end:   2026-09-29 20:59:00+00:00

NY start:  2026-06-30 18:00:00-04:00
NY end:    2026-09-29 16:59:00-04:00

Rows UTC: 11040
Rows NY:  11040


,instrument_id,symbol,raw_symbol,source_challenge_id
ts_event,,,,
2026-06-30 18:00:00-04:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 18:01:00-04:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 18:02:00-04:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 18:03:00-04:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71
2026-06-30 18:04:00-04:00,42140870,ES.v.0,ESU6,e4643e45-13a6-43d1-aa5f-1c3df2541f71


Timezone conversion checks passed.


## 11. Session reconstruction and boundary audit

For a normal session, our **candidate** minute-bar convention is:

- session open = `open` of the first bar at/after 18:00 ET,
- session close = `close` of the final bar strictly before 17:00 ET.

For a normal 23-hour session, the expected last minute timestamp is typically 16:59 ET.

We do not blindly require exactly 1,380 bars because trade-derived OHLCV may omit a minute with no qualifying events and exceptional sessions can differ. Instead we record diagnostics.

In [26]:
def extract_challenge_session(df_ny, challenge_row):
    challenge_id = challenge_row["challenge_id"]
    start = challenge_row["session_start_et"]
    end = challenge_row["session_end_et"]

    subset = df_ny.loc[
        df_ny["source_challenge_id"] == challenge_id
    ].copy()

    mask = (subset.index >= start) & (subset.index < end)
    session = subset.loc[mask].copy()

    if session.empty:
        raise ValueError(
            f"No Databento bars found for challenge {challenge_id}: "
            f"{start} -> {end}"
        )

    return session


def summarize_challenge_session(session, challenge_row):
    first = session.iloc[0]
    last = session.iloc[-1]

    db_open = float(first["open"])
    db_close = float(last["close"])
    db_return = db_close / db_open - 1.0

    expected_last_bar = (
        challenge_row["session_end_et"]
        - pd.Timedelta(minutes=1)
    )

    symbols = []

    if "raw_symbol" in session.columns:
        symbols = (
            session["raw_symbol"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    elif "symbol" in session.columns:
        symbols = (
            session["symbol"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    return {
        "db_first_bar_ts": session.index[0],
        "db_last_bar_ts": session.index[-1],
        "expected_first_bar_ts": challenge_row["session_start_et"],
        "expected_last_bar_ts": expected_last_bar,
        "first_bar_on_expected_boundary": bool(
            session.index[0] == challenge_row["session_start_et"]
        ),
        "last_bar_on_expected_boundary": bool(
            session.index[-1] == expected_last_bar
        ),
        "db_open": db_open,
        "db_close": db_close,
        "db_return": db_return,
        "db_bar_count": int(len(session)),
        "db_underlying_symbols": symbols,
        "multiple_underlying_symbols": len(symbols) > 1,
    }

## 12. Build the Arena ↔ Databento alignment table

In addition to raw price differences, we calculate:

- price differences in ES ticks,
- return difference in basis points,
- Databento distance from the nearest class boundary,
- class agreement,
- boundary timestamp diagnostics,
- actual underlying contract(s).

A tiny price difference is not automatically a failure. A large unexplained class disagreement on a clearly non-boundary challenge is much more serious.

In [27]:
def reconstruct_alignment_table(arena_df, es_ny):
    rows = []

    for _, challenge in arena_df.iterrows():
        session = extract_challenge_session(es_ny, challenge)
        s = summarize_challenge_session(session, challenge)

        db_outcome = classify_return(
            s["db_return"],
            challenge["dead_zone_pct"],
        )

        band = pct_points_to_decimal(challenge["dead_zone_pct"])

        distance_to_boundary = min(
            abs(s["db_return"] - band),
            abs(s["db_return"] + band),
        )

        open_diff = s["db_open"] - float(challenge["arena_open"])
        settle_diff = s["db_close"] - float(challenge["arena_settlement"])
        return_diff = s["db_return"] - float(challenge["arena_return"])

        row = challenge.to_dict()
        row.update(s)
        row.update({
            "db_outcome": db_outcome,
            "class_match": (
                challenge["arena_outcome"] == db_outcome
            ),
            "open_diff": open_diff,
            "open_diff_ticks": open_diff / ES_TICK_SIZE,
            "settlement_diff": settle_diff,
            "settlement_diff_ticks": settle_diff / ES_TICK_SIZE,
            "return_diff": return_diff,
            "return_diff_bps": return_diff * 10_000,
            "distance_to_boundary": distance_to_boundary,
            "distance_to_boundary_bps": distance_to_boundary * 10_000,
        })

        rows.append(row)

    return pd.DataFrame(rows)


alignment = reconstruct_alignment_table(
     arena_validation_clean,
     es_validation_ny,
 )
alignment

,validation_role,challenge_id,selection_reason,session_type,expected_contract,challenge_date,session_start_et,session_end_et,dead_zone_pct,arena_dead_zone_raw,...,db_outcome,class_match,open_diff,open_diff_ticks,settlement_diff,settlement_diff_ticks,return_diff,return_diff_bps,distance_to_boundary,distance_to_boundary_bps
0,clear_bullish,06bdca67-a2d1-4356-96f9-172aa26ca66c,Clear bullish resolved ES session; independent...,normal,NaN,2026-09-25,2026-09-24 18:00:00-04:00,2026-09-25 17:00:00-04:00,0.3,None,...,bullish,True,0.0,0.0,0.0,0.0,0.0,0.0,0.003025,30.252610
1,bullish_boundary_stress,46200d1b-2d69-40a8-b226-e8daff79b307,Resolved bullish case near the +dead-zone boun...,normal,NaN,2026-07-15,2026-07-14 18:00:00-04:00,2026-07-15 17:00:00-04:00,0.3,None,...,bullish,True,0.0,0.0,0.0,0.0,0.0,0.0,0.000459,4.586119
2,clear_bearish,097db04f-3dc6-49f6-90c5-8634ac139a25,Frozen resolved bearish ES challenge used in t...,normal,NaN,2026-09-28,2026-09-27 18:00:00-04:00,2026-09-28 17:00:00-04:00,0.3,None,...,bearish,True,0.0,0.0,0.0,0.0,0.0,0.0,0.003381,33.814777
3,clear_bearish_replication,2f41bc55-2448-4955-9f5b-7aa86c053b35,Frozen independent bearish ES challenge used i...,normal,NaN,2026-09-23,2026-09-22 18:00:00-04:00,2026-09-23 17:00:00-04:00,0.3,None,...,bearish,True,0.0,0.0,0.0,0.0,0.0,0.0,0.004312,43.116220
4,clear_neutral,e4643e45-13a6-43d1-aa5f-1c3df2541f71,Interior neutral case; tests the center of the...,normal,NaN,2026-07-01,2026-06-30 18:00:00-04:00,2026-07-01 17:00:00-04:00,0.3,None,...,neutral,True,0.0,0.0,0.0,0.0,0.0,0.0,0.001608,16.080270
5,clear_neutral_replication,167c568c-3bbc-43e4-91e7-bf0bf1896487,Independent neutral case from a different month.,normal,NaN,2026-08-28,2026-08-27 18:00:00-04:00,2026-08-28 17:00:00-04:00,0.3,None,...,neutral,True,0.0,0.0,0.0,0.0,0.0,0.0,0.001836,18.362696
6,roll_transition,b4c1d697-57ea-44a4-b7b8-fb9e75ee864c,September quarterly-roll vicinity; tests ES.v....,roll_adjacent,NaN,2026-09-18,2026-09-17 18:00:00-04:00,2026-09-18 17:00:00-04:00,0.3,None,...,neutral,True,0.0,0.0,0.0,0.0,0.0,0.0,0.000209,2.090608
7,post_roll_contract_timing,82629b32-f54a-460f-ae9d-b8386c9608be,Post-roll session publicly identified as ESZ6;...,post_roll,ESZ6,2026-09-29,2026-09-28 18:00:00-04:00,2026-09-29 17:00:00-04:00,0.3,None,...,neutral,True,0.0,0.0,0.0,0.0,0.0,0.0,0.001935,19.350029


In [28]:
def alignment_view(alignment):
    wanted = [
        "challenge_id",
        "challenge_date",
        "session_type",
        "dead_zone_pct",
        "arena_open",
        "db_open",
        "open_diff_ticks",
        "arena_settlement",
        "db_close",
        "settlement_diff_ticks",
        "arena_return",
        "db_return",
        "return_diff_bps",
        "arena_outcome",
        "db_outcome",
        "class_match",
        "distance_to_boundary_bps",
        "first_bar_on_expected_boundary",
        "last_bar_on_expected_boundary",
        "db_underlying_symbols",
        "multiple_underlying_symbols",
        "notes",
    ]

    cols = [c for c in wanted if c in alignment.columns]
    out = alignment[cols].copy()

    out["status"] = np.where(
        out["class_match"],
        "PASS_CLASS",
        "INVESTIGATE",
    )

    return out


# display(alignment_view(alignment))

### 12.1 Boundary-review flag

The following threshold is **not an Arena rule**.

It is only a research-review aid: if a reconstructed return is very close to a class boundary, tiny feed/settlement differences can flip the class. Such rows should be interpreted cautiously.

Default review buffer: **2 basis points** from the nearest boundary.

In [29]:
BOUNDARY_REVIEW_BUFFER_BPS = 2.0

def add_boundary_review_flag(alignment):
    out = alignment.copy()
    out["near_boundary"] = (
        out["distance_to_boundary_bps"] <= BOUNDARY_REVIEW_BUFFER_BPS
    )
    return out


# alignment = add_boundary_review_flag(alignment)
# alignment[[
#     "challenge_id",
#     "distance_to_boundary_bps",
#     "near_boundary",
#     "class_match",
# ]]

In [30]:
# ------------------------------------------------------------
# Final eight-challenge Arena ↔ Databento validation gate
# ------------------------------------------------------------

alignment = add_boundary_review_flag(
    alignment
)


def contract_matches_expected(row):
    expected = row["expected_contract"]

    if pd.isna(expected) or expected is None:
        return True

    return expected in row["db_underlying_symbols"]


alignment["expected_contract_match"] = alignment.apply(
    contract_matches_expected,
    axis=1,
)


# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

assert len(alignment) == EXPECTED_VALIDATION_COUNT, (
    f"Expected {EXPECTED_VALIDATION_COUNT} validation challenges, "
    f"got {len(alignment)}."
)

assert alignment["class_match"].all(), (
    "At least one Arena class does not match Databento."
)

assert alignment["first_bar_on_expected_boundary"].all(), (
    "At least one session does not begin on the expected 18:00 ET bar."
)

assert alignment["last_bar_on_expected_boundary"].all(), (
    "At least one session does not end on the expected 16:59 ET bar."
)

assert alignment["db_bar_count"].eq(1380).all(), (
    "At least one validation session does not contain 1,380 minute bars."
)

assert (~alignment["multiple_underlying_symbols"]).all(), (
    "At least one validation session contains multiple underlying contracts."
)


# ------------------------------------------------------------
# Price / return agreement checks
# ------------------------------------------------------------

assert np.isclose(
    alignment["open_diff_ticks"],
    0.0,
    atol=1e-9,
).all(), "At least one Arena opening price differs from Databento."

assert np.isclose(
    alignment["settlement_diff_ticks"],
    0.0,
    atol=1e-9,
).all(), "At least one Arena settlement differs from Databento."

assert np.isclose(
    alignment["return_diff_bps"],
    0.0,
    atol=1e-9,
).all(), "At least one reconstructed return differs from Arena."


# ------------------------------------------------------------
# Expected-contract checks
# ------------------------------------------------------------

assert alignment["expected_contract_match"].all(), (
    "At least one explicitly expected Arena contract "
    "does not match the Databento raw contract."
)


print("FINAL 8-CHALLENGE DATABENTO ALIGNMENT: PASS")
print(
    "Class matches:",
    f"{int(alignment['class_match'].sum())}/{len(alignment)}",
)
print(
    "Near-boundary rows:",
    int(alignment["near_boundary"].sum()),
)
print("Expected-contract validation: PASS")


display(
    alignment[
        [
            "validation_role",
            "challenge_date",
            "arena_open",
            "db_open",
            "open_diff_ticks",
            "arena_settlement",
            "db_close",
            "settlement_diff_ticks",
            "arena_outcome",
            "db_outcome",
            "class_match",
            "distance_to_boundary_bps",
            "near_boundary",
            "db_bar_count",
            "db_underlying_symbols",
            "expected_contract",
            "expected_contract_match",
        ]
    ]
)

FINAL 8-CHALLENGE DATABENTO ALIGNMENT: PASS
Class matches: 8/8
Near-boundary rows: 0
Expected-contract validation: PASS


,validation_role,challenge_date,arena_open,db_open,open_diff_ticks,arena_settlement,db_close,settlement_diff_ticks,arena_outcome,db_outcome,class_match,distance_to_boundary_bps,near_boundary,db_bar_count,db_underlying_symbols,expected_contract,expected_contract_match
0,clear_bullish,2026-09-25,7759.00,7759.00,0.0,7805.75,7805.75,0.0,bullish,bullish,True,30.252610,False,1380,[ESZ6],NaN,True
1,bullish_boundary_stress,2026-07-15,7589.75,7589.75,0.0,7616.00,7616.00,0.0,bullish,bullish,True,4.586119,False,1380,[ESU6],NaN,True
2,clear_bearish,2026-09-28,7796.00,7796.00,0.0,7746.25,7746.25,0.0,bearish,bearish,True,33.814777,False,1380,[ESZ6],NaN,True
3,clear_bearish_replication,2026-09-23,7830.00,7830.00,0.0,7772.75,7772.75,0.0,bearish,bearish,True,43.116220,False,1380,[ESZ6],NaN,True
4,clear_neutral,2026-07-01,7543.25,7543.25,0.0,7532.75,7532.75,0.0,neutral,neutral,True,16.080270,False,1380,[ESU6],NaN,True
5,clear_neutral_replication,2026-08-28,7733.75,7733.75,0.0,7724.75,7724.75,0.0,neutral,neutral,True,18.362696,False,1380,[ESU6],NaN,True
6,roll_transition,2026-09-18,7703.50,7703.50,0.0,7725.00,7725.00,0.0,neutral,neutral,True,2.090608,False,1380,[ESZ6],NaN,True
7,post_roll_contract_timing,2026-09-29,7746.50,7746.50,0.0,7738.25,7738.25,0.0,neutral,neutral,True,19.350029,False,1380,[ESZ6],ESZ6,True


In [31]:
post_roll_check = alignment.loc[
    alignment["validation_role"] == "post_roll_contract_timing",
    [
        "challenge_id",
        "challenge_date",
        "arena_open",
        "db_open",
        "open_diff_ticks",
        "arena_settlement",
        "db_close",
        "settlement_diff_ticks",
        "arena_return",
        "db_return",
        "return_diff_bps",
        "arena_outcome",
        "db_outcome",
        "class_match",
        "distance_to_boundary_bps",
        "near_boundary",
        "first_bar_on_expected_boundary",
        "last_bar_on_expected_boundary",
        "db_bar_count",
        "db_underlying_symbols",
        "multiple_underlying_symbols",
        "expected_contract",
        "expected_contract_match",
    ],
]

display(post_roll_check.T)

,7
challenge_id,82629b32-f54a-460f-ae9d-b8386c9608be
challenge_date,2026-09-29
arena_open,7746.5
db_open,7746.5
open_diff_ticks,0.0
arena_settlement,7738.25
db_close,7738.25
settlement_diff_ticks,0.0
arena_return,-0.001065
db_return,-0.001065


## 13. `trades` diagnostics — use only when needed

Do **not** download full historical trades.

Use small windows around the exact opening or closing boundary when:

- minute-bar price differs unexpectedly from Arena,
- a class mismatch occurs,
- a challenge is very close to the dead zone,
- the underlying contract appears wrong,
- the exact settlement convention needs diagnosis.

In [32]:
def quote_trade_window(center_utc, minutes_each_side=5):
    center_utc = pd.Timestamp(center_utc)
    start = center_utc - pd.Timedelta(minutes=minutes_each_side)
    end = center_utc + pd.Timedelta(minutes=minutes_each_side)

    cost = client.metadata.get_cost(
        dataset=DATASET,
        symbols=SYMBOL,
        stype_in=STYPE_IN,
        schema=DIAGNOSTIC_SCHEMA,
        start=start,
        end=end,
    )

    return {
        "start": start,
        "end": end,
        "quoted_cost_usd": cost,
    }


def download_trade_window(center_utc, output_path, minutes_each_side=5):
    if not ENABLE_VALIDATION_DOWNLOAD:
        raise RuntimeError("Validation download is disabled.")

    quote = quote_trade_window(center_utc, minutes_each_side)
    print("Trade-window quote:", quote)

    path = Path(output_path)

    if path.exists():
        print("Local trade file already exists — skipping paid re-download.")
        return path

    client.timeseries.get_range(
        dataset=DATASET,
        symbols=SYMBOL,
        stype_in=STYPE_IN,
        schema=DIAGNOSTIC_SCHEMA,
        start=quote["start"],
        end=quote["end"],
        path=str(path),
    )

    return path

## 14. Databento dataset-condition diagnostics

If a validation date behaves unexpectedly, inspect Databento's condition metadata before drawing conclusions.

In [33]:
def get_quality_conditions(start_date, end_date):
    return pd.DataFrame(
        client.metadata.get_dataset_condition(
            dataset=DATASET,
            start_date=start_date,
            end_date=end_date,
        )
    )


# Example:
# get_quality_conditions("2026-09-01", "2026-09-30")

## 15. Provider acceptance gate — completed

Databento is accepted only after manual review of **all eight** validation
challenges confirms the structural and price-alignment requirements below.

Completed checks:

- [x] both bullish role cases reconstruct correctly,
- [x] both bearish role cases reconstruct correctly,
- [x] both neutral role cases reconstruct correctly,
- [x] the roll-transition case has explainable underlying-contract behavior,
- [x] the post-roll contract/timing case maps correctly, including expected `ESZ6`,
- [x] normal 18:00 ET opening is reconstructed correctly,
- [x] normal 17:00 ET ending is reconstructed correctly,
- [x] no systematic one-hour timezone/DST error exists,
- [x] all eight Arena classes agree with Databento,
- [x] Arena open, settlement, and reconstructed return match exactly,
- [x] no unexplained contract mismatch remains.

Holiday/shortened-session handling is still required during the **full historical
quality audit**. The eight-challenge suite validates the provider/session plumbing;
it does not replace the later historical session-calendar audit.

`DATABENTO_ACCEPTED=True` is set only after the final assertions pass.


In [34]:
def provider_acceptance_report(alignment):
    if len(alignment) != EXPECTED_VALIDATION_COUNT:
        raise RuntimeError(
            f"Acceptance report requires all "
            f"{EXPECTED_VALIDATION_COUNT} challenges."
        )

    if "near_boundary" not in alignment.columns:
        alignment = add_boundary_review_flag(alignment)

    non_boundary = alignment.loc[
        ~alignment["near_boundary"]
    ]

    role_matches = (
        alignment.groupby("validation_role")["class_match"]
        .all()
        .to_dict()
    )

    return pd.Series({
        "provider": "Databento",
        "dataset": DATASET,
        "symbol": SYMBOL,
        "schema": PRIMARY_SCHEMA,
        "validation_challenges": len(alignment),
        "all_eight_present": (
            len(alignment) == EXPECTED_VALIDATION_COUNT
        ),
        "all_class_matches": bool(
            alignment["class_match"].all()
        ),
        "class_mismatches": int(
            (~alignment["class_match"]).sum()
        ),
        "near_boundary_rows": int(
            alignment["near_boundary"].sum()
        ),
        "non_boundary_rows": len(non_boundary),
        "non_boundary_class_mismatches": (
            int((~non_boundary["class_match"]).sum())
            if len(non_boundary)
            else np.nan
        ),
        "boundary_timestamp_issues": int(
            (
                ~alignment["first_bar_on_expected_boundary"]
                | ~alignment["last_bar_on_expected_boundary"]
            ).sum()
        ),
        "role_class_match_map": role_matches,
        "manual_acceptance_flag": bool(
            DATABENTO_ACCEPTED
        ),
    })


# provider_acceptance_report(alignment)

In [35]:
# ------------------------------------------------------------
# Provider acceptance report before manual approval
# ------------------------------------------------------------

provider_report = provider_acceptance_report(
    alignment
)

display(provider_report)

provider                                                                 Databento
dataset                                                                  GLBX.MDP3
symbol                                                                      ES.v.0
schema                                                                    ohlcv-1m
validation_challenges                                                            8
all_eight_present                                                             True
all_class_matches                                                             True
class_mismatches                                                                 0
near_boundary_rows                                                               0
non_boundary_rows                                                                8
non_boundary_class_mismatches                                                    0
boundary_timestamp_issues                                                        0
role

In [36]:
# ------------------------------------------------------------
# Manual Databento provider acceptance
# ------------------------------------------------------------
#
# Accept only after the final eight-challenge validation gate
# has passed and no unexplained structural mismatch remains.

assert len(alignment) == EXPECTED_VALIDATION_COUNT
assert alignment["class_match"].all()
assert alignment["first_bar_on_expected_boundary"].all()
assert alignment["last_bar_on_expected_boundary"].all()
assert (~alignment["multiple_underlying_symbols"]).all()
assert alignment["expected_contract_match"].all()

DATABENTO_ACCEPTED = True

print("Databento accepted:", DATABENTO_ACCEPTED)

Databento accepted: True


In [37]:
provider_report = provider_acceptance_report(
    alignment
)

display(provider_report)

provider                                                                 Databento
dataset                                                                  GLBX.MDP3
symbol                                                                      ES.v.0
schema                                                                    ohlcv-1m
validation_challenges                                                            8
all_eight_present                                                             True
all_class_matches                                                             True
class_mismatches                                                                 0
near_boundary_rows                                                               0
non_boundary_rows                                                                8
non_boundary_class_mismatches                                                    0
boundary_timestamp_issues                                                        0
role

## 16. Save the derived alignment artifact

This CSV contains derived validation results, not the licensed raw Databento history.

In [38]:
ALIGNMENT_CSV = PROCESSED_DIR / "es_arena_alignment_v0_1.csv"

def save_alignment(alignment, path=ALIGNMENT_CSV):
    path = Path(path)
    alignment.to_csv(path, index=False)
    print("Saved:", path)
    return path


# save_alignment(alignment)

In [39]:
# ------------------------------------------------------------
# Persist reviewed Arena ↔ Databento validation artifact
# ------------------------------------------------------------

if not DATABENTO_ACCEPTED:
    raise RuntimeError(
        "Do not save the final alignment artifact "
        "before Databento has been manually accepted."
    )

alignment_path = save_alignment(
    alignment
)

print("Final alignment artifact:", alignment_path)

Saved: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_alignment_v0_1.csv
Final alignment artifact: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_alignment_v0_1.csv


# Phase B — full historical acquisition

Run this phase **only after Phase A has passed and Databento has been accepted manually**.

The historical endpoint must be the last fully completed pre-experiment session, not Databento's moving current availability timestamp.

## 17. Freeze the historical acquisition window

The v0.1 historical endpoint is now frozen to the last fully completed session
selected during the provider-validation stage:

- final included ES minute: **2026-09-30 16:59 ET**,
- exclusive request endpoint: **2026-09-30 17:00 ET**,
- equivalent UTC endpoint: **2026-09-30 21:00:00Z**.

Do not move this cutoff later simply because newer market data becomes available.
Changing it would create a different experiment version.


In [40]:
HISTORY_START_UTC = "2010-06-06T00:00:00Z"

# Frozen v0.1 pre-experiment endpoint.
# Databento `end` is exclusive:
# 2026-09-30 17:00 ET == 2026-09-30 21:00 UTC.
HISTORY_END_UTC = "2026-09-30T21:00:00Z"


def validate_history_window():
    if HISTORY_END_UTC is None:
        raise RuntimeError("Freeze HISTORY_END_UTC first.")

    start = pd.Timestamp(HISTORY_START_UTC)
    end = pd.Timestamp(HISTORY_END_UTC)

    if start.tzinfo is None or end.tzinfo is None:
        raise ValueError(
            "Historical boundaries must be timezone-aware."
        )

    if end <= start:
        raise ValueError(
            "HISTORY_END_UTC must be after HISTORY_START_UTC."
        )

    expected_end_et = pd.Timestamp(
        "2026-09-30 17:00:00",
        tz=ANALYSIS_TIMEZONE,
    )

    if end.tz_convert(ANALYSIS_TIMEZONE) != expected_end_et:
        raise ValueError(
            "Frozen v0.1 history endpoint changed unexpectedly."
        )

    return start, end


## 18. Final full-history quote

Always re-quote after freezing the exact historical cutoff.

In [41]:
def final_full_history_quote():
    start, end = (
        validate_history_window()
    )

    request_kwargs = {
        "dataset": DATASET,
        "symbols": SYMBOL,
        "stype_in": STYPE_IN,
        "schema": PRIMARY_SCHEMA,
        "start": start,
        "end": end,
    }

    cost = metadata_call_with_retry(
        client.metadata.get_cost,
        **request_kwargs,
    )

    records = metadata_call_with_retry(
        client.metadata.get_record_count,
        **request_kwargs,
    )

    billable_bytes = (
        metadata_call_with_retry(
            client.metadata.get_billable_size,
            **request_kwargs,
        )
    )

    return pd.Series({
        "start_utc": start,
        "end_utc": end,
        "cost_usd": cost,
        "records": records,
        "billable_bytes":
            billable_bytes,
        "billable_gb":
            billable_bytes
            / 1_000_000_000,
    })


# This function only requests metadata.
# It does NOT submit the full historical batch.

### 18.1 Next safe action — review the frozen full-history quote

This is a **metadata quote only**. It does not submit the full historical batch
job. Review the returned cost, record count, and billable size before changing
any full-download guard.

In [42]:
final_quote = final_full_history_quote()

display(final_quote)


start_utc         2010-06-06 00:00:00+00:00
end_utc           2026-09-30 21:00:00+00:00
cost_usd                          20.827116
records                             5704833
billable_bytes                    319470648
billable_gb                        0.319471
dtype: object

## 19. Full-history batch request — triple guarded + duplicate protected

The frozen historical quote has been reviewed:

- start: `2010-06-06T00:00:00Z`
- end: `2026-09-30T21:00:00Z`
- quoted cost: approximately **$20.827116**
- quoted records: **5,704,833**
- quoted billable size: **0.319471 GB**

The batch-acquisition layer now adds four protections:

1. the original three manual guards must pass,
2. a matching Databento batch job must not already exist,
3. the exact frozen quote is rechecked immediately before submission,
4. the submitted job ID is persisted locally so a kernel restart cannot cause accidental resubmission.

**Run All is safe by default.** The notebook does not submit a paid batch unless the manual authorization switches in Section 19.1 are deliberately changed.


In [43]:
# ------------------------------------------------------------
# Full-history batch acquisition — triple guarded + duplicate protected
# ------------------------------------------------------------

FULL_HISTORY_JOB_PATH = (
    PROCESSED_DIR
    / "databento_full_history_job_v0_1.json"
)


def validate_full_download_gate():
    validate_history_window()

    if not DATABENTO_ACCEPTED:
        raise RuntimeError(
            "Databento has not been formally accepted."
        )

    if not ENABLE_FULL_HISTORY_DOWNLOAD:
        raise RuntimeError(
            "Full-history download is disabled."
        )

    if (
        FULL_DOWNLOAD_CONFIRMATION
        != "I_REVIEWED_FINAL_QUOTE"
    ):
        raise RuntimeError(
            'Set FULL_DOWNLOAD_CONFIRMATION = '
            '"I_REVIEWED_FINAL_QUOTE" only after manually '
            "reviewing final_full_history_quote()."
        )


def matching_existing_batch_jobs():
    """
    Return Databento batch jobs that match the exact frozen
    v0.1 full-history request.

    This protects against accidentally submitting the same
    paid request more than once.
    """

    jobs = client.batch.list_jobs(
        states=[
            "queued",
            "processing",
            "done",
        ],
    )

    expected_start = pd.Timestamp(
        HISTORY_START_UTC
    )

    expected_end = pd.Timestamp(
        HISTORY_END_UTC
    )

    matches = []

    for job in jobs:
        try:
            job_start = pd.Timestamp(
                job["start"]
            )
            job_end = pd.Timestamp(
                job["end"]
            )
        except Exception:
            continue

        job_symbols = str(
            job.get("symbols", "")
        )

        same_request = (
            job.get("dataset") == DATASET
            and job.get("schema") == PRIMARY_SCHEMA
            and job.get("stype_in") == STYPE_IN
            and SYMBOL in job_symbols
            and job_start == expected_start
            and job_end == expected_end
        )

        if same_request:
            matches.append(job)

    return matches


def save_full_history_job_record(
    full_history_job,
    quote,
    path=FULL_HISTORY_JOB_PATH,
):
    record = {
        "job_id": full_history_job["id"],
        "dataset": DATASET,
        "symbol": SYMBOL,
        "schema": PRIMARY_SCHEMA,
        "stype_in": STYPE_IN,
        "history_start_utc": HISTORY_START_UTC,
        "history_end_utc": HISTORY_END_UTC,
        "quoted_cost_usd": float(
            quote["cost_usd"]
        ),
        "quoted_records": int(
            quote["records"]
        ),
        "quoted_billable_bytes": int(
            quote["billable_bytes"]
        ),
        "submitted_at_utc": datetime.now(
            tz=UTC
        ).isoformat(),
    }

    Path(path).write_text(
        json.dumps(
            record,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "Saved batch-job record:",
        path,
    )

    return record


def load_full_history_job_record(
    path=FULL_HISTORY_JOB_PATH,
):
    path = Path(path)

    if not path.exists():
        return None

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def submit_full_history_batch():
    validate_full_download_gate()

    existing = matching_existing_batch_jobs()

    if existing:
        existing_ids = [
            job["id"]
            for job in existing
        ]

        raise RuntimeError(
            "A matching full-history batch job already exists. "
            "Refusing to submit another paid request.\n\n"
            f"Existing job IDs: {existing_ids}"
        )

    quote = final_full_history_quote()

    print("FINAL REVIEWED QUOTE")
    display(quote)

    job = client.batch.submit_job(
        dataset=DATASET,
        symbols=[SYMBOL],
        stype_in=STYPE_IN,
        schema=PRIMARY_SCHEMA,
        start=HISTORY_START_UTC,
        end=HISTORY_END_UTC,
        encoding="dbn",
        compression="zstd",
        split_duration="year",
    )

    save_full_history_job_record(
        job,
        quote,
    )

    return job


print(
    "Full-history submission helpers ready. "
    "No paid batch has been submitted by this cell."
)


Full-history submission helpers ready. No paid batch has been submitted by this cell.


### 19.1 Manual authorization — disabled by default

The following switches make the notebook safe to rerun from top to bottom.

Leave both values `False` for normal reproducibility runs.

Only after deliberately accepting the approximately **$20.83** frozen quote should you change both to `True` and run the submission cell immediately below **once**.


In [44]:
# ------------------------------------------------------------
# MANUAL PAID-ACQUISITION CONTROLS
# ------------------------------------------------------------
#
# SAFE DEFAULTS:
# A normal Run All will NOT submit a paid batch.

AUTHORIZE_FULL_HISTORY_PURCHASE = False
SUBMIT_FULL_HISTORY_NOW = False


if AUTHORIZE_FULL_HISTORY_PURCHASE:
    ENABLE_FULL_HISTORY_DOWNLOAD = True
    FULL_DOWNLOAD_CONFIRMATION = (
        "I_REVIEWED_FINAL_QUOTE"
    )

    print(
        "Full-history purchase authorization is ON."
    )

else:
    ENABLE_FULL_HISTORY_DOWNLOAD = False
    FULL_DOWNLOAD_CONFIRMATION = ""

    print(
        "Full-history purchase authorization is OFF. "
        "No paid batch can be submitted."
    )


Full-history purchase authorization is OFF. No paid batch can be submitted.


### 19.2 One-time submission or safe resume

This cell is also safe by default.

- If `SUBMIT_FULL_HISTORY_NOW = False`, it does **not** submit anything.
- If a saved job record already exists, it restores the job ID for later status checks.
- A paid submission occurs only when both manual switches in Section 19.1 are deliberately `True`.
- Immediately after any successful submission, the paid-acquisition gate is closed again.


In [45]:
# ------------------------------------------------------------
# ONE-TIME FULL-HISTORY SUBMISSION / RESUME
# ------------------------------------------------------------

saved_job_record = (
    load_full_history_job_record()
)

full_history_job = None
full_history_job_id = None


if saved_job_record is not None:
    full_history_job_id = (
        saved_job_record["job_id"]
    )

    print(
        "Existing saved full-history job record found."
    )
    print(
        "Job ID:",
        full_history_job_id,
    )


if SUBMIT_FULL_HISTORY_NOW:
    if not AUTHORIZE_FULL_HISTORY_PURCHASE:
        raise RuntimeError(
            "SUBMIT_FULL_HISTORY_NOW is True, but "
            "AUTHORIZE_FULL_HISTORY_PURCHASE is False."
        )

    if saved_job_record is not None:
        raise RuntimeError(
            "A saved full-history job record already exists. "
            "Refusing to submit another request.\n"
            f"Existing job ID: {full_history_job_id}"
        )

    try:
        full_history_job = (
            submit_full_history_batch()
        )

        full_history_job_id = (
            full_history_job["id"]
        )

        display(
            pd.Series(
                full_history_job
            )
        )

        print(
            "ONE-TIME FULL-HISTORY BATCH SUBMITTED."
        )
        print(
            "Job ID:",
            full_history_job_id,
        )

    finally:
        # Close the paid-acquisition gate immediately,
        # even if a later display/serialization step fails.
        ENABLE_FULL_HISTORY_DOWNLOAD = False
        FULL_DOWNLOAD_CONFIRMATION = ""

        print(
            "Paid-acquisition gate restored to OFF."
        )

else:
    print(
        "SUBMIT_FULL_HISTORY_NOW = False. "
        "No paid batch submitted."
    )


Existing saved full-history job record found.
Job ID: GLBX-20261006-RVP5LPX9NY
SUBMIT_FULL_HISTORY_NOW = False. No paid batch submitted.


In [46]:
def get_batch_job_status(job_id):
    details = client.batch.get_job_details(
        job_id=job_id
    )

    return pd.Series({
        "job_id": details["id"],
        "state": details["state"],
        "progress": details.get("progress"),
        "record_count": details.get("record_count"),
        "billed_size": details.get("billed_size"),
        "actual_size": details.get("actual_size"),
        "package_size": details.get("package_size"),
        "cost_usd": details.get("cost_usd"),
        "ts_received": details.get("ts_received"),
        "ts_process_start": details.get("ts_process_start"),
        "ts_process_done": details.get("ts_process_done"),
        "ts_expiration": details.get("ts_expiration"),
    })


def list_batch_job_files(job_id):
    return pd.DataFrame(
        client.batch.list_files(
            job_id=job_id
        )
    )


def download_batch_job(job_id):
    details = client.batch.get_job_details(
        job_id=job_id
    )

    if details["state"] != "done":
        raise RuntimeError(
            "Batch job is not ready for download. "
            f"Current state: {details['state']}"
        )

    return client.batch.download(
        job_id=job_id,
        output_dir=str(RAW_DIR),
    )


print(
    "Batch status/download helpers ready."
)


Batch status/download helpers ready.


### 19.3 Check the submitted batch job

If a full-history job ID has been saved, this cell checks its current Databento state.

This is safe: checking status does not submit another paid batch.


In [47]:
# ------------------------------------------------------------
# SAFE STATUS CHECK
# ------------------------------------------------------------

if full_history_job_id is None:
    saved_job_record = (
        load_full_history_job_record()
    )

    if saved_job_record is not None:
        full_history_job_id = (
            saved_job_record["job_id"]
        )


if full_history_job_id is None:
    full_history_status = None

    print(
        "No full-history batch job has been submitted yet."
    )

else:
    full_history_status = (
        get_batch_job_status(
            full_history_job_id
        )
    )

    display(
        full_history_status
    )


job_id                    GLBX-20261006-RVP5LPX9NY
state                                         done
progress                                       100
record_count                               5704833
billed_size                              319470648
actual_size                              319470648
package_size                              79183207
cost_usd                                 20.827116
ts_received         2026-10-06T16:22:38.457396000Z
ts_process_start    2026-10-06T16:42:38.636094000Z
ts_process_done     2026-10-06T16:45:52.889301000Z
ts_expiration       2026-11-05T16:50:00.000000000Z
dtype: object

### 19.4 Inspect/download a completed batch — disabled by default

Downloading does not create a new market-data request, but it can transfer a large set of files.

Leave `DOWNLOAD_COMPLETED_BATCH_NOW = False` during normal Run All.

After the status above shows `state == "done"`, set it to `True` and run the cell once.


In [48]:
# ------------------------------------------------------------
# DOWNLOAD COMPLETED BATCH — SAFE DEFAULT OFF
# ------------------------------------------------------------

DOWNLOAD_COMPLETED_BATCH_NOW = False


if not DOWNLOAD_COMPLETED_BATCH_NOW:
    print(
        "Historical batch download is OFF."
    )

elif full_history_job_id is None:
    raise RuntimeError(
        "No full-history job ID is available."
    )

else:
    full_history_status = (
        get_batch_job_status(
            full_history_job_id
        )
    )

    if full_history_status["state"] != "done":
        raise RuntimeError(
            "Batch is not ready. "
            f"Current state: {full_history_status['state']}"
        )

    batch_files = list_batch_job_files(
        full_history_job_id
    )

    display(
        batch_files[
            [
                col
                for col in [
                    "filename",
                    "size",
                    "hash",
                ]
                if col in batch_files.columns
            ]
        ]
    )

    downloaded_paths = (
        download_batch_job(
            full_history_job_id
        )
    )

    print(
        "Downloaded files:",
        len(downloaded_paths),
    )

    for path in downloaded_paths:
        print(path)


Historical batch download is OFF.


## 20. Local raw-data manifest and hashes

After the batch files are downloaded, create a manifest so the exact raw inputs can be identified later without publishing the licensed files themselves.

In [49]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)

    return h.hexdigest()


def build_raw_manifest(raw_dir=RAW_DIR):
    paths = sorted(
        p for p in Path(raw_dir).rglob("*")
        if p.is_file()
    )

    rows = []
    for p in paths:
        rows.append({
            "relative_path": str(p.relative_to(PROJECT_ROOT)),
            "size_bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

    return pd.DataFrame(rows)


def save_raw_manifest(
    raw_manifest,
    path=None,
):
    if path is None:
        path = (
            PROCESSED_DIR
            / "es_raw_manifest_v0_1.csv"
        )

    path = Path(path)

    if raw_manifest.empty:
        raise RuntimeError(
            "Refusing to save an empty raw-data manifest."
        )

    required_columns = {
        "relative_path",
        "size_bytes",
        "sha256",
    }

    missing = (
        required_columns
        - set(raw_manifest.columns)
    )

    if missing:
        raise ValueError(
            "Raw manifest is missing required columns: "
            f"{sorted(missing)}"
        )

    raw_manifest.to_csv(
        path,
        index=False,
    )

    print("Saved raw manifest:", path)
    print("Manifest rows:", len(raw_manifest))

    return path

# Build the final raw-data manifest only from the completed
# full-history batch directory, not from all of RAW_DIR.
#
# Example after the batch has been downloaded:
#
# full_history_raw_dir = (
#     batch_job_directory(
#         full_history_job_id
#     )
# )
#
# raw_manifest = build_raw_manifest(
#     raw_dir=full_history_raw_dir
# )
#
# raw_manifest_path = save_raw_manifest(
#     raw_manifest
# )
#
# display(raw_manifest.head())


## 20.1 Full-history local load and contract resolution

Run this section **only after** the full-history batch job is `done` and its files
have been downloaded locally.

Databento batch downloads are stored under `{RAW_DIR}/{job_id}/`. The helpers
below:

1. discover only DBN/DBN-Zstd files inside that job directory,
2. load and concatenate them locally,
3. resolve every observed `instrument_id` to one raw futures symbol,
4. fail closed if an instrument ID is unresolved or ambiguously maps to multiple
   raw symbols across the frozen history window,
5. preserve UTC first, then convert the verified dataframe to New York time.

The resolver intentionally does not guess through ambiguous historical mappings.

In [50]:
def batch_job_directory(job_id, raw_dir=RAW_DIR):
    return Path(raw_dir) / str(job_id)


def list_local_batch_dbn_files(job_id, raw_dir=RAW_DIR):
    job_dir = batch_job_directory(
        job_id,
        raw_dir=raw_dir,
    )

    if not job_dir.exists():
        raise FileNotFoundError(
            f"Batch job directory does not exist: {job_dir}"
        )

    paths = sorted(
        p
        for p in job_dir.rglob("*")
        if p.is_file()
        and (
            p.name.endswith(".dbn.zst")
            or p.suffix.lower() == ".dbn"
        )
    )

    if not paths:
        raise FileNotFoundError(
            f"No DBN files found under: {job_dir}"
        )

    return paths


def load_full_history_batch(job_id, raw_dir=RAW_DIR):
    paths = list_local_batch_dbn_files(
        job_id,
        raw_dir=raw_dir,
    )

    frames = []

    for path in paths:
        store = db.DBNStore.from_file(path)

        frame = store.to_df(
            pretty_ts=True,
            map_symbols=True,
            tz="UTC",
        )

        if frame.empty:
            continue

        frame = frame.copy()
        frame["source_file"] = str(
            path.relative_to(PROJECT_ROOT)
        )
        frames.append(frame)

    if not frames:
        raise RuntimeError(
            "Full-history batch files were found, "
            "but no time-series rows were loaded."
        )

    combined = pd.concat(
        frames,
        axis=0,
    ).sort_index()

    if combined.index.tz is None:
        raise ValueError(
            "Expected a timezone-aware UTC Databento index."
        )

    if combined.index.has_duplicates:
        duplicate_count = int(
            combined.index.duplicated().sum()
        )
        raise ValueError(
            "Duplicate timestamps detected after concatenating "
            f"the batch files: {duplicate_count}"
        )

    return combined, paths


def _chunks(values, size=500):
    values = list(values)

    for start in range(0, len(values), size):
        yield values[start:start + size]


def resolve_full_history_raw_symbols(
    es_history_utc,
    chunk_size=500,
):
    if "instrument_id" not in es_history_utc.columns:
        raise ValueError(
            "instrument_id is required for raw-symbol resolution."
        )

    instrument_ids = (
        es_history_utc["instrument_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if not instrument_ids:
        raise ValueError(
            "No instrument IDs were found in the full history."
        )

    start_date = (
        pd.Timestamp(es_history_utc.index.min())
        .tz_convert("UTC")
        .date()
        .isoformat()
    )

    end_date = (
        (
            pd.Timestamp(es_history_utc.index.max())
            .tz_convert("UTC")
            + pd.Timedelta(days=1)
        )
        .date()
        .isoformat()
    )

    raw_mapping = {}
    ambiguous = {}
    unresolved = []

    for ids in _chunks(
        instrument_ids,
        size=chunk_size,
    ):
        result = client.symbology.resolve(
            dataset=DATASET,
            symbols=ids,
            stype_in="instrument_id",
            stype_out="raw_symbol",
            start_date=start_date,
            end_date=end_date,
        )

        unresolved.extend(
            int(x)
            for x in result.get("not_found", [])
        )

        for instrument_id, mappings in result["result"].items():
            symbols = sorted({
                str(mapping["s"])
                for mapping in mappings
                if mapping.get("s") is not None
            })

            instrument_id = int(instrument_id)

            if len(symbols) == 1:
                raw_mapping[instrument_id] = symbols[0]
            else:
                ambiguous[instrument_id] = symbols

    if unresolved:
        raise RuntimeError(
            "Unresolved historical instrument IDs: "
            f"{sorted(set(unresolved))}"
        )

    if ambiguous:
        raise RuntimeError(
            "Historical instrument IDs with ambiguous raw-symbol "
            f"mappings: {ambiguous}. "
            "Do not guess; resolve these mappings explicitly."
        )

    missing_mapping = sorted(
        set(instrument_ids) - set(raw_mapping)
    )

    if missing_mapping:
        raise RuntimeError(
            "Missing raw-symbol mappings for historical "
            f"instrument IDs: {missing_mapping}"
        )

    out = es_history_utc.copy()

    out["raw_symbol"] = (
        out["instrument_id"]
        .astype(int)
        .map(raw_mapping)
    )

    if out["raw_symbol"].isna().any():
        raise RuntimeError(
            "raw_symbol contains missing values after "
            "historical symbology resolution."
        )

    mapping_table = (
        pd.DataFrame(
            sorted(raw_mapping.items()),
            columns=[
                "instrument_id",
                "raw_symbol",
            ],
        )
    )

    return out, mapping_table


# Run only after the full-history batch is complete and downloaded:
#
# full_history_job_id = full_history_job["id"]
#
# es_history_utc, full_history_files = load_full_history_batch(
#     full_history_job_id
# )
#
# es_history_utc, full_history_contract_map = (
#     resolve_full_history_raw_symbols(
#         es_history_utc
#     )
# )
#
# es_history_ny = to_new_york(
#     es_history_utc
# )
#
# display(full_history_contract_map.tail())
# print("Historical rows:", len(es_history_ny))
# print("Historical NY start:", es_history_ny.index.min())
# print("Historical NY end:", es_history_ny.index.max())


In [51]:
# ------------------------------------------------------------
# Recovery check after PC / kernel restart
# ------------------------------------------------------------

assert full_history_job_id is not None, (
    "full_history_job_id was not restored."
)

EXPECTED_FULL_HISTORY_JOB_ID = (
    "GLBX-20261006-RVP5LPX9NY"
)

assert (
    full_history_job_id
    == EXPECTED_FULL_HISTORY_JOB_ID
), (
    "Unexpected Databento job ID: "
    f"{full_history_job_id}"
)


full_history_raw_dir = (
    batch_job_directory(
        full_history_job_id
    )
)

assert full_history_raw_dir.exists(), (
    "Downloaded full-history batch directory "
    f"is missing: {full_history_raw_dir}"
)


dbn_files_on_disk = (
    list_local_batch_dbn_files(
        full_history_job_id
    )
)

all_files_on_disk = sorted(
    p
    for p in full_history_raw_dir.iterdir()
    if p.is_file()
)


print(
    "Recovered job ID:",
    full_history_job_id,
)

print(
    "Batch directory:",
    full_history_raw_dir,
)

print(
    "DBN files on disk:",
    len(dbn_files_on_disk),
)

print(
    "Total files in batch directory:",
    len(all_files_on_disk),
)


assert len(dbn_files_on_disk) == 17, (
    "Expected 17 historical DBN files."
)

assert len(all_files_on_disk) == 20, (
    "Expected 20 total files in the "
    "downloaded Databento batch."
)


print(
    "RAW HISTORICAL ACQUISITION RECOVERY: PASS"
)

Recovered job ID: GLBX-20261006-RVP5LPX9NY
Batch directory: C:\Users\melny\Market-Mood-Forecasting\data\raw\databento\GLBX-20261006-RVP5LPX9NY
DBN files on disk: 17
Total files in batch directory: 20
RAW HISTORICAL ACQUISITION RECOVERY: PASS


In [52]:
# ------------------------------------------------------------
# Check saved SHA-256 raw manifest
# ------------------------------------------------------------

RAW_MANIFEST_PATH = (
    PROCESSED_DIR
    / "es_raw_manifest_v0_1.csv"
)

print(
    "Manifest path:",
    RAW_MANIFEST_PATH,
)

print(
    "Manifest exists:",
    RAW_MANIFEST_PATH.exists(),
)


if RAW_MANIFEST_PATH.exists():
    existing_raw_manifest = pd.read_csv(
        RAW_MANIFEST_PATH
    )

    print(
        "Manifest rows:",
        len(existing_raw_manifest),
    )

    display(
        existing_raw_manifest.head()
    )

Manifest path: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_raw_manifest_v0_1.csv
Manifest exists: True
Manifest rows: 20


,relative_path,size_bytes,sha256
0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\co...,618330,5be68219f26de2cbae5ac05ad97c858b47e0a9bc098c53...
1,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...,2517220,d9b03579db8df69280834165ecbc636e187e3692510c79...
2,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...,4589482,da13c34cb7f4e98594d510e3f931660c472c8ce316fbf6...
3,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...,4328565,a70388284441881831e095112e1fb77d4e037310fc8f1c...
4,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...,4166472,7edf4cb37c251ed0eede6d71312800cc6f5681917734ee...



### 20.1.1 Verify the saved raw-data manifest against local files

Independently verify the frozen full-history Databento files against
their saved SHA-256 manifest.

The audit checks:

- all 20 expected files exist;
- every file remains inside the frozen batch directory;
- file sizes match the saved manifest;
- SHA-256 hashes match the original recorded values;
- no unrecorded files exist inside the batch directory.

This verification is strictly read-only.

It does not download data, contact Databento, purchase data,
rebuild the raw manifest, or modify historical files.

A mismatch must stop the pipeline rather than be silently repaired.


In [53]:

# ------------------------------------------------------------
# Verify saved raw manifest against actual historical files
# Read-only: no downloads and no paid requests
# ------------------------------------------------------------

RAW_MANIFEST_VERIFIED = False

assert RAW_MANIFEST_PATH.is_file(), (
    "Frozen raw manifest does not exist: "
    f"{RAW_MANIFEST_PATH}"
)

manifest = pd.read_csv(
    RAW_MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "size_bytes",
    "sha256",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(manifest.columns)
)

assert not missing_manifest_columns, (
    "Raw manifest is missing columns: "
    f"{sorted(missing_manifest_columns)}"
)

assert len(manifest) == 20, (
    "Expected exactly 20 frozen raw files, "
    f"found {len(manifest)}"
)

assert not manifest[
    "relative_path"
].duplicated().any(), (
    "Duplicate relative paths in the raw manifest."
)

assert manifest[
    list(required_manifest_columns)
].notna().all().all(), (
    "Null values found in the raw manifest."
)

# ------------------------------------------------------------
# Resolve the frozen Databento batch directory
# ------------------------------------------------------------

batch_root = Path(
    full_history_raw_dir
).resolve()

assert batch_root.is_dir(), (
    "Historical batch directory does not exist: "
    f"{batch_root}"
)

actual_files = {
    path.resolve()
    for path in batch_root.rglob("*")
    if path.is_file()
}

verified_paths = set()

# ------------------------------------------------------------
# Verify every recorded file
# ------------------------------------------------------------

for row in manifest.itertuples(index=False):

    path = (
        PROJECT_ROOT / row.relative_path
    ).resolve()

    # The manifest must not reference files outside
    # the purchased historical batch directory.

    assert path.is_relative_to(batch_root), (
        "Manifest references a file outside "
        f"the frozen batch directory: {path}"
    )

    assert path.is_file(), (
        f"Missing raw file: {path}"
    )

    actual_size = path.stat().st_size

    assert actual_size == int(
        row.size_bytes
    ), (
        f"File-size mismatch: {path}\n"
        f"Expected: {row.size_bytes}\n"
        f"Actual: {actual_size}"
    )

    actual_sha256 = sha256_file(
        path
    )

    expected_sha256 = str(
        row.sha256
    ).strip().lower()

    assert actual_sha256.lower() == expected_sha256, (
        f"SHA-256 mismatch: {path}\n"
        f"Expected: {expected_sha256}\n"
        f"Actual: {actual_sha256}"
    )

    verified_paths.add(path)

# ------------------------------------------------------------
# Verify the manifest describes the entire batch directory
# ------------------------------------------------------------

unexpected_files = (
    actual_files - verified_paths
)

missing_from_directory = (
    verified_paths - actual_files
)

assert not unexpected_files, (
    "Unrecorded files detected in the batch directory: "
    f"{sorted(map(str, unexpected_files))}"
)

assert not missing_from_directory, (
    "Manifest files are missing from the batch directory: "
    f"{sorted(map(str, missing_from_directory))}"
)

assert verified_paths == actual_files, (
    "The raw manifest and frozen batch directory "
    "do not contain identical file sets."
)

RAW_MANIFEST_VERIFIED = True

print(
    "Frozen batch directory:",
    batch_root,
)

print(
    "Manifest entries:",
    len(manifest),
)

print(
    "Verified raw files:",
    len(verified_paths),
)

print(
    "Raw manifest verified:",
    RAW_MANIFEST_VERIFIED,
)

print(
    "RAW SHA-256 INTEGRITY: PASS"
)


Frozen batch directory: C:\Users\melny\Market-Mood-Forecasting\data\raw\databento\GLBX-20261006-RVP5LPX9NY
Manifest entries: 20
Verified raw files: 20
Raw manifest verified: True
RAW SHA-256 INTEGRITY: PASS


### 20.2 Build the full-history SHA-256 manifest — disabled by default

The final manifest should describe **only the downloaded frozen full-history batch directory**, not the earlier eight validation files.

Leave this switch `False` until the full-history batch has been downloaded.


In [54]:
# ------------------------------------------------------------
# FULL-HISTORY MANIFEST — SAFE DEFAULT OFF
# ------------------------------------------------------------

BUILD_FULL_HISTORY_MANIFEST_NOW = False


if not BUILD_FULL_HISTORY_MANIFEST_NOW:
    print(
        "Full-history manifest build is OFF."
    )

elif full_history_job_id is None:
    raise RuntimeError(
        "No full-history job ID is available."
    )

else:
    full_history_raw_dir = (
        batch_job_directory(
            full_history_job_id
        )
    )

    raw_manifest = build_raw_manifest(
        raw_dir=full_history_raw_dir
    )

    raw_manifest_path = (
        save_raw_manifest(
            raw_manifest
        )
    )

    display(
        raw_manifest.head()
    )

    print(
        "Manifest saved:",
        raw_manifest_path,
    )


Full-history manifest build is OFF.


In [55]:
# ------------------------------------------------------------
# Load frozen full-history Databento batch locally
# ------------------------------------------------------------

assert full_history_job_id is not None, (
    "full_history_job_id is missing."
)

es_history_utc, full_history_files = (
    load_full_history_batch(
        full_history_job_id
    )
)

print(
    "Historical DBN files loaded:",
    len(full_history_files),
)

print(
    "Historical UTC rows:",
    len(es_history_utc),
)

print(
    "Historical UTC start:",
    es_history_utc.index.min(),
)

print(
    "Historical UTC end:",
    es_history_utc.index.max(),
)

print(
    "Columns:",
    es_history_utc.columns.tolist(),
)

Historical DBN files loaded: 17
Historical UTC rows: 5704833
Historical UTC start: 2010-06-07 00:00:00+00:00
Historical UTC end: 2026-09-30 20:59:00+00:00
Columns: ['rtype', 'publisher_id', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'source_file']


### 20.4 Inspect the loaded historical UTC data

Verify the reconstructed frozen Databento history before any
session-level transformation.

In [56]:
# ------------------------------------------------------------
# Inspect loaded full-history UTC data
# ------------------------------------------------------------

display(
    es_history_utc.head()
)

display(
    es_history_utc.tail()
)


print("\nDtypes:")

print(
    es_history_utc.dtypes
)


duplicate_timestamps = (
    es_history_utc.index
    .duplicated()
    .sum()
)


print(
    "\nDuplicate timestamps:",
    duplicate_timestamps,
)

print(
    "Timezone:",
    es_history_utc.index.tz,
)

print(
    "Rows:",
    len(es_history_utc),
)

print(
    "Start:",
    es_history_utc.index.min(),
)

print(
    "End:",
    es_history_utc.index.max(),
)

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,source_file
ts_event,,,,,,,,,,
2010-06-07 00:00:00+00:00,33,1,6640,1061.25,1061.75,1061.25,1061.25,206,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2010-06-07 00:01:00+00:00,33,1,6640,1061.25,1062.00,1061.25,1062.00,495,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2010-06-07 00:02:00+00:00,33,1,6640,1061.75,1062.00,1061.50,1061.50,476,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2010-06-07 00:03:00+00:00,33,1,6640,1061.50,1061.50,1061.00,1061.00,270,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2010-06-07 00:04:00+00:00,33,1,6640,1061.25,1062.25,1061.00,1062.25,349,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...


,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,source_file
ts_event,,,,,,,,,,
2026-09-30 20:55:00+00:00,33,1,10252,7722.50,7723.00,7722.25,7722.50,474,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2026-09-30 20:56:00+00:00,33,1,10252,7722.50,7723.00,7721.50,7722.25,329,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2026-09-30 20:57:00+00:00,33,1,10252,7722.25,7722.75,7721.75,7722.00,441,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2026-09-30 20:58:00+00:00,33,1,10252,7722.00,7722.00,7720.75,7720.75,550,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...
2026-09-30 20:59:00+00:00,33,1,10252,7720.75,7722.25,7720.00,7721.50,1078,ES.v.0,data\raw\databento\GLBX-20261006-RVP5LPX9NY\gl...



Dtypes:
rtype              uint8
publisher_id      uint16
instrument_id     uint32
open             float64
high             float64
low              float64
close            float64
volume            uint64
symbol               str
source_file          str
dtype: object

Duplicate timestamps: 0
Timezone: UTC
Rows: 5704833
Start: 2010-06-07 00:00:00+00:00
End: 2026-09-30 20:59:00+00:00


### 20.5 Full-history load integrity gate

Fail closed if the restored full-history dataset does not match
the completed Databento batch.

In [57]:
# ------------------------------------------------------------
# Full-history load integrity gate
# ------------------------------------------------------------

EXPECTED_FULL_HISTORY_RECORDS = 5_704_833


assert (
    len(es_history_utc)
    == EXPECTED_FULL_HISTORY_RECORDS
), (
    "Historical row count does not match "
    "the completed Databento batch."
)


assert (
    es_history_utc.index
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate historical timestamps detected."
)


assert str(
    es_history_utc.index.tz
) == "UTC", (
    "Historical index is not UTC."
)


required_columns = {
    "open",
    "high",
    "low",
    "close",
    "volume",
    "instrument_id",
}


missing_columns = (
    required_columns
    - set(es_history_utc.columns)
)


assert not missing_columns, (
    "Historical dataset is missing "
    "required columns: "
    f"{sorted(missing_columns)}"
)


print(
    "FULL-HISTORY LOAD INTEGRITY GATE: PASS"
)

FULL-HISTORY LOAD INTEGRITY GATE: PASS


### 20.5.1 Date-aware historical Databento symbology resolver

Historical Databento instrument IDs cannot be treated as globally
one-to-one with raw symbols across the full 2010–2026 interval.

Resolve each `instrument_id` using Databento's effective mapping
intervals (`d0`, `d1`) and apply the mapping valid at each bar's UTC
timestamp.

Only outright quarterly E-mini S&P 500 futures (`ES[HMUZ]Y`) are
eligible for this continuous-ES dataset.

The resolver fails closed on missing or overlapping mappings.

In [58]:
# ------------------------------------------------------------
# Date-aware historical Databento symbology resolver
# Robust timestamp-unit-safe version
# ------------------------------------------------------------

import re


ES_RAW_SYMBOL_PATTERN = re.compile(
    r"^ES[HMUZ]\d$"
)


def _mapping_date_to_utc(value):
    """
    Convert a Databento mapping date to UTC midnight.

    Databento d0/d1 mapping intervals are treated as:

        [d0, d1)

    meaning d0 is inclusive and d1 is exclusive.
    """

    ts = pd.Timestamp(value)

    if ts.tzinfo is None:
        return ts.tz_localize("UTC")

    return ts.tz_convert("UTC")


def _datetime_index_to_utc_ns_i8(values):
    """
    Normalize datetime-like values to UTC nanoseconds
    represented as int64.

    This deliberately removes pandas datetime-unit differences
    such as datetime64[us, UTC] versus datetime64[ns, UTC].
    """

    idx = pd.DatetimeIndex(values)

    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")

    # Pandas 2.x supports as_unit().
    # Fall back to astype() if necessary.
    try:
        idx = idx.as_unit("ns")
    except AttributeError:
        idx = idx.astype(
            "datetime64[ns, UTC]"
        )

    return idx.asi8


def resolve_full_history_raw_symbols(
    es_history_utc,
    chunk_size=2000,
):
    """
    Resolve historical Databento instrument IDs to the actual
    outright ES futures contract valid at every individual
    historical bar timestamp.

    Why date-aware resolution is required
    -------------------------------------
    Databento instrument IDs can be reused over long historical
    periods. Therefore:

        instrument_id -> raw_symbol

    is NOT globally one-to-one across 2010-2026.

    Correct resolution uses:

        instrument_id
        + bar timestamp
        + Databento [d0, d1) validity interval

    Only outright quarterly E-mini S&P 500 futures are eligible:

        ESHx
        ESMx
        ESUx
        ESZx

    where x is the one-digit contract-year code.

    The function fails closed if:

    - historical data are not UTC,
    - Databento returns a true unresolved request,
    - no eligible ES mapping exists for a historical instrument,
    - eligible mapping intervals overlap,
    - a historical bar falls outside all eligible mappings,
    - a non-ES symbol survives filtering,
    - or not all historical rows are accounted for.
    """

    # --------------------------------------------------------
    # 1. Input integrity
    # --------------------------------------------------------

    if not isinstance(
        es_history_utc.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "Historical index must be a DatetimeIndex."
        )

    if str(
        es_history_utc.index.tz
    ) != "UTC":
        raise RuntimeError(
            "Historical data must be UTC before "
            "historical symbol resolution."
        )

    if "instrument_id" not in (
        es_history_utc.columns
    ):
        raise RuntimeError(
            "instrument_id is missing from "
            "historical data."
        )

    if es_history_utc.empty:
        raise RuntimeError(
            "Historical dataframe is empty."
        )

    # --------------------------------------------------------
    # 2. Frozen symbology request window
    # --------------------------------------------------------

    mapping_start_date = (
        pd.Timestamp(
            HISTORY_START_UTC
        )
        .date()
        .isoformat()
    )

    mapping_end_date = (
        (
            pd.Timestamp(
                HISTORY_END_UTC
            )
            .normalize()
            + pd.Timedelta(days=1)
        )
        .date()
        .isoformat()
    )

    print(
        "Historical symbology window:",
        mapping_start_date,
        "->",
        mapping_end_date,
        "(end exclusive)",
    )

    # --------------------------------------------------------
    # 3. Historical instrument IDs actually present
    # --------------------------------------------------------

    instrument_ids = sorted(
        pd.Series(
            es_history_utc[
                "instrument_id"
            ]
        )
        .dropna()
        .astype("int64")
        .unique()
        .tolist()
    )

    if not instrument_ids:
        raise RuntimeError(
            "No historical instrument IDs found."
        )

    print(
        "Unique historical instrument IDs:",
        len(instrument_ids),
    )

    # --------------------------------------------------------
    # 4. Retrieve date-effective Databento mappings
    # --------------------------------------------------------

    mapping_rows = []

    not_found = []
    partial = []
    request_statuses = []

    for id_chunk in _chunks(
        instrument_ids,
        chunk_size,
    ):
        response = metadata_call_with_retry(
            client.symbology.resolve,
            dataset=DATASET,
            symbols=[
                int(x)
                for x in id_chunk
            ],
            stype_in="instrument_id",
            stype_out="raw_symbol",
            start_date=mapping_start_date,
            end_date=mapping_end_date,
        )

        status = int(
            response.get(
                "status",
                2,
            )
        )

        message = str(
            response.get(
                "message",
                "",
            )
        )

        request_statuses.append(
            {
                "status": status,
                "message": message,
            }
        )

        # ----------------------------------------------------
        # status 0 = fully resolved
        # status 1 = partially resolved
        #
        # Partial resolution is allowed here because historical
        # instrument IDs generally do not represent ES for the
        # entire sixteen-year requested window.
        #
        # Actual bar-level coverage is validated below.
        # ----------------------------------------------------

        if status not in {0, 1}:
            raise RuntimeError(
                "Databento symbology request failed. "
                f"status={status}, "
                f"message={message}, "
                "not_found="
                f"{response.get('not_found', [])}"
            )

        not_found.extend(
            response.get(
                "not_found",
                [],
            )
        )

        partial.extend(
            response.get(
                "partial",
                [],
            )
        )

        result = response.get(
            "result",
            {},
        )

        for instrument_id in (
            id_chunk
        ):
            entries = result.get(
                str(instrument_id),
                result.get(
                    instrument_id,
                    [],
                ),
            )

            for entry in entries:
                mapping_rows.append(
                    {
                        "instrument_id":
                            int(instrument_id),

                        "d0":
                            _mapping_date_to_utc(
                                entry["d0"]
                            ),

                        "d1":
                            _mapping_date_to_utc(
                                entry["d1"]
                            ),

                        "raw_symbol":
                            str(entry["s"]),
                    }
                )

    print(
        "Symbology request statuses:",
        request_statuses,
    )

    if not_found:
        raise RuntimeError(
            "Historical instrument IDs were "
            "not found by Databento: "
            f"{sorted(set(map(str, not_found)))}"
        )

    if not mapping_rows:
        raise RuntimeError(
            "Databento returned no historical "
            "symbology mappings."
        )

    # --------------------------------------------------------
    # 5. Complete Databento mapping table
    # --------------------------------------------------------

    all_mapping_intervals = (
        pd.DataFrame(
            mapping_rows
        )
        .drop_duplicates()
        .sort_values(
            [
                "instrument_id",
                "d0",
                "d1",
                "raw_symbol",
            ]
        )
        .reset_index(drop=True)
    )

    print(
        "All returned mapping intervals:",
        len(all_mapping_intervals),
    )

    partial_as_text = [
        str(x)
        for x in partial
    ]

    print(
        "Partially-resolved input IDs:",
        len(
            set(
                partial_as_text
            )
        ),
    )

    # --------------------------------------------------------
    # 6. Keep only outright quarterly ES contracts
    # --------------------------------------------------------

    es_mapping_intervals = (
        all_mapping_intervals.loc[
            all_mapping_intervals[
                "raw_symbol"
            ]
            .astype("string")
            .str.fullmatch(
                ES_RAW_SYMBOL_PATTERN,
                na=False,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    if es_mapping_intervals.empty:
        raise RuntimeError(
            "No valid outright quarterly ES "
            "futures mappings were returned."
        )

    print(
        "Candidate ES mapping intervals:",
        len(es_mapping_intervals),
    )

    # --------------------------------------------------------
    # 7. Resolve every historical bar
    #
    # IMPORTANT:
    #
    # We deliberately do NOT use pandas IntervalIndex here.
    #
    # Pandas may internally represent mapping dates as
    # datetime64[us, UTC] while the historical bars are
    # datetime64[ns, UTC].
    #
    # Instead, all timestamps are normalized to UTC
    # nanoseconds represented as int64.
    # --------------------------------------------------------

    resolved_raw_symbols = np.empty(
        len(es_history_utc),
        dtype=object,
    )

    resolved_raw_symbols[:] = None

    used_mapping_rows = []

    unresolved_examples = []

    grouped_positions = (
        es_history_utc
        .groupby(
            "instrument_id",
            sort=False,
        )
        .indices
    )

    for instrument_id, positions in (
        grouped_positions.items()
    ):
        instrument_id = int(
            instrument_id
        )

        positions = np.asarray(
            positions,
            dtype=np.int64,
        )

        mappings = (
            es_mapping_intervals.loc[
                es_mapping_intervals[
                    "instrument_id"
                ]
                == instrument_id
            ]
            .sort_values(
                [
                    "d0",
                    "d1",
                ]
            )
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Every instrument ID present in the continuous ES
        # dataset must have at least one eligible ES interval.
        # ----------------------------------------------------

        if mappings.empty:
            sample_times = (
                es_history_utc.index
                .take(
                    positions[:10]
                )
                .astype(str)
                .tolist()
            )

            unresolved_examples.append(
                {
                    "instrument_id":
                        instrument_id,

                    "reason":
                        "no eligible ES mapping intervals",

                    "sample_times":
                        sample_times,
                }
            )

            continue

        # ----------------------------------------------------
        # Normalize both mapping intervals and row timestamps
        # to exactly the same unit: UTC nanoseconds.
        # ----------------------------------------------------

        starts_ns = (
            _datetime_index_to_utc_ns_i8(
                mappings["d0"]
            )
        )

        ends_ns = (
            _datetime_index_to_utc_ns_i8(
                mappings["d1"]
            )
        )

        row_timestamps = (
            es_history_utc.index
            .take(
                positions
            )
        )

        row_ns = (
            _datetime_index_to_utc_ns_i8(
                row_timestamps
            )
        )

        # ----------------------------------------------------
        # Sanity-check every mapping interval.
        # ----------------------------------------------------

        invalid_interval_mask = (
            ends_ns
            <= starts_ns
        )

        if invalid_interval_mask.any():
            raise RuntimeError(
                "Invalid date-effective mapping "
                "interval detected for instrument_id "
                f"{instrument_id}.\n\n"
                f"{mappings.to_string(index=False)}"
            )

        # ----------------------------------------------------
        # Check for overlapping ES intervals.
        #
        # Adjacent intervals where:
        #
        # previous d1 == next d0
        #
        # are allowed.
        # ----------------------------------------------------

        if len(starts_ns) > 1:
            overlap_mask = (
                starts_ns[1:]
                < ends_ns[:-1]
            )

            if overlap_mask.any():
                raise RuntimeError(
                    "Overlapping date-effective ES "
                    "mappings detected for "
                    f"instrument_id {instrument_id}."
                    "\n\n"
                    f"{mappings.to_string(index=False)}"
                )

        # ----------------------------------------------------
        # Find the latest mapping whose d0 is <= each bar.
        # ----------------------------------------------------

        mapping_locations = (
            np.searchsorted(
                starts_ns,
                row_ns,
                side="right",
            )
            - 1
        )

        safe_locations = np.clip(
            mapping_locations,
            0,
            len(ends_ns) - 1,
        )

        valid_mapping_mask = (
            (mapping_locations >= 0)
            &
            (
                row_ns
                < ends_ns[
                    safe_locations
                ]
            )
        )

        # ----------------------------------------------------
        # Every bar must fall inside one valid [d0, d1)
        # eligible ES interval.
        # ----------------------------------------------------

        if not valid_mapping_mask.all():
            bad_positions = np.flatnonzero(
                ~valid_mapping_mask
            )

            sample_bad_positions = (
                bad_positions[:10]
            )

            sample_missing_times = (
                row_timestamps.take(
                    sample_bad_positions
                )
                .astype(str)
                .tolist()
            )

            unresolved_examples.append(
                {
                    "instrument_id":
                        instrument_id,

                    "reason":
                        "bar timestamp outside all "
                        "eligible ES mapping intervals",

                    "sample_times":
                        sample_missing_times,
                }
            )

            continue

        mapped_symbols = (
            mappings[
                "raw_symbol"
            ]
            .to_numpy(
                dtype=object
            )[
                mapping_locations
            ]
        )

        # ----------------------------------------------------
        # Defensive ES-symbol validation.
        # ----------------------------------------------------

        mapped_symbol_series = pd.Series(
            mapped_symbols,
            dtype="string",
        )

        invalid_symbol_mask = ~(
            mapped_symbol_series
            .str.fullmatch(
                ES_RAW_SYMBOL_PATTERN,
                na=False,
            )
        )

        if invalid_symbol_mask.any():
            bad_symbols = sorted(
                mapped_symbol_series.loc[
                    invalid_symbol_mask
                ]
                .dropna()
                .unique()
                .tolist()
            )

            raise RuntimeError(
                "Non-ES raw symbols survived "
                "date-aware resolution: "
                f"{bad_symbols}"
            )

        # ----------------------------------------------------
        # Assign resolved symbols to the original dataframe
        # row positions.
        # ----------------------------------------------------

        resolved_raw_symbols[
            positions
        ] = mapped_symbols

        # ----------------------------------------------------
        # Record mapping intervals actually used by bars.
        # ----------------------------------------------------

        unique_locations, counts = (
            np.unique(
                mapping_locations,
                return_counts=True,
            )
        )

        for (
            mapping_location,
            rows_matched,
        ) in zip(
            unique_locations,
            counts,
        ):
            mapping_location = int(
                mapping_location
            )

            mapping_row = mappings.iloc[
                mapping_location
            ]

            used_mapping_rows.append(
                {
                    "instrument_id":
                        instrument_id,

                    "d0":
                        mapping_row["d0"],

                    "d1":
                        mapping_row["d1"],

                    "raw_symbol":
                        mapping_row[
                            "raw_symbol"
                        ],

                    "rows_matched":
                        int(rows_matched),
                }
            )

    # --------------------------------------------------------
    # 8. Fail closed if actual historical rows remain
    #    unresolved.
    # --------------------------------------------------------

    if unresolved_examples:
        raise RuntimeError(
            "Historical ES rows could not be "
            "resolved date-effectively."
            "\n\nExamples:\n"
            f"{unresolved_examples[:10]}"
        )

    unresolved_count = int(
        sum(
            value is None
            for value in (
                resolved_raw_symbols
            )
        )
    )

    if unresolved_count != 0:
        raise RuntimeError(
            f"{unresolved_count} historical rows "
            "remain unresolved."
        )

    # --------------------------------------------------------
    # 9. Attach raw_symbol efficiently
    # --------------------------------------------------------

    resolved_df = (
        es_history_utc.copy()
    )

    resolved_df[
        "raw_symbol"
    ] = pd.Categorical(
        resolved_raw_symbols
    )

    # --------------------------------------------------------
    # 10. Build mapping audit table
    # --------------------------------------------------------

    if not used_mapping_rows:
        raise RuntimeError(
            "No date-effective mappings were used."
        )

    used_contract_map = (
        pd.DataFrame(
            used_mapping_rows
        )
        .groupby(
            [
                "instrument_id",
                "d0",
                "d1",
                "raw_symbol",
            ],
            as_index=False,
            observed=True,
        )[
            "rows_matched"
        ]
        .sum()
        .sort_values(
            [
                "d0",
                "instrument_id",
            ]
        )
        .reset_index(drop=True)
    )

    matched_rows = int(
        used_contract_map[
            "rows_matched"
        ]
        .sum()
    )

    if matched_rows != len(
        resolved_df
    ):
        raise RuntimeError(
            "Date-aware mapping row-count "
            "mismatch: "
            f"{matched_rows:,} mapped vs "
            f"{len(resolved_df):,} "
            "historical rows."
        )

    missing_raw_symbols = int(
        resolved_df[
            "raw_symbol"
        ]
        .isna()
        .sum()
    )

    if missing_raw_symbols != 0:
        raise RuntimeError(
            f"{missing_raw_symbols} resolved "
            "rows still have missing raw_symbol."
        )

    # --------------------------------------------------------
    # 11. Final diagnostics
    # --------------------------------------------------------

    print(
        "Date-aware mappings actually used:",
        len(used_contract_map),
    )

    print(
        "Rows matched by mappings:",
        matched_rows,
    )

    print(
        "Missing raw_symbol values:",
        missing_raw_symbols,
    )

    print(
        "DATE-AWARE HISTORICAL "
        "SYMBOLOGY RESOLUTION: PASS"
    )

    return (
        resolved_df,
        used_contract_map,
    )


print(
    "Date-aware historical symbology "
    "resolver defined."
)

Date-aware historical symbology resolver defined.


### 20.6 Resolve historical Databento instrument IDs to raw ES contracts

Resolve every historical `instrument_id` to the actual raw
ES futures contract.

This step fails closed if an instrument ID is historically ambiguous.

In [59]:
# ------------------------------------------------------------
# Resolve historical instrument IDs -> raw ES contracts
# ------------------------------------------------------------

es_history_utc, full_history_contract_map = (
    resolve_full_history_raw_symbols(
        es_history_utc
    )
)


print(
    "Resolved contract mappings:",
    len(full_history_contract_map),
)


display(
    full_history_contract_map
)


print(
    "Missing raw_symbol values:",
    es_history_utc[
        "raw_symbol"
    ]
    .isna()
    .sum(),
)

Historical symbology window: 2010-06-06 -> 2026-10-01 (end exclusive)
Unique historical instrument IDs: 67
Symbology request statuses: [{'status': 1, 'message': 'Partially resolved'}]
All returned mapping intervals: 1467
Partially-resolved input IDs: 25
Candidate ES mapping intervals: 67
Date-aware mappings actually used: 67
Rows matched by mappings: 5704833
Missing raw_symbol values: 0
DATE-AWARE HISTORICAL SYMBOLOGY RESOLUTION: PASS
Resolved contract mappings: 67


,instrument_id,d0,d1,raw_symbol,rows_matched
0,3076,2010-06-04 00:00:00+00:00,2011-02-17 00:00:00+00:00,ESZ0,88324
1,6640,2010-06-04 00:00:00+00:00,2011-02-18 00:00:00+00:00,ESM0,6874
2,26714,2010-06-04 00:00:00+00:00,2011-04-13 00:00:00+00:00,ESU0,88188
3,49278,2010-06-04 00:00:00+00:00,2011-10-18 00:00:00+00:00,ESU1,88725
4,70248,2010-06-04 00:00:00+00:00,2011-06-19 00:00:00+00:00,ESH1,86323
...,...,...,...,...,...
62,4916,2023-03-05 00:00:00+00:00,2026-09-02 00:00:00+00:00,ESM5,88078
63,14160,2023-06-04 00:00:00+00:00,2026-10-01 00:00:00+00:00,ESU5,88728
64,42140864,2023-08-17 00:00:00+00:00,2026-07-02 00:00:00+00:00,ESM6,88990
65,42140870,2023-08-17 00:00:00+00:00,2026-10-01 00:00:00+00:00,ESU6,88968


Missing raw_symbol values: 0


### 20.7 Historical contract-resolution integrity gate

Validate that the date-aware Databento symbology resolution accounts for
every historical minute bar and contains only eligible quarterly ES futures.

This is a fail-closed checkpoint before timezone conversion.

In [60]:
# ------------------------------------------------------------
# Historical contract-resolution integrity gate
# ------------------------------------------------------------

assert "raw_symbol" in es_history_utc.columns, (
    "raw_symbol was not added to historical data."
)


missing_raw_symbols = int(
    es_history_utc[
        "raw_symbol"
    ]
    .isna()
    .sum()
)


assert missing_raw_symbols == 0, (
    f"{missing_raw_symbols} historical rows "
    "have unresolved raw symbols."
)


mapped_rows = int(
    full_history_contract_map[
        "rows_matched"
    ]
    .sum()
)


assert mapped_rows == len(
    es_history_utc
), (
    "Historical mapping intervals do not "
    "account for every bar. "
    f"Mapped={mapped_rows:,}; "
    f"expected={len(es_history_utc):,}."
)


assert (
    full_history_contract_map[
        "rows_matched"
    ] > 0
).all(), (
    "At least one contract mapping has "
    "zero matched historical rows."
)


bad_raw_symbols = (
    full_history_contract_map.loc[
        ~full_history_contract_map[
            "raw_symbol"
        ]
        .astype("string")
        .str.fullmatch(
            ES_RAW_SYMBOL_PATTERN,
            na=False,
        ),
        "raw_symbol",
    ]
    .astype(str)
    .unique()
    .tolist()
)


assert not bad_raw_symbols, (
    "Non-ES quarterly futures symbols found "
    "in historical contract map: "
    f"{bad_raw_symbols}"
)


print(
    "Mapping intervals used:",
    len(full_history_contract_map),
)

print(
    "Rows accounted for:",
    mapped_rows,
)

print(
    "Missing raw symbols:",
    missing_raw_symbols,
)

print(
    "First mapping date:",
    full_history_contract_map[
        "d0"
    ].min(),
)

print(
    "Last mapping end:",
    full_history_contract_map[
        "d1"
    ].max(),
)

print(
    "HISTORICAL CONTRACT RESOLUTION: PASS"
)

Mapping intervals used: 67
Rows accounted for: 5704833
Missing raw symbols: 0
First mapping date: 2010-06-04 00:00:00+00:00
Last mapping end: 2026-10-01 00:00:00+00:00
HISTORICAL CONTRACT RESOLUTION: PASS


### 20.8 Convert verified historical bars from UTC to America/New_York

Convert the fully validated historical ES minute data from UTC to
`America/New_York` for exchange-session construction.

The timezone conversion must preserve all rows, keep timestamps unique,
and preserve the resolved raw contract symbols.

In [61]:
# ------------------------------------------------------------
# Convert verified historical bars UTC -> America/New_York
# ------------------------------------------------------------

es_history_ny = to_new_york(
    es_history_utc
)


assert (
    len(es_history_ny)
    == len(es_history_utc)
), (
    "Row count changed during "
    "timezone conversion."
)


assert (
    str(es_history_ny.index.tz)
    == "America/New_York"
), (
    "Historical analysis dataframe "
    "is not in America/New_York."
)


assert (
    es_history_ny.index
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate timestamps appeared "
    "during timezone conversion."
)


assert (
    es_history_ny[
        "raw_symbol"
    ]
    .isna()
    .sum()
    == 0
), (
    "raw_symbol became incomplete "
    "during timezone conversion."
)


print(
    "Historical rows:",
    len(es_history_ny),
)

print(
    "Historical NY start:",
    es_history_ny.index.min(),
)

print(
    "Historical NY end:",
    es_history_ny.index.max(),
)

print(
    "Timezone:",
    es_history_ny.index.tz,
)

print(
    "Missing raw_symbol values:",
    es_history_ny[
        "raw_symbol"
    ]
    .isna()
    .sum(),
)

print(
    "FULL-HISTORY TIMEZONE CONVERSION: PASS"
)

Historical rows: 5704833
Historical NY start: 2010-06-06 20:00:00-04:00
Historical NY end: 2026-09-30 16:59:00-04:00
Timezone: America/New_York
Missing raw_symbol values: 0
FULL-HISTORY TIMEZONE CONVERSION: PASS


### 20.9 Phase B raw-history checkpoint

Confirm that the complete historical Databento acquisition and validation
layer is ready for session-level processing.

Phase C may begin only after this checkpoint passes.

In [62]:
# ------------------------------------------------------------
# Phase B raw-history checkpoint
# ------------------------------------------------------------

phase_b_checkpoint = pd.Series({
    "job_id":
        full_history_job_id,

    "dbn_files":
        len(full_history_files),

    "rows":
        len(es_history_ny),

    "utc_start":
        es_history_utc.index.min(),

    "utc_end":
        es_history_utc.index.max(),

    "ny_start":
        es_history_ny.index.min(),

    "ny_end":
        es_history_ny.index.max(),

    "instrument_ids":
        es_history_utc[
            "instrument_id"
        ]
        .nunique(),

    "mapping_intervals_used":
        len(full_history_contract_map),

    "unique_raw_symbol_labels":
        es_history_ny[
            "raw_symbol"
        ]
        .nunique(),

    "mapped_rows":
        int(
            full_history_contract_map[
                "rows_matched"
            ]
            .sum()
        ),

    "missing_raw_symbols":
        int(
            es_history_ny[
                "raw_symbol"
            ]
            .isna()
            .sum()
        ),

    "duplicate_timestamps":
        int(
            es_history_ny.index
            .duplicated()
            .sum()
        ),
})


display(
    phase_b_checkpoint
)


assert int(
    phase_b_checkpoint["rows"]
) == 5_704_833, (
    "Phase B historical row count failed."
)


assert int(
    phase_b_checkpoint["dbn_files"]
) == 17, (
    "Phase B expected 17 historical DBN files."
)


assert int(
    phase_b_checkpoint["instrument_ids"]
) == 67, (
    "Unexpected historical instrument-ID count."
)


assert int(
    phase_b_checkpoint[
        "mapping_intervals_used"
    ]
) == 67, (
    "Unexpected number of used mapping intervals."
)


assert int(
    phase_b_checkpoint[
        "mapped_rows"
    ]
) == 5_704_833, (
    "Phase B mapping coverage is incomplete."
)


assert int(
    phase_b_checkpoint[
        "missing_raw_symbols"
    ]
) == 0, (
    "Phase B contains unresolved raw symbols."
)


assert int(
    phase_b_checkpoint[
        "duplicate_timestamps"
    ]
) == 0, (
    "Phase B contains duplicate timestamps."
)


print(
    "PHASE B RAW-HISTORY CHECKPOINT: PASS"
)

job_id                       GLBX-20261006-RVP5LPX9NY
dbn_files                                          17
rows                                          5704833
utc_start                   2010-06-07 00:00:00+00:00
utc_end                     2026-09-30 20:59:00+00:00
ny_start                    2010-06-06 20:00:00-04:00
ny_end                      2026-09-30 16:59:00-04:00
instrument_ids                                     67
mapping_intervals_used                             67
unique_raw_symbol_labels                           40
mapped_rows                                   5704833
missing_raw_symbols                                 0
duplicate_timestamps                                0
dtype: object

PHASE B RAW-HISTORY CHECKPOINT: PASS


# Phase C — one-row-per-session historical table

Phase B has passed all raw-history integrity checks.

Phase C converts the validated minute history into one row per historical
Headline Arena analog session.

The v0.1 historical target window is fixed to:

- start: `18:00 America/New_York` on the previous calendar day,
- end: `17:00 America/New_York` on the session date,
- final eligible minute: `16:59 ET`.

This fixed window reproduces the timing contract used by the validated
Headline Arena challenges.

Historical CME trading schedules changed during the sample. In particular,
normal ES sessions contained a scheduled 15-minute pause before June 28,
2021. From trade date June 28, 2021 onward that pause was eliminated.

Therefore normal expected minute counts are:

- before `2021-06-28`: 1,365 bars,
- on/after `2021-06-28`: 1,380 bars.

Holiday, shortened, incomplete, and roll-transition sessions are not silently
accepted. They are explicitly flagged for review before target construction.


In [63]:
# ------------------------------------------------------------
# Headline Arena analog session-date helper
# ------------------------------------------------------------

ARENA_SESSION_START_TIME = "18:00:00"
ARENA_SESSION_END_TIME = "17:00:00"
ARENA_EXPECTED_LAST_BAR_TIME = "16:59:00"

# CME eliminated the regular 15-minute equity-index pause
# effective trade date Monday, 2021-06-28.
ES_PAUSE_ELIMINATION_SESSION_DATE = pd.Timestamp(
    "2021-06-28"
)


def add_arena_session_date(df_ny):
    """
    Assign each timestamp to the calendar date on which the
    18:00 ET -> 17:00 ET Arena analog session ends.

    The +6 hour shift is only the grouping key. The exact
    [18:00, 17:00) time window is enforced later.
    """

    out = df_ny.copy()

    out["session_date"] = (
        out.index
        + pd.Timedelta(hours=6)
    ).date

    return out


def expected_arena_bar_count(session_date):
    """
    Expected number of one-minute bars in a normal Arena
    analog session.

    Before 2021-06-28:
        23 hours = 1380 minutes
        minus scheduled 15-minute CME pause
        = 1365

    From 2021-06-28 onward:
        pause eliminated
        = 1380
    """

    session_date = pd.Timestamp(
        session_date
    ).normalize()

    if (
        session_date
        < ES_PAUSE_ELIMINATION_SESSION_DATE
    ):
        return 1365

    return 1380


print(
    "Arena session helpers ready."
)

Arena session helpers ready.


## 21. Build historical Arena-session summaries

Construct one diagnostic row for every candidate historical session.

Each group is explicitly restricted to the frozen Arena analog window:

`18:00 ET previous day <= timestamp < 17:00 ET session day`

The summary records:

- observed and expected boundaries,
- expected and observed bar counts,
- historical pause regime,
- OHLCV values,
- session return,
- actual raw ES contract(s),
- roll transitions,
- bars occurring outside the Arena target window,
- whether the session requires manual review.

No target is created yet.

In [64]:
# ------------------------------------------------------------
# Build historical Arena-session summaries
# ------------------------------------------------------------

def build_session_summaries(df_ny):
    """
    Collapse validated minute history into one row per
    Headline Arena analog session.

    Target window:
        [18:00 ET previous day, 17:00 ET session day)

    Holiday, shortened, incomplete, and roll-contaminated
    sessions are flagged rather than silently accepted.
    """

    if str(df_ny.index.tz) != ANALYSIS_TIMEZONE:
        raise RuntimeError(
            "Session construction requires "
            f"{ANALYSIS_TIMEZONE} timestamps."
        )

    work = add_arena_session_date(
        df_ny
    )

    rows = []

    for session_date, g_all in work.groupby(
        "session_date",
        sort=True,
    ):
        g_all = g_all.sort_index()

        if g_all.empty:
            continue

        session_date = pd.Timestamp(
            session_date
        ).normalize()

        previous_date = (
            session_date
            - pd.Timedelta(days=1)
        )

        expected_start_et = pd.Timestamp(
            f"{previous_date.date()} "
            f"{ARENA_SESSION_START_TIME}",
            tz=ANALYSIS_TIMEZONE,
        )

        expected_end_exclusive_et = pd.Timestamp(
            f"{session_date.date()} "
            f"{ARENA_SESSION_END_TIME}",
            tz=ANALYSIS_TIMEZONE,
        )

        expected_last_bar_et = pd.Timestamp(
            f"{session_date.date()} "
            f"{ARENA_EXPECTED_LAST_BAR_TIME}",
            tz=ANALYSIS_TIMEZONE,
        )

        # ----------------------------------------------------
        # Enforce exact Headline Arena analog window.
        #
        # This deliberately excludes any historical bars at
        # or after 17:00 ET even if older CME schedules traded
        # later.
        # ----------------------------------------------------

        window_mask = (
            (g_all.index >= expected_start_et)
            &
            (
                g_all.index
                < expected_end_exclusive_et
            )
        )

        g = g_all.loc[
            window_mask
        ].copy()

        outside_window_bars = int(
            len(g_all)
            - len(g)
        )

        if g.empty:
            rows.append(
                {
                    "session_date":
                        session_date,

                    "expected_session_start_et":
                        expected_start_et,

                    "expected_session_end_exclusive_et":
                        expected_end_exclusive_et,

                    "expected_last_bar_et":
                        expected_last_bar_et,

                    "session_start_observed_et":
                        pd.NaT,

                    "session_end_last_bar_observed_et":
                        pd.NaT,

                    "first_bar_on_expected_boundary":
                        False,

                    "last_bar_on_expected_boundary":
                        False,

                    "open":
                        np.nan,

                    "high":
                        np.nan,

                    "low":
                        np.nan,

                    "close":
                        np.nan,

                    "volume":
                        np.nan,

                    "session_return":
                        np.nan,

                    "bar_count":
                        0,

                    "expected_bar_count":
                        expected_arena_bar_count(
                            session_date
                        ),

                    "expected_bar_count_match":
                        False,

                    "bar_count_delta":
                        -expected_arena_bar_count(
                            session_date
                        ),

                    "scheduled_pause_regime":
                        (
                            "15m_pause"
                            if session_date
                            < ES_PAUSE_ELIMINATION_SESSION_DATE
                            else "no_regular_pause"
                        ),

                    "underlying_symbols":
                        [],

                    "multiple_underlying_symbols":
                        False,

                    "outside_arena_window_bars":
                        outside_window_bars,
                }
            )

            continue

        expected_bar_count = (
            expected_arena_bar_count(
                session_date
            )
        )

        symbols = (
            g["raw_symbol"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        session_open = float(
            g.iloc[0]["open"]
        )

        session_close = float(
            g.iloc[-1]["close"]
        )

        rows.append(
            {
                "session_date":
                    session_date,

                "expected_session_start_et":
                    expected_start_et,

                "expected_session_end_exclusive_et":
                    expected_end_exclusive_et,

                "expected_last_bar_et":
                    expected_last_bar_et,

                "session_start_observed_et":
                    g.index[0],

                "session_end_last_bar_observed_et":
                    g.index[-1],

                "first_bar_on_expected_boundary":
                    bool(
                        g.index[0]
                        == expected_start_et
                    ),

                "last_bar_on_expected_boundary":
                    bool(
                        g.index[-1]
                        == expected_last_bar_et
                    ),

                "open":
                    session_open,

                "high":
                    float(
                        g["high"].max()
                    ),

                "low":
                    float(
                        g["low"].min()
                    ),

                "close":
                    session_close,

                "volume":
                    float(
                        g["volume"].sum()
                    ),

                "session_return":
                    (
                        session_close
                        / session_open
                        - 1.0
                    ),

                "bar_count":
                    int(
                        len(g)
                    ),

                "expected_bar_count":
                    expected_bar_count,

                "expected_bar_count_match":
                    bool(
                        len(g)
                        == expected_bar_count
                    ),

                "bar_count_delta":
                    int(
                        len(g)
                        - expected_bar_count
                    ),

                "scheduled_pause_regime":
                    (
                        "15m_pause"
                        if session_date
                        < ES_PAUSE_ELIMINATION_SESSION_DATE
                        else "no_regular_pause"
                    ),

                "underlying_symbols":
                    symbols,

                "multiple_underlying_symbols":
                    bool(
                        len(symbols) > 1
                    ),

                "outside_arena_window_bars":
                    outside_window_bars,
            }
        )

    sessions = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "session_date"
        )
        .reset_index(
            drop=True
        )
    )

    sessions[
        "primary_symbol"
    ] = (
        sessions[
            "underlying_symbols"
        ]
        .apply(
            lambda xs: (
                xs[0]
                if isinstance(
                    xs,
                    list,
                )
                and len(xs)
                else None
            )
        )
    )

    sessions[
        "roll_flag"
    ] = (
        sessions[
            "primary_symbol"
        ]
        .ne(
            sessions[
                "primary_symbol"
            ]
            .shift(1)
        )
    )

    if len(sessions):
        sessions.loc[
            0,
            "roll_flag",
        ] = False

    sessions[
        "needs_session_review"
    ] = (
        ~sessions[
            "first_bar_on_expected_boundary"
        ]
        |
        ~sessions[
            "last_bar_on_expected_boundary"
        ]
        |
        ~sessions[
            "expected_bar_count_match"
        ]
        |
        sessions[
            "multiple_underlying_symbols"
        ]
        |
        sessions[
            "session_return"
        ]
        .isna()
    )

    return sessions


print(
    "Arena session-summary builder ready."
)

Arena session-summary builder ready.


### 21.1 Build the raw historical session table

Collapse the validated minute history into one row per Arena analog session.

This is still a diagnostic table. No session is excluded and no target is
created yet.

In [65]:
# ------------------------------------------------------------
# Build raw Arena analog session table
# ------------------------------------------------------------

es_sessions_raw = (
    build_session_summaries(
        es_history_ny
    )
)


assert not (
    es_sessions_raw[
        "session_date"
    ]
    .duplicated()
    .any()
), (
    "Duplicate session_date values detected."
)


print(
    "Historical session rows:",
    len(es_sessions_raw),
)

print(
    "First session date:",
    es_sessions_raw[
        "session_date"
    ].min(),
)

print(
    "Last session date:",
    es_sessions_raw[
        "session_date"
    ].max(),
)

print(
    "Duplicate session dates:",
    es_sessions_raw[
        "session_date"
    ]
    .duplicated()
    .sum(),
)


display(
    es_sessions_raw.head()
)

display(
    es_sessions_raw.tail()
)

Historical session rows: 4203
First session date: 2010-06-07 00:00:00
Last session date: 2026-09-30 00:00:00
Duplicate session dates: 0


,session_date,expected_session_start_et,expected_session_end_exclusive_et,expected_last_bar_et,session_start_observed_et,session_end_last_bar_observed_et,first_bar_on_expected_boundary,last_bar_on_expected_boundary,open,high,...,expected_bar_count,expected_bar_count_match,bar_count_delta,scheduled_pause_regime,underlying_symbols,multiple_underlying_symbols,outside_arena_window_bars,primary_symbol,roll_flag,needs_session_review
0,2010-06-07,2010-06-06 18:00:00-04:00,2010-06-07 17:00:00-04:00,2010-06-07 16:59:00-04:00,2010-06-06 20:00:00-04:00,2010-06-07 16:59:00-04:00,False,True,1061.25,1070.75,...,1365,False,-120,15m_pause,[ESM0],False,30,ESM0,False,True
1,2010-06-08,2010-06-07 18:00:00-04:00,2010-06-08 17:00:00-04:00,2010-06-08 16:59:00-04:00,2010-06-07 18:00:00-04:00,2010-06-08 16:59:00-04:00,True,True,1049.00,1063.25,...,1365,False,-2,15m_pause,[ESM0],False,30,ESM0,False,True
2,2010-06-09,2010-06-08 18:00:00-04:00,2010-06-09 17:00:00-04:00,2010-06-09 16:59:00-04:00,2010-06-08 18:00:00-04:00,2010-06-09 16:59:00-04:00,True,True,1060.50,1077.75,...,1365,True,0,15m_pause,[ESM0],False,30,ESM0,False,False
3,2010-06-10,2010-06-09 18:00:00-04:00,2010-06-10 17:00:00-04:00,2010-06-10 16:59:00-04:00,2010-06-09 18:00:00-04:00,2010-06-10 16:59:00-04:00,True,True,1052.50,1087.75,...,1365,False,-2,15m_pause,[ESM0],False,26,ESM0,False,True
4,2010-06-11,2010-06-10 18:00:00-04:00,2010-06-11 17:00:00-04:00,2010-06-11 16:59:00-04:00,2010-06-10 18:00:00-04:00,2010-06-11 16:14:00-04:00,True,False,1085.50,1092.25,...,1365,False,-52,15m_pause,[ESM0],False,0,ESM0,False,True


,session_date,expected_session_start_et,expected_session_end_exclusive_et,expected_last_bar_et,session_start_observed_et,session_end_last_bar_observed_et,first_bar_on_expected_boundary,last_bar_on_expected_boundary,open,high,...,expected_bar_count,expected_bar_count_match,bar_count_delta,scheduled_pause_regime,underlying_symbols,multiple_underlying_symbols,outside_arena_window_bars,primary_symbol,roll_flag,needs_session_review
4198,2026-09-24,2026-09-23 18:00:00-04:00,2026-09-24 17:00:00-04:00,2026-09-24 16:59:00-04:00,2026-09-23 18:00:00-04:00,2026-09-24 16:59:00-04:00,True,True,7774.75,7783.50,...,1380,True,0,no_regular_pause,[ESZ6],False,0,ESZ6,False,False
4199,2026-09-25,2026-09-24 18:00:00-04:00,2026-09-25 17:00:00-04:00,2026-09-25 16:59:00-04:00,2026-09-24 18:00:00-04:00,2026-09-25 16:59:00-04:00,True,True,7759.00,7814.75,...,1380,True,0,no_regular_pause,[ESZ6],False,0,ESZ6,False,False
4200,2026-09-28,2026-09-27 18:00:00-04:00,2026-09-28 17:00:00-04:00,2026-09-28 16:59:00-04:00,2026-09-27 18:00:00-04:00,2026-09-28 16:59:00-04:00,True,True,7796.00,7803.00,...,1380,True,0,no_regular_pause,[ESZ6],False,0,ESZ6,False,False
4201,2026-09-29,2026-09-28 18:00:00-04:00,2026-09-29 17:00:00-04:00,2026-09-29 16:59:00-04:00,2026-09-28 18:00:00-04:00,2026-09-29 16:59:00-04:00,True,True,7746.50,7770.75,...,1380,True,0,no_regular_pause,[ESZ6],False,0,ESZ6,False,False
4202,2026-09-30,2026-09-29 18:00:00-04:00,2026-09-30 17:00:00-04:00,2026-09-30 16:59:00-04:00,2026-09-29 18:00:00-04:00,2026-09-30 16:59:00-04:00,True,True,7739.00,7782.00,...,1380,True,0,no_regular_pause,[ESZ6],False,0,ESZ6,False,False


### 21.2 Session-structure diagnostics

Measure how many sessions exactly match the expected Arena analog structure
and how many require review because of boundaries, bar counts, holidays,
data gaps, or contract transitions.

In [66]:
# ------------------------------------------------------------
# Session-structure diagnostics
# ------------------------------------------------------------

session_diagnostics = pd.Series({
    "session_rows":
        len(es_sessions_raw),

    "duplicate_session_dates":
        int(
            es_sessions_raw[
                "session_date"
            ]
            .duplicated()
            .sum()
        ),

    "review_rows":
        int(
            es_sessions_raw[
                "needs_session_review"
            ]
            .sum()
        ),

    "missing_expected_start":
        int(
            (
                ~es_sessions_raw[
                    "first_bar_on_expected_boundary"
                ]
            )
            .sum()
        ),

    "missing_expected_end":
        int(
            (
                ~es_sessions_raw[
                    "last_bar_on_expected_boundary"
                ]
            )
            .sum()
        ),

    "unexpected_bar_count":
        int(
            (
                ~es_sessions_raw[
                    "expected_bar_count_match"
                ]
            )
            .sum()
        ),

    "multiple_symbol_sessions":
        int(
            es_sessions_raw[
                "multiple_underlying_symbols"
            ]
            .sum()
        ),

    "roll_flag_rows":
        int(
            es_sessions_raw[
                "roll_flag"
            ]
            .fillna(False)
            .sum()
        ),

    "outside_arena_window_bars":
        int(
            es_sessions_raw[
                "outside_arena_window_bars"
            ]
            .sum()
        ),

    "missing_session_return":
        int(
            es_sessions_raw[
                "session_return"
            ]
            .isna()
            .sum()
        ),
})


display(
    session_diagnostics
)


print(
    "\nExpected bar-count regime:"
)

display(
    pd.crosstab(
        es_sessions_raw[
            "scheduled_pause_regime"
        ],
        es_sessions_raw[
            "expected_bar_count_match"
        ],
        margins=True,
    )
)


print(
    "\nMost common observed bar counts:"
)

display(
    es_sessions_raw[
        "bar_count"
    ]
    .value_counts()
    .head(20)
    .rename_axis(
        "bar_count"
    )
    .to_frame(
        "sessions"
    )
)

session_rows                  4203
duplicate_session_dates          0
review_rows                   2428
missing_expected_start          14
missing_expected_end           270
unexpected_bar_count          2412
multiple_symbol_sessions        66
roll_flag_rows                  68
outside_arena_window_bars    24656
missing_session_return           1
dtype: int64


Expected bar-count regime:


expected_bar_count_match,False,True,All
scheduled_pause_regime,,,
15m_pause,2100,742,2842
no_regular_pause,312,1049,1361
All,2412,1791,4203



Most common observed bar counts:


,sessions
bar_count,
1380,1049
1365,742
1364,233
1379,145
1363,127
1362,125
1360,100
1361,92
1358,80


### 21.3 Review every structurally unusual session

Do not automatically drop or repair flagged sessions.

First inspect whether each exception is explained by:

- a holiday or shortened CME session,
- the beginning/end of the acquired history,
- a genuine missing-data problem,
- a contract-roll transition,
- or another exchange-calendar exception.

In [67]:
# ------------------------------------------------------------
# Review flagged historical sessions
# ------------------------------------------------------------

review_sessions = (
    es_sessions_raw.loc[
        es_sessions_raw[
            "needs_session_review"
        ]
    ]
    .copy()
)


review_columns = [
    "session_date",
    "session_start_observed_et",
    "session_end_last_bar_observed_et",
    "bar_count",
    "expected_bar_count",
    "bar_count_delta",
    "first_bar_on_expected_boundary",
    "last_bar_on_expected_boundary",
    "underlying_symbols",
    "multiple_underlying_symbols",
    "roll_flag",
    "outside_arena_window_bars",
]


print(
    "Sessions requiring review:",
    len(review_sessions),
)


display(
    review_sessions[
        review_columns
    ].head(30)
)


display(
    review_sessions[
        review_columns
    ].tail(30)
)

Sessions requiring review: 2428


,session_date,session_start_observed_et,session_end_last_bar_observed_et,bar_count,expected_bar_count,bar_count_delta,first_bar_on_expected_boundary,last_bar_on_expected_boundary,underlying_symbols,multiple_underlying_symbols,roll_flag,outside_arena_window_bars
0,2010-06-07,2010-06-06 20:00:00-04:00,2010-06-07 16:59:00-04:00,1245,1365,-120,False,True,[ESM0],False,False,30
1,2010-06-08,2010-06-07 18:00:00-04:00,2010-06-08 16:59:00-04:00,1363,1365,-2,True,True,[ESM0],False,False,30
3,2010-06-10,2010-06-09 18:00:00-04:00,2010-06-10 16:59:00-04:00,1363,1365,-2,True,True,[ESM0],False,False,26
4,2010-06-11,2010-06-10 18:00:00-04:00,2010-06-11 16:14:00-04:00,1313,1365,-52,True,False,[ESM0],False,False,0
5,2010-06-14,2010-06-13 18:00:00-04:00,2010-06-14 16:59:00-04:00,1337,1365,-28,True,True,"[ESM0, ESU0]",True,False,30
6,2010-06-15,2010-06-14 18:00:00-04:00,2010-06-15 16:59:00-04:00,1336,1365,-29,True,True,[ESU0],False,True,30
7,2010-06-16,2010-06-15 18:00:00-04:00,2010-06-16 16:59:00-04:00,1345,1365,-20,True,True,[ESU0],False,False,30
8,2010-06-17,2010-06-16 18:00:00-04:00,2010-06-17 16:59:00-04:00,1355,1365,-10,True,True,[ESU0],False,False,30
9,2010-06-18,2010-06-17 18:00:00-04:00,2010-06-18 16:14:00-04:00,1320,1365,-45,True,False,[ESU0],False,False,0
10,2010-06-21,2010-06-20 18:00:00-04:00,2010-06-21 16:59:00-04:00,1364,1365,-1,True,True,[ESU0],False,False,29


,session_date,session_start_observed_et,session_end_last_bar_observed_et,bar_count,expected_bar_count,bar_count_delta,first_bar_on_expected_boundary,last_bar_on_expected_boundary,underlying_symbols,multiple_underlying_symbols,roll_flag,outside_arena_window_bars
3939,2025-09-24,2025-09-23 18:00:00-04:00,2025-09-24 16:59:00-04:00,1379,1380,-1,True,True,[ESZ5],False,False,0
3943,2025-09-30,2025-09-29 18:00:00-04:00,2025-09-30 16:59:00-04:00,1379,1380,-1,True,True,[ESZ5],False,False,0
3945,2025-10-02,2025-10-01 18:00:00-04:00,2025-10-02 16:59:00-04:00,1378,1380,-2,True,True,[ESZ5],False,False,0
3947,2025-10-06,2025-10-05 18:00:00-04:00,2025-10-06 16:59:00-04:00,1378,1380,-2,True,True,[ESZ5],False,False,0
3948,2025-10-07,2025-10-06 18:00:00-04:00,2025-10-07 16:59:00-04:00,1379,1380,-1,True,True,[ESZ5],False,False,0
3950,2025-10-09,2025-10-08 18:00:00-04:00,2025-10-09 16:59:00-04:00,1379,1380,-1,True,True,[ESZ5],False,False,0
3985,2025-11-27,2025-11-26 18:00:00-05:00,2025-11-27 12:59:00-05:00,1139,1380,-241,True,False,[ESZ5],False,False,0
3986,2025-11-28,2025-11-27 18:00:00-05:00,2025-11-28 13:14:00-05:00,510,1380,-870,True,False,[ESZ5],False,False,0
3999,2025-12-17,2025-12-16 18:00:00-05:00,2025-12-17 16:59:00-05:00,1380,1380,0,True,True,"[ESZ5, ESH6]",True,False,0
4004,2025-12-24,2025-12-23 18:00:00-05:00,2025-12-24 13:14:00-05:00,1153,1380,-227,True,False,[ESH6],False,False,0


### 21.4 Contract-roll audit

Inspect sessions where the volume-ranked continuous ES contract changes,
or where more than one underlying raw contract occurs inside the fixed
Arena session window.

These rows require explicit treatment before they can enter the modeling
table.

In [68]:
# ------------------------------------------------------------
# Contract-roll audit
# ------------------------------------------------------------

roll_review = (
    es_sessions_raw.loc[
        es_sessions_raw[
            "roll_flag"
        ]
        |
        es_sessions_raw[
            "multiple_underlying_symbols"
        ]
    ]
    .copy()
)


roll_columns = [
    "session_date",
    "primary_symbol",
    "underlying_symbols",
    "multiple_underlying_symbols",
    "roll_flag",
    "bar_count",
    "expected_bar_count",
    "session_return",
    "needs_session_review",
]


print(
    "Roll-related sessions:",
    len(roll_review),
)

print(
    "Sessions containing multiple raw contracts:",
    int(
        roll_review[
            "multiple_underlying_symbols"
        ]
        .sum()
    ),
)


display(
    roll_review[
        roll_columns
    ].head(40)
)


display(
    roll_review[
        roll_columns
    ].tail(40)
)

Roll-related sessions: 134
Sessions containing multiple raw contracts: 66


,session_date,primary_symbol,underlying_symbols,multiple_underlying_symbols,roll_flag,bar_count,expected_bar_count,session_return,needs_session_review
5,2010-06-14,ESM0,"[ESM0, ESU0]",True,False,1337,1365,-0.004356,True
6,2010-06-15,ESU0,[ESU0],False,True,1336,1365,0.020948,True
70,2010-09-13,ESU0,"[ESU0, ESZ0]",True,False,1347,1365,0.004494,True
71,2010-09-14,ESZ0,[ESZ0],False,True,1350,1365,-0.004470,True
135,2010-12-13,ESZ0,"[ESZ0, ESH1]",True,False,1330,1365,-0.004831,True
136,2010-12-14,ESH1,[ESH1],False,True,1342,1365,0.000202,True
199,2011-03-14,ESH1,"[ESH1, ESM1]",True,False,1361,1365,-0.008447,True
200,2011-03-15,ESM1,[ESM1],False,True,1362,1365,-0.012972,True
263,2011-06-13,ESM1,"[ESM1, ESU1]",True,False,1332,1365,-0.002165,True
264,2011-06-14,ESU1,[ESU1],False,True,1335,1365,0.012229,True


,session_date,primary_symbol,underlying_symbols,multiple_underlying_symbols,roll_flag,bar_count,expected_bar_count,session_return,needs_session_review
2962,2021-12-13,ESZ1,"[ESZ1, ESH2]",True,False,1380,1380,-0.010135,True
2963,2021-12-14,ESH2,[ESH2],False,True,1380,1380,-0.007238,False
3026,2022-03-14,ESH2,"[ESH2, ESM2]",True,False,1380,1380,-0.010614,True
3027,2022-03-15,ESM2,[ESM2],False,True,1380,1380,0.018879,False
3092,2022-06-15,ESM2,"[ESM2, ESU2]",True,False,1380,1380,0.014372,True
3093,2022-06-16,ESU2,[ESU2],False,True,1380,1380,-0.031324,False
3157,2022-09-14,ESU2,"[ESU2, ESZ2]",True,False,1380,1380,0.009399,True
3158,2022-09-15,ESZ2,[ESZ2],False,True,1380,1380,-0.015609,False
3222,2022-12-14,ESZ2,"[ESZ2, ESH3]",True,False,1379,1380,0.004103,True
3223,2022-12-15,ESH3,[ESH3],False,True,1380,1380,-0.028425,False


### 21.5 Session-review summary by year

Summarize structural exceptions by year so systematic historical
trading-hour regimes can be distinguished from isolated data problems.

In [69]:
# ------------------------------------------------------------
# Session-review summary by year
# ------------------------------------------------------------

session_audit_by_year = (
    es_sessions_raw
    .assign(
        year=lambda x: (
            x["session_date"]
            .dt.year
        )
    )
    .groupby(
        "year"
    )
    .agg(
        sessions=(
            "session_date",
            "size",
        ),

        review_rows=(
            "needs_session_review",
            "sum",
        ),

        unexpected_bar_count=(
            "expected_bar_count_match",
            lambda s: (
                ~s
            ).sum(),
        ),

        missing_start=(
            "first_bar_on_expected_boundary",
            lambda s: (
                ~s
            ).sum(),
        ),

        missing_end=(
            "last_bar_on_expected_boundary",
            lambda s: (
                ~s
            ).sum(),
        ),

        multiple_symbol_sessions=(
            "multiple_underlying_symbols",
            "sum",
        ),

        roll_rows=(
            "roll_flag",
            "sum",
        ),
    )
)


display(
    session_audit_by_year
)

,sessions,review_rows,unexpected_bar_count,missing_start,missing_end,multiple_symbol_sessions,roll_rows
year,,,,,,,
2010,149,146,146,1,35,3,3
2011,258,216,216,1,60,4,4
2012,258,249,248,2,56,4,4
2013,252,243,243,3,5,4,4
2014,250,239,239,3,8,4,4
2015,259,226,225,1,9,4,4
2016,258,187,186,0,7,4,4
2017,257,246,246,0,8,4,4
2018,259,153,153,2,11,4,6


### 21.6 Freeze the v0.1 historical-session eligibility policy

The raw session audit shows that exact one-minute bar-count equality is too
strict to use as a modeling exclusion rule.

For the v0.1 Arena target, a historical session is eligible when:

- the expected `18:00 ET` opening bar exists,
- the expected `16:59 ET` final bar exists,
- the session open and close are valid positive prices,
- the session return is finite,
- exactly one raw ES contract is present inside the target window.

A bar-count mismatch remains an explicit data-quality warning, but does not
by itself invalidate an otherwise complete open-to-close target.

A session containing multiple raw contracts is excluded because its open and
close could belong to different futures contracts.

`roll_flag` by itself is diagnostic and does not automatically exclude the
following single-contract session.

In [70]:
# ------------------------------------------------------------
# Freeze v0.1 historical-session eligibility policy
# ------------------------------------------------------------

es_sessions_policy = (
    es_sessions_raw.copy()
)


# Exact bar-count equality is diagnostic, not automatically
# a reason to remove the target.
es_sessions_policy[
    "bar_count_warning"
] = (
    ~es_sessions_policy[
        "expected_bar_count_match"
    ]
)


valid_prices = (
    es_sessions_policy[
        "open"
    ].notna()
    &
    es_sessions_policy[
        "close"
    ].notna()
    &
    (
        es_sessions_policy[
            "open"
        ] > 0
    )
    &
    (
        es_sessions_policy[
            "close"
        ] > 0
    )
)


finite_return = (
    es_sessions_policy[
        "session_return"
    ]
    .notna()
    &
    np.isfinite(
        es_sessions_policy[
            "session_return"
        ]
        .astype(float)
    )
)


es_sessions_policy[
    "target_eligible"
] = (
    es_sessions_policy[
        "first_bar_on_expected_boundary"
    ]
    &
    es_sessions_policy[
        "last_bar_on_expected_boundary"
    ]
    &
    valid_prices
    &
    finite_return
    &
    (
        ~es_sessions_policy[
            "multiple_underlying_symbols"
        ]
    )
)


def historical_session_exclusion_reasons(row):
    reasons = []

    if not row[
        "first_bar_on_expected_boundary"
    ]:
        reasons.append(
            "missing_18_00_open"
        )

    if not row[
        "last_bar_on_expected_boundary"
    ]:
        reasons.append(
            "missing_16_59_close"
        )

    if (
        pd.isna(row["open"])
        or pd.isna(row["close"])
        or row["open"] <= 0
        or row["close"] <= 0
    ):
        reasons.append(
            "invalid_open_close"
        )

    if (
        pd.isna(
            row["session_return"]
        )
        or not np.isfinite(
            float(
                row["session_return"]
            )
        )
    ):
        reasons.append(
            "invalid_session_return"
        )

    if row[
        "multiple_underlying_symbols"
    ]:
        reasons.append(
            "multiple_raw_contracts"
        )

    return (
        "|".join(reasons)
        if reasons
        else ""
    )


es_sessions_policy[
    "exclusion_reasons"
] = (
    es_sessions_policy.apply(
        historical_session_exclusion_reasons,
        axis=1,
    )
)


eligible_sessions = int(
    es_sessions_policy[
        "target_eligible"
    ].sum()
)

excluded_sessions = int(
    (
        ~es_sessions_policy[
            "target_eligible"
        ]
    ).sum()
)


print(
    "Total historical sessions:",
    len(es_sessions_policy),
)

print(
    "Target-eligible sessions:",
    eligible_sessions,
)

print(
    "Excluded sessions:",
    excluded_sessions,
)

print(
    "Eligible sessions with bar-count warning:",
    int(
        (
            es_sessions_policy[
                "target_eligible"
            ]
            &
            es_sessions_policy[
                "bar_count_warning"
            ]
        ).sum()
    ),
)

print(
    "Multiple-contract sessions:",
    int(
        es_sessions_policy[
            "multiple_underlying_symbols"
        ].sum()
    ),
)

print(
    "SESSION ELIGIBILITY POLICY: BUILT"
)

Total historical sessions: 4203
Target-eligible sessions: 3855
Excluded sessions: 348
Eligible sessions with bar-count warning: 2080
Multiple-contract sessions: 66
SESSION ELIGIBILITY POLICY: BUILT


### 21.7 Audit the frozen session-eligibility policy

Review the resulting eligible/excluded population before creating the
three-class target.

The goal is to verify that exclusions are driven by meaningful structural
problems such as missing target boundaries or contract transitions rather
than ordinary missing one-minute observations.

In [71]:
# ------------------------------------------------------------
# Audit v0.1 session eligibility policy
# ------------------------------------------------------------

eligibility_audit = pd.Series({
    "total_sessions":
        len(es_sessions_policy),

    "target_eligible":
        int(
            es_sessions_policy[
                "target_eligible"
            ].sum()
        ),

    "target_excluded":
        int(
            (
                ~es_sessions_policy[
                    "target_eligible"
                ]
            ).sum()
        ),

    "eligible_with_bar_count_warning":
        int(
            (
                es_sessions_policy[
                    "target_eligible"
                ]
                &
                es_sessions_policy[
                    "bar_count_warning"
                ]
            ).sum()
        ),

    "missing_18_00_open":
        int(
            (
                ~es_sessions_policy[
                    "first_bar_on_expected_boundary"
                ]
            ).sum()
        ),

    "missing_16_59_close":
        int(
            (
                ~es_sessions_policy[
                    "last_bar_on_expected_boundary"
                ]
            ).sum()
        ),

    "multiple_raw_contracts":
        int(
            es_sessions_policy[
                "multiple_underlying_symbols"
            ].sum()
        ),

    "null_session_return":
        int(
            es_sessions_policy[
                "session_return"
            ]
            .isna()
            .sum()
        ),
})


display(
    eligibility_audit
)


print(
    "\nExclusion reasons:"
)

display(
    es_sessions_policy.loc[
        ~es_sessions_policy[
            "target_eligible"
        ],
        "exclusion_reasons",
    ]
    .value_counts()
    .rename_axis(
        "exclusion_reasons"
    )
    .to_frame(
        "sessions"
    )
)


print(
    "\nEligibility by year:"
)

eligibility_by_year = (
    es_sessions_policy
    .assign(
        year=lambda x: (
            x[
                "session_date"
            ].dt.year
        )
    )
    .groupby(
        "year"
    )
    .agg(
        sessions=(
            "session_date",
            "size",
        ),

        eligible=(
            "target_eligible",
            "sum",
        ),

        excluded=(
            "target_eligible",
            lambda s: (
                ~s
            ).sum(),
        ),

        bar_count_warnings=(
            "bar_count_warning",
            "sum",
        ),

        multi_contract=(
            "multiple_underlying_symbols",
            "sum",
        ),
    )
)


display(
    eligibility_by_year
)


print(
    "\nSample excluded sessions:"
)

display(
    es_sessions_policy.loc[
        ~es_sessions_policy[
            "target_eligible"
        ],
        [
            "session_date",
            "session_start_observed_et",
            "session_end_last_bar_observed_et",
            "bar_count",
            "expected_bar_count",
            "bar_count_delta",
            "underlying_symbols",
            "roll_flag",
            "exclusion_reasons",
        ],
    ].head(40)
)

total_sessions                     4203
target_eligible                    3855
target_excluded                     348
eligible_with_bar_count_warning    2080
missing_18_00_open                   14
missing_16_59_close                 270
multiple_raw_contracts               66
null_session_return                   1
dtype: int64


Exclusion reasons:


,sessions
exclusion_reasons,
missing_16_59_close,268
multiple_raw_contracts,65
missing_18_00_open,13
missing_18_00_open|missing_16_59_close|invalid_open_close|invalid_session_return,1
missing_16_59_close|multiple_raw_contracts,1



Eligibility by year:


,sessions,eligible,excluded,bar_count_warnings,multi_contract
year,,,,,
2010,149,110,39,146,3
2011,258,193,65,216,4
2012,258,196,62,248,4
2013,252,240,12,243,4
2014,250,235,15,239,4
2015,259,245,14,225,4
2016,258,247,11,186,4
2017,257,245,12,246,4
2018,259,243,16,153,4



Sample excluded sessions:


,session_date,session_start_observed_et,session_end_last_bar_observed_et,bar_count,expected_bar_count,bar_count_delta,underlying_symbols,roll_flag,exclusion_reasons
0,2010-06-07,2010-06-06 20:00:00-04:00,2010-06-07 16:59:00-04:00,1245,1365,-120,[ESM0],False,missing_18_00_open
4,2010-06-11,2010-06-10 18:00:00-04:00,2010-06-11 16:14:00-04:00,1313,1365,-52,[ESM0],False,missing_16_59_close
5,2010-06-14,2010-06-13 18:00:00-04:00,2010-06-14 16:59:00-04:00,1337,1365,-28,"[ESM0, ESU0]",False,multiple_raw_contracts
9,2010-06-18,2010-06-17 18:00:00-04:00,2010-06-18 16:14:00-04:00,1320,1365,-45,[ESU0],False,missing_16_59_close
14,2010-06-25,2010-06-24 18:00:00-04:00,2010-06-25 16:14:00-04:00,1332,1365,-33,[ESU0],False,missing_16_59_close
19,2010-07-02,2010-07-01 18:00:00-04:00,2010-07-02 16:14:00-04:00,1327,1365,-38,[ESU0],False,missing_16_59_close
20,2010-07-05,2010-07-04 18:00:00-04:00,2010-07-05 11:29:00-04:00,1039,1365,-326,[ESU0],False,missing_16_59_close
24,2010-07-09,2010-07-08 18:00:00-04:00,2010-07-09 16:15:00-04:00,1323,1365,-42,[ESU0],False,missing_16_59_close
29,2010-07-16,2010-07-15 18:00:00-04:00,2010-07-16 16:14:00-04:00,1326,1365,-39,[ESU0],False,missing_16_59_close
34,2010-07-23,2010-07-22 18:00:00-04:00,2010-07-23 16:14:00-04:00,1312,1365,-53,[ESU0],False,missing_16_59_close


### 21.8 Historical calendar and bar-coverage audit

Quantify historical session completeness across trading-calendar regimes,
years, and weekdays.

This is a diagnostic analysis only. It does not modify the frozen
v0.1 eligibility policy or target labels.

In [72]:

# ------------------------------------------------------------
# Historical calendar and coverage diagnostics
# ------------------------------------------------------------

coverage_audit = es_sessions_policy.copy()

coverage_audit["year"] = (
    coverage_audit["session_date"].dt.year
)

coverage_audit["weekday"] = (
    coverage_audit["session_date"].dt.day_name()
)

coverage_audit["bar_coverage_pct"] = (
    coverage_audit["bar_count"]
    / coverage_audit["expected_bar_count"]
    * 100.0
)

coverage_audit["missing_expected_bars"] = (
    coverage_audit["expected_bar_count"]
    - coverage_audit["bar_count"]
).clip(lower=0)

coverage_audit["historical_regime"] = np.select(
    [
        coverage_audit["session_date"]
        < pd.Timestamp("2012-11-19"),

        coverage_audit["session_date"]
        < pd.Timestamp("2021-06-28"),
    ],
    [
        "pre_2012_hours",
        "2012_to_2021_hours",
    ],
    default="post_2021_hours",
)

yearly_coverage = (
    coverage_audit
    .groupby("year")
    .agg(
        sessions=("session_date", "size"),
        eligible=("target_eligible", "sum"),
        mean_bar_coverage_pct=(
            "bar_coverage_pct", "mean"
        ),
        median_bar_coverage_pct=(
            "bar_coverage_pct", "median"
        ),
    )
)

yearly_coverage["excluded"] = (
    yearly_coverage["sessions"]
    - yearly_coverage["eligible"]
)

yearly_coverage["excluded_pct"] = (
    yearly_coverage["excluded"]
    / yearly_coverage["sessions"]
    * 100
)

regime_weekday_coverage = (
    coverage_audit
    .groupby(
        ["historical_regime", "weekday"]
    )
    .agg(
        sessions=("session_date", "size"),
        eligible=("target_eligible", "sum"),
        mean_bar_coverage_pct=(
            "bar_coverage_pct", "mean"
        ),
    )
)

regime_weekday_coverage["excluded_pct"] = (
    (
        regime_weekday_coverage["sessions"]
        - regime_weekday_coverage["eligible"]
    )
    / regime_weekday_coverage["sessions"]
    * 100
)

eligible_low_coverage = (
    coverage_audit.loc[
        coverage_audit["target_eligible"]
    ]
    .sort_values("bar_coverage_pct")
    .head(30)
)

print("Historical coverage by year:")
display(yearly_coverage)

print("Historical coverage by regime and weekday:")
display(regime_weekday_coverage)

print("Lowest-coverage eligible sessions:")
display(
    eligible_low_coverage[
        [
            "session_date",
            "weekday",
            "bar_count",
            "expected_bar_count",
            "bar_coverage_pct",
            "missing_expected_bars",
        ]
    ]
)

print("HISTORICAL COVERAGE DIAGNOSTIC: COMPLETE")


Historical coverage by year:


,sessions,eligible,mean_bar_coverage_pct,median_bar_coverage_pct,excluded,excluded_pct
year,,,,,,
2010,149,110,97.711729,98.901099,39,26.174497
2011,258,193,98.135843,99.487179,65,25.193798
2012,258,196,97.323168,99.084249,62,24.031008
2013,252,240,98.130705,99.047619,12,4.761905
2014,250,235,97.199414,98.608059,15,6.000000
2015,259,245,98.282490,99.267399,14,5.405405
2016,258,247,99.108953,99.780220,11,4.263566
2017,257,245,98.282522,99.120879,12,4.669261
2018,259,243,98.697159,99.926740,16,6.177606


Historical coverage by regime and weekday:


sessions  eligible  mean_bar_coverage_pct  \
historical_regime  weekday                                                
2012_to_2021_hours Friday          437       422              98.524572   
                   Monday          439       374              97.825394   
                   Sunday            1         0               0.000000   
                   Thursday        442       429              98.744137   
                   Tuesday         444       436              99.017754   
                   Wednesday       443       429              98.747798   
post_2021_hours    Friday          270       260              99.052872   
                   Monday          271       242              98.304455   
                   Thursday        272       263              99.348359   
                   Tuesday         275       273              99.826614   
                   Wednesday       273       253              99.768275   
pre_2012_hours     Friday          126         0              96.374208   
                   Monday          126       102              96.819001   
                   Thursday        128       124              98.737408   
                   Tuesday         128       123              97.945284   
                   Wednesday       128       125              99.036172   

                              excluded_pct  
historical_regime  weekday                  
2012_to_2021_hours Friday         3.432494  
                   Monday        14.806378  
                   Sunday       100.000000  
                   Thursday       2.941176  
                   Tuesday        1.801802  
                   Wednesday      3.160271  
post_2021_hours    Friday         3.703704  
                   Monday        10.701107  
                   Thursday       3.308824  
                   Tuesday        0.727273  
                   Wednesday      7.326007  
pre_2012_hours     Friday       100.000000  
                   Monday        19.047619  
                   Thursday       3.125000  
                   Tuesday        3.906250  
                   Wednesday      2.343750

Lowest-coverage eligible sessions:


,session_date,weekday,bar_count,expected_bar_count,bar_coverage_pct,missing_expected_bars
2510,2020-03-16,Monday,759,1365,55.604396,606
1164,2014-12-26,Friday,1122,1365,82.197802,243
2505,2020-03-09,Monday,1160,1365,84.981685,205
537,2012-07-03,Tuesday,1177,1365,86.227106,188
2240,2019-02-27,Wednesday,1181,1365,86.520147,184
916,2013-12-31,Tuesday,1189,1365,87.106227,176
991,2014-04-21,Monday,1230,1365,90.109890,135
1937,2017-12-26,Tuesday,1248,1365,91.428571,117
1680,2016-12-27,Tuesday,1249,1365,91.501832,116
1030,2014-06-17,Tuesday,1267,1365,92.820513,98


HISTORICAL COVERAGE DIAGNOSTIC: COMPLETE


## 22. Historical target-policy gate — frozen for v0.1

The historical target uses the same challenge contract already validated against
Headline Arena:

- **Bearish:** session return `< -0.30%`
- **Neutral:** `-0.30% <= return <= +0.30%`
- **Bullish:** session return `> +0.30%`

Exact `+/-0.30%` boundaries belong to Neutral.

This value is part of the v0.1 experiment contract. Do not change it after
reviewing model performance. A materially different dead zone should become a
new experiment version.


In [73]:
# ------------------------------------------------------------
# Freeze and construct the v0.1 historical target
# ------------------------------------------------------------

# ------------------------------------------------------------
# Frozen v0.1 target contract
#
# Keep this value explicit here so Section 22 remains
# reproducible even after a Jupyter kernel restart.
#
# Earlier Phase A validation uses the same +/-0.30% contract.
# ------------------------------------------------------------

FROZEN_ES_DEAD_ZONE_PCT_V0_1 = 0.30


# ------------------------------------------------------------
# Recover / verify the project-level constant
# ------------------------------------------------------------

if "ES_DEAD_ZONE_PCT_V0_1" in globals():
    if not np.isclose(
        float(ES_DEAD_ZONE_PCT_V0_1),
        FROZEN_ES_DEAD_ZONE_PCT_V0_1,
        atol=1e-12,
        rtol=0,
    ):
        raise RuntimeError(
            "Existing ES_DEAD_ZONE_PCT_V0_1 conflicts "
            "with the frozen v0.1 target contract. "
            f"Existing={ES_DEAD_ZONE_PCT_V0_1}; "
            f"expected={FROZEN_ES_DEAD_ZONE_PCT_V0_1}."
        )
else:
    ES_DEAD_ZONE_PCT_V0_1 = (
        FROZEN_ES_DEAD_ZONE_PCT_V0_1
    )


HISTORICAL_TARGET_BAND_PCT = float(
    ES_DEAD_ZONE_PCT_V0_1
)


TARGET_LABELS_V0_1 = (
    "bearish",
    "neutral",
    "bullish",
)


EXPECTED_HISTORICAL_SESSIONS_V0_1 = 4_203

EXPECTED_TARGET_ELIGIBLE_SESSIONS_V0_1 = 3_855

EXPECTED_TARGET_EXCLUDED_SESSIONS_V0_1 = 348


# ------------------------------------------------------------
# Local target helpers
#
# These are intentionally defined here so target construction
# does not depend on an earlier kernel cell having been run.
# ------------------------------------------------------------

def historical_pct_points_to_decimal(
    dead_zone_pct,
):
    return float(
        dead_zone_pct
    ) / 100.0


def classify_historical_return(
    session_return,
    dead_zone_pct=HISTORICAL_TARGET_BAND_PCT,
):
    band = (
        historical_pct_points_to_decimal(
            dead_zone_pct
        )
    )

    session_return = float(
        session_return
    )

    if session_return > band:
        return "bullish"

    if session_return < -band:
        return "bearish"

    return "neutral"


def assert_target_policy_frozen():
    if HISTORICAL_TARGET_BAND_PCT is None:
        raise RuntimeError(
            "HISTORICAL_TARGET_BAND_PCT "
            "must be frozen before target creation."
        )

    if not np.isclose(
        HISTORICAL_TARGET_BAND_PCT,
        FROZEN_ES_DEAD_ZONE_PCT_V0_1,
        atol=1e-12,
        rtol=0,
    ):
        raise RuntimeError(
            "Historical target band no longer "
            "matches the frozen v0.1 Arena contract."
        )


def add_historical_target(
    session_df,
):
    """
    Add the Arena-compatible three-class historical target.

    Rules:
    - only target_eligible sessions receive a target,
    - excluded sessions remain unlabeled,
    - exactly -0.30% and +0.30% are Neutral.
    """

    assert_target_policy_frozen()

    required_columns = {
        "session_date",
        "session_return",
        "target_eligible",
    }

    missing_columns = (
        required_columns
        - set(
            session_df.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Target construction is missing columns: "
            f"{sorted(missing_columns)}"
        )

    out = (
        session_df
        .sort_values(
            "session_date"
        )
        .copy()
    )


    out[
        "target_band_pct"
    ] = HISTORICAL_TARGET_BAND_PCT


    # --------------------------------------------------------
    # Only structurally eligible sessions receive a target.
    # --------------------------------------------------------

    out[
        "target_return"
    ] = (
        out[
            "session_return"
        ]
        .where(
            out[
                "target_eligible"
            ]
        )
    )


    target = pd.Series(
        pd.NA,
        index=out.index,
        dtype="string",
    )


    eligible_mask = (
        out[
            "target_eligible"
        ]
        &
        out[
            "target_return"
        ]
        .notna()
    )


    target.loc[
        eligible_mask
    ] = (
        out.loc[
            eligible_mask,
            "target_return",
        ]
        .astype(float)
        .map(
            classify_historical_return
        )
        .astype("string")
    )


    out[
        "target"
    ] = target


    return out


# ------------------------------------------------------------
# Exact-boundary unit checks
# ------------------------------------------------------------

band_decimal = (
    historical_pct_points_to_decimal(
        HISTORICAL_TARGET_BAND_PCT
    )
)


assert classify_historical_return(
    band_decimal
) == "neutral"


assert classify_historical_return(
    -band_decimal
) == "neutral"


assert classify_historical_return(
    band_decimal + 1e-8
) == "bullish"


assert classify_historical_return(
    -band_decimal - 1e-8
) == "bearish"


assert_target_policy_frozen()


print(
    "ES_DEAD_ZONE_PCT_V0_1:",
    ES_DEAD_ZONE_PCT_V0_1,
)

print(
    "Historical target policy frozen at:",
    f"+/-{HISTORICAL_TARGET_BAND_PCT:.2f}%",
)

print(
    "Exact +/-0.30% boundaries: NEUTRAL"
)

print(
    "TARGET POLICY UNIT CHECKS: PASS"
)

ES_DEAD_ZONE_PCT_V0_1: 0.3
Historical target policy frozen at: +/-0.30%
Exact +/-0.30% boundaries: NEUTRAL
TARGET POLICY UNIT CHECKS: PASS


### 22.1 Construct the historical three-class target

Apply the frozen Headline Arena target definition only to sessions that passed
the v0.1 historical-session eligibility policy.

Excluded sessions remain in the audit table but receive no target label.

In [74]:
# ------------------------------------------------------------
# Construct historical target
# ------------------------------------------------------------

es_sessions_targeted = (
    add_historical_target(
        es_sessions_policy
    )
)


total_sessions = len(
    es_sessions_targeted
)

eligible_sessions = int(
    es_sessions_targeted[
        "target_eligible"
    ].sum()
)

excluded_sessions = int(
    (
        ~es_sessions_targeted[
            "target_eligible"
        ]
    ).sum()
)

labeled_sessions = int(
    es_sessions_targeted[
        "target"
    ]
    .notna()
    .sum()
)


assert total_sessions == 4_203

assert eligible_sessions == 3_855

assert excluded_sessions == 348

assert labeled_sessions == eligible_sessions


assert (
    es_sessions_targeted.loc[
        ~es_sessions_targeted[
            "target_eligible"
        ],
        "target",
    ]
    .isna()
    .all()
)


observed_labels = set(
    es_sessions_targeted[
        "target"
    ]
    .dropna()
    .unique()
    .tolist()
)


assert observed_labels.issubset(
    {
        "bearish",
        "neutral",
        "bullish",
    }
)


print(
    "Historical sessions:",
    total_sessions,
)

print(
    "Target-eligible sessions:",
    eligible_sessions,
)

print(
    "Excluded sessions:",
    excluded_sessions,
)

print(
    "Labeled sessions:",
    labeled_sessions,
)

print(
    "HISTORICAL TARGET CONSTRUCTION: PASS"
)

Historical sessions: 4203
Target-eligible sessions: 3855
Excluded sessions: 348
Labeled sessions: 3855
HISTORICAL TARGET CONSTRUCTION: PASS


### 22.2 Audit historical target distribution

Inspect the Bearish / Neutral / Bullish balance before any modeling.

The target distribution is descriptive only. It must not be used to alter the
frozen +/-0.30% target definition.

In [75]:
# ------------------------------------------------------------
# Target distribution audit
# ------------------------------------------------------------

target_counts = (
    es_sessions_targeted.loc[
        es_sessions_targeted[
            "target_eligible"
        ],
        "target",
    ]
    .value_counts()
    .reindex(
        [
            "bearish",
            "neutral",
            "bullish",
        ],
        fill_value=0,
    )
)


target_distribution = pd.DataFrame({
    "sessions":
        target_counts.astype(int),

    "share_pct":
        (
            target_counts
            / target_counts.sum()
            * 100
        ),
})


display(
    target_distribution
)


assert int(
    target_counts.sum()
) == 3_855


assert (
    target_counts > 0
).all()


eligible_target_rows = (
    es_sessions_targeted.loc[
        es_sessions_targeted[
            "target_eligible"
        ]
    ]
    .copy()
)


eligible_target_rows[
    "target_return_pct"
] = (
    eligible_target_rows[
        "target_return"
    ]
    * 100
)


eligible_target_rows[
    "distance_to_boundary"
] = np.minimum(
    (
        eligible_target_rows[
            "target_return_pct"
        ]
        - 0.30
    ).abs(),

    (
        eligible_target_rows[
            "target_return_pct"
        ]
        + 0.30
    ).abs(),
)


print(
    "Sessions nearest the +/-0.30% boundaries:"
)


display(
    eligible_target_rows[
        [
            "session_date",
            "target_return_pct",
            "target",
            "distance_to_boundary",
        ]
    ]
    .sort_values(
        "distance_to_boundary"
    )
    .head(20)
)


print(
    "TARGET DISTRIBUTION AUDIT: PASS"
)

,sessions,share_pct
target,,
bearish,1101,28.560311
neutral,1307,33.904021
bullish,1447,37.535668


Sessions nearest the +/-0.30% boundaries:


,session_date,target_return_pct,target,distance_to_boundary
2732,2021-01-25,0.299831,neutral,0.000169
3001,2022-02-07,-0.300200,bearish,0.000200
2936,2021-11-05,0.299674,neutral,0.000326
744,2013-04-26,-0.300348,bearish,0.000348
3709,2024-11-01,0.300418,bullish,0.000418
2869,2021-08-04,-0.300471,bearish,0.000471
1504,2016-04-22,0.300517,bullish,0.000517
2251,2019-03-14,-0.301312,bearish,0.001312
1676,2016-12-20,0.298640,neutral,0.001360
2553,2020-05-15,0.298639,neutral,0.001361


TARGET DISTRIBUTION AUDIT: PASS


# Phase D — leakage-safe ES features

The first v0.1 feature layer should be conservative.

The helper below creates **lagged** market features only. Every predictor is shifted so today's final session result cannot leak into today's forecast row.

These features are examples of the data-engineering layer; Notebook 09 will decide what is actually used by the model.

In [76]:
# ------------------------------------------------------------
# Conservative v0.1 ES feature set
# Previous VALID completed sessions only
# ------------------------------------------------------------

ES_FEATURE_COLUMNS_V0_1 = [
    "es_return_lag1",
    "es_return_lag2",
    "es_return_lag3",
    "es_return_lag5",
    "es_range_pct_lag1",
    "es_log_volume_lag1",
    "es_return_mean_5",
    "es_return_mean_20",
    "es_vol_5",
    "es_vol_20",
    "es_abs_return_mean_5",
    "es_abs_return_mean_20",
]


def build_safe_es_features(
    session_df,
):
    """
    Build deterministic ES predictors from previous
    target-eligible completed sessions only.

    Excluded sessions never become feature sources.

    No imputation, scaling, resampling, or modeling
    occurs in Notebook 08.
    """

    out = (
        session_df
        .sort_values(
            "session_date"
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    required_columns = {
        "session_date",
        "target_eligible",
        "open",
        "high",
        "low",
        "volume",
        "session_return",
    }

    missing_columns = (
        required_columns
        - set(out.columns)
    )

    if missing_columns:
        raise ValueError(
            "Feature construction is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if (
        out["session_date"]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate session dates detected before "
            "feature construction."
        )

    source_ok = (
        out["target_eligible"]
        .fillna(False)
        .astype(bool)
    )

    raw_range_pct = (
        (
            out["high"]
            - out["low"]
        )
        / out["open"]
    )

    out[
        "_feature_source_return"
    ] = (
        out["session_return"]
        .where(source_ok)
    )

    out[
        "_feature_source_range_pct"
    ] = (
        raw_range_pct
        .where(source_ok)
    )

    out[
        "_feature_source_log_volume"
    ] = (
        np.log1p(
            out["volume"]
        )
        .where(source_ok)
    )

    # --------------------------------------------------------
    # Build the chronological sequence of VALID source sessions.
    #
    # Each row below represents information known AFTER that
    # completed valid session.
    # --------------------------------------------------------

    eligible_feature_rows = (
        out.loc[
            source_ok,
            [
                "session_date",
                "_feature_source_return",
                "_feature_source_range_pct",
                "_feature_source_log_volume",
            ],
        ]
        .copy()
        .sort_values(
            "session_date"
        )
        .reset_index(
            drop=True
        )
    )

    source_return = (
        eligible_feature_rows[
            "_feature_source_return"
        ]
        .astype(float)
    )

    # --------------------------------------------------------
    # Feature state after each completed valid source session.
    #
    # When this state is later mapped to a future session:
    #
    # current source row       -> lag1
    # previous valid source    -> lag2
    # two valid sources back   -> lag3
    # four valid sources back  -> lag5
    # --------------------------------------------------------

    eligible_feature_rows[
        "es_return_lag1"
    ] = source_return

    eligible_feature_rows[
        "es_return_lag2"
    ] = source_return.shift(1)

    eligible_feature_rows[
        "es_return_lag3"
    ] = source_return.shift(2)

    eligible_feature_rows[
        "es_return_lag5"
    ] = source_return.shift(4)

    eligible_feature_rows[
        "es_range_pct_lag1"
    ] = (
        eligible_feature_rows[
            "_feature_source_range_pct"
        ]
        .astype(float)
    )

    eligible_feature_rows[
        "es_log_volume_lag1"
    ] = (
        eligible_feature_rows[
            "_feature_source_log_volume"
        ]
        .astype(float)
    )

    eligible_feature_rows[
        "es_return_mean_5"
    ] = (
        source_return
        .rolling(
            5,
            min_periods=5,
        )
        .mean()
    )

    eligible_feature_rows[
        "es_return_mean_20"
    ] = (
        source_return
        .rolling(
            20,
            min_periods=20,
        )
        .mean()
    )

    eligible_feature_rows[
        "es_vol_5"
    ] = (
        source_return
        .rolling(
            5,
            min_periods=5,
        )
        .std()
    )

    eligible_feature_rows[
        "es_vol_20"
    ] = (
        source_return
        .rolling(
            20,
            min_periods=20,
        )
        .std()
    )

    eligible_feature_rows[
        "es_abs_return_mean_5"
    ] = (
        source_return
        .abs()
        .rolling(
            5,
            min_periods=5,
        )
        .mean()
    )

    eligible_feature_rows[
        "es_abs_return_mean_20"
    ] = (
        source_return
        .abs()
        .rolling(
            20,
            min_periods=20,
        )
        .mean()
    )

    # --------------------------------------------------------
    # Map the latest PRIOR valid completed-session state onto
    # every chronological session row.
    #
    # allow_exact_matches=False is essential:
    # a row may not use its own same-session result.
    # --------------------------------------------------------

    feature_lookup = (
        eligible_feature_rows[
            ["session_date"]
            + ES_FEATURE_COLUMNS_V0_1
        ]
        .rename(
            columns={
                "session_date":
                    "_feature_asof_session_date"
            }
        )
        .sort_values(
            "_feature_asof_session_date"
        )
    )

    out = pd.merge_asof(
        out.sort_values(
            "session_date"
        ),
        feature_lookup,
        left_on="session_date",
        right_on="_feature_asof_session_date",
        direction="backward",
        allow_exact_matches=False,
    )

    out = (
        out
        .drop(
            columns=[
                "_feature_asof_session_date"
            ]
        )
        .sort_values(
            "session_date"
        )
        .reset_index(
            drop=True
        )
    )

    return out


print(
    "Conservative v0.1 ES feature builder ready."
)

Conservative v0.1 ES feature builder ready.


## 23. Feature leakage audit

Every candidate feature must answer:

> Could this exact value have been known at or before the frozen forecast cutoff on that historical date?

If historical availability cannot be reconstructed reliably, the feature stays out of v0.1.

### 23.1 Build the conservative ES feature frame

Construct candidate predictors over the complete chronological session sequence.

The feature frame is built before filtering to modeling rows so that the
historical spacing of sessions is preserved.

Only information from completed prior sessions is allowed to enter a predictor.

No imputation, scaling, resampling, class weighting, or model fitting occurs
in Notebook 08.

In [77]:
# ------------------------------------------------------------
# Build deterministic ES feature frame
# ------------------------------------------------------------

es_feature_frame = (
    build_safe_es_features(
        es_sessions_targeted
    )
)


# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

assert (
    len(es_feature_frame)
    == len(es_sessions_targeted)
), (
    "Feature engineering changed the historical "
    "session-row count."
)


assert (
    es_feature_frame[
        "session_date"
    ]
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate session dates appeared during "
    "feature construction."
)


assert (
    es_feature_frame[
        "session_date"
    ]
    .is_monotonic_increasing
), (
    "Feature frame is not in chronological order."
)


# ------------------------------------------------------------
# Feature missingness diagnostic
#
# Missing values are NOT automatically errors.
#
# They are expected at the beginning of history because:
#
# - lag1 needs one earlier session
# - lag5 needs five earlier sessions
# - rolling 20 needs sufficient prior history
#
# Additional NaNs may occur because structurally invalid
# sessions are deliberately prevented from becoming feature
# sources.
# ------------------------------------------------------------

feature_missingness = pd.DataFrame({
    "missing_rows":
        es_feature_frame[
            ES_FEATURE_COLUMNS_V0_1
        ]
        .isna()
        .sum(),

    "non_missing_rows":
        es_feature_frame[
            ES_FEATURE_COLUMNS_V0_1
        ]
        .notna()
        .sum(),

    "missing_pct":
        (
            es_feature_frame[
                ES_FEATURE_COLUMNS_V0_1
            ]
            .isna()
            .mean()
            * 100.0
        ),
})


print(
    "Feature-frame rows:",
    len(es_feature_frame),
)

print(
    "Candidate ES feature columns:",
    len(
        ES_FEATURE_COLUMNS_V0_1
    ),
)

print(
    "Duplicate session dates:",
    int(
        es_feature_frame[
            "session_date"
        ]
        .duplicated()
        .sum()
    ),
)

print(
    "Chronological order:",
    bool(
        es_feature_frame[
            "session_date"
        ]
        .is_monotonic_increasing
    ),
)


display(
    feature_missingness
)


print(
    "ES FEATURE FRAME: BUILT"
)

Feature-frame rows: 4203
Candidate ES feature columns: 12
Duplicate session dates: 0
Chronological order: True


,missing_rows,non_missing_rows,missing_pct
es_return_lag1,2,4201,0.047585
es_return_lag2,3,4200,0.071378
es_return_lag3,4,4199,0.095170
es_return_lag5,8,4195,0.190340
es_range_pct_lag1,2,4201,0.047585
es_log_volume_lag1,2,4201,0.047585
es_return_mean_5,8,4195,0.190340
es_return_mean_20,28,4175,0.666191
es_vol_5,8,4195,0.190340
es_vol_20,28,4175,0.666191


ES FEATURE FRAME: BUILT


### 23.2 Mechanical feature-leakage audit

Verify programmatically that candidate predictors use prior-session information
only.

Same-session target components such as the current session open, high, low,
close, volume, return, and target label are forbidden as model predictors.

The lag identities are checked directly against their source series.

In [78]:
# ------------------------------------------------------------
# Mechanical feature-leakage audit
# Previous VALID session semantics
# ------------------------------------------------------------

FORBIDDEN_SAME_SESSION_MODEL_COLUMNS = {
    "open",
    "high",
    "low",
    "close",
    "volume",
    "session_return",
    "target_return",
    "target",
}


forbidden_overlap = (
    set(
        ES_FEATURE_COLUMNS_V0_1
    )
    &
    FORBIDDEN_SAME_SESSION_MODEL_COLUMNS
)


assert not forbidden_overlap, (
    "Same-session leakage columns entered "
    "the feature set: "
    f"{sorted(forbidden_overlap)}"
)


# ------------------------------------------------------------
# Reconstruct eligible feature-source sequence
# ------------------------------------------------------------

eligible_feature_rows = (
    es_feature_frame.loc[
        es_feature_frame[
            "target_eligible"
        ].astype(bool),
        [
            "session_date",
            "_feature_source_return",
            "_feature_source_range_pct",
            "_feature_source_log_volume",
        ],
    ]
    .copy()
    .sort_values(
        "session_date"
    )
    .reset_index(
        drop=True
    )
)


assert (
    eligible_feature_rows[
        "_feature_source_return"
    ]
    .notna()
    .all()
), (
    "An eligible feature-source row has "
    "a missing return."
)


source_return = (
    eligible_feature_rows[
        "_feature_source_return"
    ]
    .astype(float)
)


# ------------------------------------------------------------
# Independently reconstruct the feature state available
# immediately AFTER each valid completed source session.
# ------------------------------------------------------------

expected_lookup = pd.DataFrame({
    "_expected_source_session_date":
        eligible_feature_rows[
            "session_date"
        ],

    "es_return_lag1":
        source_return,

    "es_return_lag2":
        source_return.shift(1),

    "es_return_lag3":
        source_return.shift(2),

    "es_return_lag5":
        source_return.shift(4),

    "es_range_pct_lag1":
        eligible_feature_rows[
            "_feature_source_range_pct"
        ].astype(float),

    "es_log_volume_lag1":
        eligible_feature_rows[
            "_feature_source_log_volume"
        ].astype(float),

    "es_return_mean_5":
        source_return
        .rolling(
            5,
            min_periods=5,
        )
        .mean(),

    "es_return_mean_20":
        source_return
        .rolling(
            20,
            min_periods=20,
        )
        .mean(),

    "es_vol_5":
        source_return
        .rolling(
            5,
            min_periods=5,
        )
        .std(),

    "es_vol_20":
        source_return
        .rolling(
            20,
            min_periods=20,
        )
        .std(),

    "es_abs_return_mean_5":
        source_return
        .abs()
        .rolling(
            5,
            min_periods=5,
        )
        .mean(),

    "es_abs_return_mean_20":
        source_return
        .abs()
        .rolling(
            20,
            min_periods=20,
        )
        .mean(),
})


# ------------------------------------------------------------
# Map only STRICTLY PRIOR valid-session state onto each row.
#
# This is the key no-leakage condition:
# allow_exact_matches=False prevents a session from using
# its own completed return/range/volume.
# ------------------------------------------------------------

expected_features = pd.merge_asof(
    es_feature_frame[
        ["session_date"]
    ]
    .sort_values(
        "session_date"
    ),
    expected_lookup.sort_values(
        "_expected_source_session_date"
    ),
    left_on="session_date",
    right_on="_expected_source_session_date",
    direction="backward",
    allow_exact_matches=False,
)


# ------------------------------------------------------------
# Exact feature identity checks
# ------------------------------------------------------------

for feature_name in ES_FEATURE_COLUMNS_V0_1:
    pd.testing.assert_series_equal(
        es_feature_frame[
            feature_name
        ]
        .reset_index(
            drop=True
        ),
        expected_features[
            feature_name
        ]
        .reset_index(
            drop=True
        ),
        check_names=False,
    )


# ------------------------------------------------------------
# Explicit same-session source check
# ------------------------------------------------------------

same_session_source_used = (
    expected_features[
        "_expected_source_session_date"
    ]
    == expected_features[
        "session_date"
    ]
)


assert not same_session_source_used.any(), (
    "At least one row used its own same-session "
    "completed data as a predictor source."
)


print(
    "Forbidden feature overlap:",
    sorted(
        forbidden_overlap
    ),
)

print(
    "Eligible feature-source sessions:",
    len(
        eligible_feature_rows
    ),
)

print(
    "Feature identities checked:",
    len(
        ES_FEATURE_COLUMNS_V0_1
    ),
)

print(
    "Same-session feature sources used:",
    int(
        same_session_source_used.sum()
    ),
)

print(
    "FEATURE LEAKAGE AUDIT: PASS"
)

Forbidden feature overlap: []
Eligible feature-source sessions: 3855
Feature identities checked: 12
Same-session feature sources used: 0
FEATURE LEAKAGE AUDIT: PASS


### 23.3 Freeze the v0.1 feature scope

The first exported v0.1 modeling table is deliberately conservative.

It contains only lagged ES-derived predictors whose historical availability is
unambiguous.

The following feature families are intentionally excluded from the v0.1 core
table:

- same-day pre-10:00 ET ES features,
- VIX / volatility-market features,
- macroeconomic-release features,
- Google / sentiment features,
- timestamped event-risk features.

Those features require separate historical timestamp-availability audits before
they can be added safely.

No feature is included merely because it improves model performance.

In [79]:
# ------------------------------------------------------------
# Freeze v0.1 feature-scope decision
# ------------------------------------------------------------

FEATURE_SET_NAME_V0_1 = (
    "es_only_conservative_v0_1"
)


EXTERNAL_FEATURES_INCLUDED_V0_1 = False


SAME_DAY_PREDEADLINE_FEATURES_INCLUDED_V0_1 = False


assert (
    EXTERNAL_FEATURES_INCLUDED_V0_1
    is False
)


assert (
    SAME_DAY_PREDEADLINE_FEATURES_INCLUDED_V0_1
    is False
)


assert len(
    ES_FEATURE_COLUMNS_V0_1
) == 12


print(
    "Feature set:",
    FEATURE_SET_NAME_V0_1,
)

print(
    "Feature count:",
    len(
        ES_FEATURE_COLUMNS_V0_1
    ),
)

print(
    "External features included:",
    EXTERNAL_FEATURES_INCLUDED_V0_1,
)

print(
    "Same-day pre-deadline features included:",
    SAME_DAY_PREDEADLINE_FEATURES_INCLUDED_V0_1,
)

print(
    "V0.1 FEATURE SCOPE: FROZEN"
)

Feature set: es_only_conservative_v0_1
Feature count: 12
External features included: False
Same-day pre-deadline features included: False
V0.1 FEATURE SCOPE: FROZEN


## 24. Build the final v0.1 modeling table

Create the frozen modeling artifact passed from Notebook 08 to Notebook 09.

Only sessions that passed the historical target-eligibility policy are retained.

The exported table contains:

- session date,
- numerical target return,
- Bearish / Neutral / Bullish target label,
- frozen target-band metadata,
- the explicitly approved leakage-safe ES feature set.

Missing predictor values remain missing.

No imputation, scaling, resampling, class weighting, or model fitting occurs in
Notebook 08.

In [80]:
# ------------------------------------------------------------
# Build final v0.1 modeling table
# ------------------------------------------------------------

FINAL_MODELING_BASE_COLUMNS = [
    "session_date",
    "target_return",
    "target",
    "target_band_pct",
]


FINAL_MODELING_COLUMNS = (
    FINAL_MODELING_BASE_COLUMNS
    + ES_FEATURE_COLUMNS_V0_1
)


# ------------------------------------------------------------
# Keep only structurally target-eligible sessions.
#
# Feature NaNs remain untouched.
# ------------------------------------------------------------

final_modeling_df = (
    es_feature_frame.loc[
        es_feature_frame[
            "target_eligible"
        ],
        FINAL_MODELING_COLUMNS,
    ]
    .copy()
    .sort_values(
        "session_date"
    )
    .reset_index(
        drop=True
    )
)


final_modeling_df[
    "target"
] = (
    final_modeling_df[
        "target"
    ]
    .astype("string")
)


# ------------------------------------------------------------
# Structural gates
# ------------------------------------------------------------

assert (
    len(final_modeling_df)
    == 3_855
), (
    "Unexpected final modeling row count: "
    f"{len(final_modeling_df)}"
)


assert (
    final_modeling_df[
        "session_date"
    ]
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate modeling dates detected."
)


assert (
    final_modeling_df[
        "session_date"
    ]
    .is_monotonic_increasing
), (
    "Final modeling table is not chronological."
)


assert (
    final_modeling_df[
        "target"
    ]
    .isna()
    .sum()
    == 0
), (
    "Missing target labels detected."
)


assert (
    final_modeling_df[
        "target_return"
    ]
    .isna()
    .sum()
    == 0
), (
    "Missing target returns detected."
)


observed_targets = set(
    final_modeling_df[
        "target"
    ]
    .unique()
    .tolist()
)


assert observed_targets.issubset(
    {
        "bearish",
        "neutral",
        "bullish",
    }
), (
    "Unexpected target labels: "
    f"{observed_targets}"
)


print(
    "Final modeling rows:",
    len(final_modeling_df),
)

print(
    "Final modeling columns:",
    len(
        final_modeling_df.columns
    ),
)

print(
    "Feature columns:",
    len(
        ES_FEATURE_COLUMNS_V0_1
    ),
)

print(
    "First modeling date:",
    final_modeling_df[
        "session_date"
    ].min(),
)

print(
    "Last modeling date:",
    final_modeling_df[
        "session_date"
    ].max(),
)


display(
    final_modeling_df.head()
)


display(
    final_modeling_df.tail()
)


print(
    "FINAL MODELING TABLE: BUILT"
)

Final modeling rows: 3855
Final modeling columns: 16
Feature columns: 12
First modeling date: 2010-06-08 00:00:00
Last modeling date: 2026-09-30 00:00:00


,session_date,target_return,target,target_band_pct,es_return_lag1,es_return_lag2,es_return_lag3,es_return_lag5,es_range_pct_lag1,es_log_volume_lag1,es_return_mean_5,es_return_mean_20,es_vol_5,es_vol_20,es_abs_return_mean_5,es_abs_return_mean_20
0,2010-06-08,0.010724,bullish,0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-06-09,-0.005658,bearish,0.3,0.010724,NaN,NaN,NaN,0.020972,15.011626,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-06-10,0.030879,bullish,0.3,-0.005658,0.010724,NaN,NaN,0.024988,14.803375,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-06-15,0.020948,bullish,0.3,0.030879,-0.005658,0.010724,NaN,0.033729,14.406641,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-06-16,0.000451,neutral,0.3,0.020948,0.030879,-0.005658,NaN,0.025092,14.617635,NaN,NaN,NaN,NaN,NaN,NaN


,session_date,target_return,target,target_band_pct,es_return_lag1,es_return_lag2,es_return_lag3,es_return_lag5,es_range_pct_lag1,es_log_volume_lag1,es_return_mean_5,es_return_mean_20,es_vol_5,es_vol_20,es_abs_return_mean_5,es_abs_return_mean_20
3850,2026-09-24,-0.001961,neutral,0.3,-0.007312,-0.000287,0.013889,0.010264,0.010792,14.212384,0.003869,0.000750,0.008437,0.006110,0.006908,0.004933
3851,2026-09-25,0.006025,bullish,0.3,-0.001961,-0.007312,-0.000287,0.002791,0.009807,14.338040,0.001424,0.000555,0.007873,0.006133,0.005248,0.004934
3852,2026-09-28,-0.006381,bearish,0.3,0.006025,-0.001961,-0.007312,0.013889,0.008538,14.264297,0.002071,0.000736,0.008142,0.006243,0.005895,0.005115
3853,2026-09-29,-0.001065,neutral,0.3,-0.006381,0.006025,-0.001961,-0.000287,0.009877,14.340808,-0.001983,0.000346,0.005356,0.006438,0.004393,0.005363
3854,2026-09-30,-0.002261,neutral,0.3,-0.001065,-0.006381,0.006025,-0.007312,0.007552,14.243735,-0.002139,0.000351,0.005306,0.006437,0.004549,0.005358


FINAL MODELING TABLE: BUILT


### 24.1 Final modeling-table quality audit

Inspect missingness and numerical validity before serialization.

Missing feature values are allowed because any imputation must be fit only on
the training portion of each walk-forward fold in Notebook 09.

The following are not allowed:

- duplicate session dates,
- missing target labels,
- missing target returns,
- infinite predictor values,
- completely empty predictor columns.

In [81]:
# ------------------------------------------------------------
# Final modeling-table quality audit
# ------------------------------------------------------------

feature_block = (
    final_modeling_df[
        ES_FEATURE_COLUMNS_V0_1
    ]
)


# ------------------------------------------------------------
# 1. Missingness report
#
# NaNs are allowed in Notebook 08.
# They must NOT be globally imputed here.
# ------------------------------------------------------------

feature_missing_report = pd.DataFrame({
    "missing_rows":
        feature_block
        .isna()
        .sum(),

    "non_missing_rows":
        feature_block
        .notna()
        .sum(),

    "missing_pct":
        (
            feature_block
            .isna()
            .mean()
            * 100.0
        ),
})


display(
    feature_missing_report
)


# ------------------------------------------------------------
# 2. No feature may be entirely missing
# ------------------------------------------------------------

assert (
    feature_missing_report[
        "non_missing_rows"
    ] > 0
).all(), (
    "At least one predictor column is "
    "completely empty."
)


# ------------------------------------------------------------
# 3. Convert feature block explicitly to numerical ndarray
#
# Missing values become np.nan.
# np.nan is allowed.
# +/-inf is NOT allowed.
# ------------------------------------------------------------

feature_numeric = (
    feature_block
    .apply(
        pd.to_numeric,
        errors="raise",
    )
)


feature_array = (
    feature_numeric
    .to_numpy(
        dtype=float,
        na_value=np.nan,
    )
)


# ------------------------------------------------------------
# 4. Explicit infinity check
#
# Unlike np.isfinite(...).all(), this does NOT incorrectly
# reject our deliberately retained NaN feature values.
# ------------------------------------------------------------

positive_inf_count = int(
    np.isposinf(
        feature_array
    ).sum()
)


negative_inf_count = int(
    np.isneginf(
        feature_array
    ).sum()
)


total_inf_count = (
    positive_inf_count
    + negative_inf_count
)


assert total_inf_count == 0, (
    "Infinite predictor values detected. "
    f"+inf={positive_inf_count}, "
    f"-inf={negative_inf_count}."
)


# ------------------------------------------------------------
# 5. Verify that every NON-MISSING feature value is finite
# ------------------------------------------------------------

non_missing_mask = ~np.isnan(
    feature_array
)


non_missing_values = (
    feature_array[
        non_missing_mask
    ]
)


assert np.isfinite(
    non_missing_values
).all(), (
    "A non-missing predictor value is not finite."
)


# ------------------------------------------------------------
# 6. Target return may NOT contain NaN or infinity
# ------------------------------------------------------------

target_return_array = (
    pd.to_numeric(
        final_modeling_df[
            "target_return"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


assert not np.isnan(
    target_return_array
).any(), (
    "Missing target returns detected."
)


assert np.isfinite(
    target_return_array
).all(), (
    "Non-finite target returns detected."
)


# ------------------------------------------------------------
# 7. Structural quality report
# ------------------------------------------------------------

quality_report = pd.Series({
    "rows":
        len(
            final_modeling_df
        ),

    "columns":
        len(
            final_modeling_df.columns
        ),

    "feature_count":
        len(
            ES_FEATURE_COLUMNS_V0_1
        ),

    "duplicate_session_dates":
        int(
            final_modeling_df[
                "session_date"
            ]
            .duplicated()
            .sum()
        ),

    "missing_target":
        int(
            final_modeling_df[
                "target"
            ]
            .isna()
            .sum()
        ),

    "missing_target_return":
        int(
            final_modeling_df[
                "target_return"
            ]
            .isna()
            .sum()
        ),

    "rows_with_any_missing_feature":
        int(
            feature_block
            .isna()
            .any(
                axis=1
            )
            .sum()
        ),

    "total_missing_feature_cells":
        int(
            feature_block
            .isna()
            .sum()
            .sum()
        ),

    "positive_inf_cells":
        positive_inf_count,

    "negative_inf_cells":
        negative_inf_count,

    "total_inf_cells":
        total_inf_count,
})


display(
    quality_report
)


print(
    "FINAL MODELING QUALITY AUDIT: PASS"
)

,missing_rows,non_missing_rows,missing_pct
es_return_lag1,1,3854,0.025940
es_return_lag2,2,3853,0.051881
es_return_lag3,3,3852,0.077821
es_return_lag5,5,3850,0.129702
es_range_pct_lag1,1,3854,0.025940
es_log_volume_lag1,1,3854,0.025940
es_return_mean_5,5,3850,0.129702
es_return_mean_20,20,3835,0.518807
es_vol_5,5,3850,0.129702
es_vol_20,20,3835,0.518807


rows                             3855
columns                            16
feature_count                      12
duplicate_session_dates             0
missing_target                      0
missing_target_return               0
rows_with_any_missing_feature      20
total_missing_feature_cells        88
positive_inf_cells                  0
negative_inf_cells                  0
total_inf_cells                     0
dtype: int64

FINAL MODELING QUALITY AUDIT: PASS


## 25. Final output paths

Planned derived artifacts:

```text
data/processed/es_arena_alignment_v0_1.csv
data/processed/es_daily_modeling_v0_1.parquet
data/processed/es_daily_modeling_v0_1_metadata.json
data/processed/es_raw_manifest_v0_1.csv
```

The raw licensed Databento files stay local and untracked.

In [82]:
ALIGNMENT_CSV = PROCESSED_DIR / "es_arena_alignment_v0_1.csv"
MODEL_TABLE_PATH = PROCESSED_DIR / "es_daily_modeling_v0_1.parquet"
METADATA_PATH = PROCESSED_DIR / "es_daily_modeling_v0_1_metadata.json"
RAW_MANIFEST_PATH = PROCESSED_DIR / "es_raw_manifest_v0_1.csv"

print("Alignment:", ALIGNMENT_CSV)
print("Model table:", MODEL_TABLE_PATH)
print("Metadata:", METADATA_PATH)
print("Raw manifest:", RAW_MANIFEST_PATH)

Alignment: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_alignment_v0_1.csv
Model table: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_daily_modeling_v0_1.parquet
Metadata: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_daily_modeling_v0_1_metadata.json
Raw manifest: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_raw_manifest_v0_1.csv


## 26. Metadata artifact

The metadata file is part of the scientific record. It should make the exact data contract and experiment state recoverable later.

In [83]:

# ------------------------------------------------------------
# Build final scientific metadata artifact
# ------------------------------------------------------------

FORECAST_DEADLINE_ET_V0_1 = "10:00:00"


def build_metadata(
    model_df,
    feature_columns,
    raw_manifest,
):
    class_counts = (
        model_df[
            "target"
        ]
        .value_counts()
        .reindex(
            [
                "bearish",
                "neutral",
                "bullish",
            ],
            fill_value=0,
        )
    )

    feature_missing = (
        model_df[
            feature_columns
        ]
        .isna()
        .sum()
    )

    metadata = {
        "experiment":
            "ES Daily Forward Experiment",

        "experiment_version":
            "v0.1",

        "stable_project_version":
            "v1.2.0",

        "feature_set_name":
            FEATURE_SET_NAME_V0_1,

        "provider_contract": {
            "provider":
                "Databento",

            "dataset":
                DATASET,

            "symbol":
                SYMBOL,

            "stype_in":
                STYPE_IN,

            "primary_schema":
                PRIMARY_SCHEMA,

            "diagnostic_schema":
                DIAGNOSTIC_SCHEMA,

            "databento_accepted":
                bool(
                    DATABENTO_ACCEPTED
                ),
        },

        "history_contract": {
            "history_start_utc":
                HISTORY_START_UTC,

            "history_end_utc":
                HISTORY_END_UTC,

            "analysis_timezone":
                ANALYSIS_TIMEZONE,

            "expected_full_history_records":
                int(
                    EXPECTED_FULL_HISTORY_RECORDS
                ),
        },

        "session_contract": {
            "session_start_et":
                ARENA_SESSION_START_TIME,

            "forecast_deadline_et":
                FORECAST_DEADLINE_ET_V0_1,

            "session_end_et":
                ARENA_SESSION_END_TIME,

            "historical_target_type":
                "synthetic_arena_style_es_session",

            "expected_historical_sessions":
                int(
                    EXPECTED_HISTORICAL_SESSIONS_V0_1
                ),

            "target_eligible_sessions":
                int(
                    EXPECTED_TARGET_ELIGIBLE_SESSIONS_V0_1
                ),

            "target_excluded_sessions":
                int(
                    EXPECTED_TARGET_EXCLUDED_SESSIONS_V0_1
                ),

            "feature_source_semantics":
                (
                    "previous target-eligible "
                    "completed sessions only"
                ),

            "feature_availability_rule":
                (
                    "previous target-eligible completed "
                    "sessions only; no current-session "
                    "information"
                ),

            "excluded_sessions_can_be_feature_sources":
                False,
        },

        "target_contract": {
            "target_band_pct":
                float(
                    HISTORICAL_TARGET_BAND_PCT
                ),

            "labels":
                list(
                    TARGET_LABELS_V0_1
                ),

            "boundary_rule":
                (
                    "exact +/-0.30% boundaries "
                    "are neutral"
                ),
        },

        "feature_contract": {
            "feature_count":
                int(
                    len(
                        feature_columns
                    )
                ),

            "feature_columns":
                list(
                    feature_columns
                ),

            "missing_rows_by_feature": {
                str(name):
                    int(value)
                for name, value
                in feature_missing.items()
            },

            "external_features_included":
                bool(
                    EXTERNAL_FEATURES_INCLUDED_V0_1
                ),

            "same_day_predeadline_features_included":
                bool(
                    SAME_DAY_PREDEADLINE_FEATURES_INCLUDED_V0_1
                ),

            "imputation_performed_in_notebook_08":
                False,

            "scaling_performed_in_notebook_08":
                False,

            "resampling_performed_in_notebook_08":
                False,

            "model_fitting_performed_in_notebook_08":
                False,
        },

        "modeling_table": {
            "rows":
                int(
                    len(
                        model_df
                    )
                ),

            "columns":
                int(
                    len(
                        model_df.columns
                    )
                ),

            "first_session_date":
                str(
                    model_df[
                        "session_date"
                    ].min()
                ),

            "last_session_date":
                str(
                    model_df[
                        "session_date"
                    ].max()
                ),

            "class_counts": {
                str(name):
                    int(value)
                for name, value
                in class_counts.items()
            },
        },

        "raw_manifest": {
            "rows":
                int(
                    len(
                        raw_manifest
                    )
                ),

            "total_size_bytes":
                int(
                    raw_manifest[
                        "size_bytes"
                    ].sum()
                ),
        },

        "safety_state": {
            "authorize_full_history_purchase":
                bool(
                    AUTHORIZE_FULL_HISTORY_PURCHASE
                ),

            "submit_full_history_now":
                bool(
                    SUBMIT_FULL_HISTORY_NOW
                ),

            "download_completed_batch_now":
                bool(
                    DOWNLOAD_COMPLETED_BATCH_NOW
                ),

            "build_full_history_manifest_now":
                bool(
                    BUILD_FULL_HISTORY_MANIFEST_NOW
                ),

            "enable_full_history_download":
                bool(
                    ENABLE_FULL_HISTORY_DOWNLOAD
                ),
        },

        "created_at_utc":
            datetime.now(
                tz=UTC
            ).isoformat(),

        "notes": (
            "Notebook 08 is a frozen data-engineering "
            "and scientific-artifact notebook. "
            "No predictive model is trained here."
        ),
    }

    return metadata


print(
    "Final metadata builder ready."
)


Final metadata builder ready.


## 27. Final export gate

The final modeling table should not be exported until:

- Databento is accepted,
- the historical cutoff is frozen,
- the historical target band is frozen,
- provider alignment artifact exists,
- leakage review is complete,
- the selected feature set is explicitly documented.

In [84]:
# ------------------------------------------------------------
# Final export gate
# ------------------------------------------------------------

def validate_export_gate(
    model_df,
):
    if not DATABENTO_ACCEPTED:
        raise RuntimeError(
            "Databento provider acceptance is False."
        )

    validate_history_window()
    assert_target_policy_frozen()

    if not ALIGNMENT_CSV.exists():
        raise RuntimeError(
            "Arena ↔ Databento alignment artifact "
            "is missing."
        )

    if not RAW_MANIFEST_PATH.exists():
        raise RuntimeError(
            "Full-history raw SHA-256 manifest "
            "is missing."
        )

    required_columns = set(
        FINAL_MODELING_COLUMNS
    )

    missing_columns = (
        required_columns
        - set(
            model_df.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Modeling table is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if len(model_df) != 3_855:
        raise ValueError(
            "Unexpected modeling row count: "
            f"{len(model_df)}"
        )

    if (
        model_df[
            "session_date"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate session_date values detected."
        )

    if not (
        model_df[
            "session_date"
        ]
        .is_monotonic_increasing
    ):
        raise ValueError(
            "Modeling table is not chronological."
        )

    if (
        model_df[
            "target"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Missing target values detected."
        )

    if (
        model_df[
            "target_return"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Missing target_return values detected."
        )

    feature_array = (
        model_df[
            ES_FEATURE_COLUMNS_V0_1
        ]
        .apply(
            pd.to_numeric,
            errors="raise",
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan,
        )
    )

    if np.isinf(
        feature_array
    ).any():
        raise ValueError(
            "Infinite predictor values detected."
        )

    if (
        model_df[
            ES_FEATURE_COLUMNS_V0_1
        ]
        .notna()
        .sum()
        .eq(0)
        .any()
    ):
        raise ValueError(
            "At least one predictor column is "
            "completely empty."
        )

    unsafe_controls = {
        "AUTHORIZE_FULL_HISTORY_PURCHASE":
            AUTHORIZE_FULL_HISTORY_PURCHASE,
        "SUBMIT_FULL_HISTORY_NOW":
            SUBMIT_FULL_HISTORY_NOW,
        "DOWNLOAD_COMPLETED_BATCH_NOW":
            DOWNLOAD_COMPLETED_BATCH_NOW,
        "BUILD_FULL_HISTORY_MANIFEST_NOW":
            BUILD_FULL_HISTORY_MANIFEST_NOW,
        "ENABLE_FULL_HISTORY_DOWNLOAD":
            ENABLE_FULL_HISTORY_DOWNLOAD,
    }

    enabled_controls = [
        name
        for name, value
        in unsafe_controls.items()
        if bool(value)
    ]

    if enabled_controls:
        raise RuntimeError(
            "Final export requires all paid/download "
            "controls to be OFF. Enabled: "
            f"{enabled_controls}"
        )

    print(
        "FINAL EXPORT GATE: PASS"
    )


def export_modeling_table(
    model_df,
):
    validate_export_gate(
        model_df
    )

    model_df.to_parquet(
        MODEL_TABLE_PATH,
        index=False,
    )

    print(
        "Saved model table:",
        MODEL_TABLE_PATH,
    )

    return MODEL_TABLE_PATH

### 27.1 Export and verify the v0.1 modeling table

Write the final derived parquet and reload it immediately to verify the
serialized artifact.

In [85]:
model_table_path = (
    export_modeling_table(
        final_modeling_df
    )
)


roundtrip_model_df = (
    pd.read_parquet(
        model_table_path
    )
)


assert len(
    roundtrip_model_df
) == 3_855


assert list(
    roundtrip_model_df.columns
) == list(
    final_modeling_df.columns
)


assert (
    roundtrip_model_df[
        "session_date"
    ]
    .duplicated()
    .sum()
    == 0
)


assert (
    roundtrip_model_df[
        "target"
    ]
    .isna()
    .sum()
    == 0
)


# ------------------------------------------------------------
# Verify that every serialized value matches the source table
# ------------------------------------------------------------

pd.testing.assert_frame_equal(
    roundtrip_model_df.reset_index(
        drop=True
    ),
    final_modeling_df.reset_index(
        drop=True
    ),
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print(
    "FULL PARQUET VALUE COMPARISON: PASS"
)


print(
    "MODEL TABLE ROUNDTRIP VERIFICATION: PASS"
)

FINAL EXPORT GATE: PASS
Saved model table: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_daily_modeling_v0_1.parquet
FULL PARQUET VALUE COMPARISON: PASS
MODEL TABLE ROUNDTRIP VERIFICATION: PASS


### 27.2 Save the metadata artifact

Persist the exact experiment, target, session, and feature contract used to
create the v0.1 modeling table.

In [86]:
# ------------------------------------------------------------
# Save final scientific metadata
# ------------------------------------------------------------

raw_manifest_for_metadata = pd.read_csv(
    RAW_MANIFEST_PATH
)

metadata = build_metadata(
    model_df=final_modeling_df,
    feature_columns=ES_FEATURE_COLUMNS_V0_1,
    raw_manifest=raw_manifest_for_metadata,
)

metadata["model_table_sha256"] = sha256_file(
    MODEL_TABLE_PATH
)

metadata["alignment_sha256"] = sha256_file(
    ALIGNMENT_CSV
)

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

assert METADATA_PATH.exists()

print("Saved metadata:", METADATA_PATH)
print("METADATA ARTIFACT: SAVED")

Saved metadata: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_daily_modeling_v0_1_metadata.json
METADATA ARTIFACT: SAVED


### 27.3 Notebook 08 final completion gate

Notebook 08 is complete only when all frozen derived artifacts exist,
the saved metadata matches the exported modeling table, and all paid
download controls remain disabled.

In [87]:
# ------------------------------------------------------------
# Notebook 08 final completion gate
# ------------------------------------------------------------

required_artifacts = {
    "alignment": ALIGNMENT_CSV,
    "raw_manifest": RAW_MANIFEST_PATH,
    "model_table": MODEL_TABLE_PATH,
    "metadata": METADATA_PATH,
}

missing_artifacts = {
    name: path
    for name, path in required_artifacts.items()
    if not Path(path).exists()
}

assert not missing_artifacts, (
    f"Missing final artifacts: {missing_artifacts}"
)

assert RAW_MANIFEST_VERIFIED is True, (
    "Frozen raw-file SHA-256 verification "
    "was not completed successfully."
)

assert len(final_modeling_df) == 3_855
assert DATABENTO_ACCEPTED is True

assert AUTHORIZE_FULL_HISTORY_PURCHASE is False
assert SUBMIT_FULL_HISTORY_NOW is False
assert DOWNLOAD_COMPLETED_BATCH_NOW is False
assert BUILD_FULL_HISTORY_MANIFEST_NOW is False
assert ENABLE_FULL_HISTORY_DOWNLOAD is False
assert ENABLE_VALIDATION_DOWNLOAD is False

# Verify persisted metadata against actual artifacts.
saved_metadata = json.loads(
    METADATA_PATH.read_text(encoding="utf-8")
)

assert saved_metadata["modeling_table"]["rows"] == 3_855

assert saved_metadata["model_table_sha256"] == sha256_file(
    MODEL_TABLE_PATH
)

assert saved_metadata["alignment_sha256"] == sha256_file(
    ALIGNMENT_CSV
)

assert (
    saved_metadata[
        "session_contract"
    ][
        "forecast_deadline_et"
    ]
    == FORECAST_DEADLINE_ET_V0_1
), (
    "Saved metadata does not contain "
    "the frozen forecast deadline."
)

print("NOTEBOOK 08 FINAL CHECKPOINT: PASS")

NOTEBOOK 08 FINAL CHECKPOINT: PASS



# 28. End-of-notebook checklist

## A. Provider validation — complete

- [x] Freeze eight concrete Headline Arena challenge IDs.
- [x] Validate bullish, bearish, neutral, and roll scenarios.
- [x] Reconstruct all eight challenges using Databento.
- [x] Verify opening and closing boundaries.
- [x] Obtain eight out of eight class agreements.
- [x] Confirm zero observed opening/closing price differences.
- [x] Validate instrument and contract mapping.
- [x] Pass the provider-acceptance gate.
- [x] Save the provider-alignment artifact.

## B. Historical acquisition — complete

- [x] Freeze the historical acquisition window.
- [x] Acquire the historical Databento batch once.
- [x] Reuse the existing completed batch on subsequent runs.
- [x] Restore all purchase/download switches to False.
- [x] Recover 17 historical DBN files.
- [x] Confirm 20 files in the completed batch directory.
- [x] Confirm the saved raw SHA-256 manifest exists.
- [x] Load 5,704,833 historical minute records.
- [x] Resolve historical instrument identifiers.
- [x] Validate all historical contract mappings.
- [x] Convert timestamps to America/New_York.
- [x] Pass the Phase B raw-history checkpoint.

## C. Historical session and target construction — complete

- [x] Construct 4,203 historical sessions.
- [x] Audit historical session boundaries.
- [x] Identify missing minute bars and other exceptions.
- [x] Audit contract-roll transitions.
- [x] Freeze the historical eligibility policy.
- [x] Retain 3,855 target-eligible sessions.
- [x] Exclude 348 structurally ineligible sessions.
- [x] Freeze the +/-0.30% target.
- [x] Construct bearish, neutral, and bullish targets.
- [x] Audit the target distribution.

## D. Feature engineering — complete

- [x] Freeze the conservative 12-feature ES-only baseline.
- [x] Use previous eligible completed sessions only.
- [x] Prevent excluded sessions from becoming feature sources.
- [x] Prevent same-session target leakage.
- [x] Verify lagged and rolling feature calculations.
- [x] Preserve missing values without global imputation.
- [x] Pass the mechanical feature-leakage audit.

## E. Modeling-table exports — complete

- [x] Build the 3,855-row modeling table.
- [x] Verify 16 exported columns.
- [x] Pass the final modeling-table quality audit.
- [x] Export es_daily_modeling_v0_1.parquet.
- [x] Reload and verify the parquet structure.
- [x] Export es_daily_modeling_v0_1_metadata.json.
- [x] Record the model-table SHA-256.
- [x] Record the provider-alignment SHA-256.
- [x] Verify that all final artifacts exist.
- [x] Pass NOTEBOOK 08 FINAL CHECKPOINT: PASS.

## F. Scientific limitations and follow-up validation

- [ ] Quantify historical trading-calendar and weekday-selection effects.
- [ ] Investigate sensitivity to incomplete intraday coverage.
- [ ] Compare model performance across historical market regimes.
- [ ] Establish a leak-free 10:00 ET prediction-time contract.
- [ ] Conduct chronological walk-forward model evaluation.
- [ ] Evaluate probability calibration and Brier diagnostics.
- [ ] Introduce event/news features in a separately audited experiment.
- [ ] Conduct genuinely prospective Headline Arena forecasts.

The incomplete items in Section F are scientific follow-up work.
They do not undo the successful Notebook 08 baseline export.

## Explicitly not performed in Notebook 08

- No model training.
- No hyperparameter tuning.
- No global imputation or scaling.
- No random train/test shuffling.
- No synthetic oversampling.
- No probability calibration.
- No live Arena submissions.
- No automated trading or forecast submissions.
- No modifications to the stable v1.2.0 system.
- No merge into main.

## Notebook 08 completion status

**ES Daily Forward Experiment v0.1 historical-data baseline:
ENGINEERING COMPLETE.**

The final modeling data is ready for additional scientific
quality checks and leakage-safe evaluation in Notebook 09.

The historical labels are Arena-style analog labels and
must not be represented as independently settled historical
Headline Arena challenges.

The experiment remains separate from the frozen
Market Mood Forecasting v1.2.0 system.
